# Laboratorio de features

Notebook de trabajo para **probar features nuevas rápido**, sin correr la comparación de los 14 modelos (`13_comparacion_modelos.ipynb` se corre **una sola vez**, al final, con las features congeladas).

**Qué tiene armado:**
- La **Base actual** = configuración ganadora de `11b_marginales.ipynb` con el HistGB fijo: 0,9339 ± 0,0018.
- `probar(...)`: arma el pipeline de la Base con lo que le sumes (o saques) y lo mide con la misma CV de siempre.
- `comparar_pareado(...)`: la comparación fold por fold de los pasos 3 a 6.
- **Caché del modelo de texto**: `PrediccionOOF` es lo más lento (6 regresiones logísticas sobre decenas de miles de columnas por fold). Como casi ninguna prueba lo toca, se guarda en disco (`notebooks/.cache_laboratorio/`) la **primera vez** y después se reusa, **incluso en otra sesión**. Mientras no cambies el modelo de texto, cada prueba tarda ~1–2 minutos.
- **Registro**: cada prueba se guarda en `notebooks/.cache_laboratorio/resultados.csv` (accuracy de cada fold), así puedo comparar contra pruebas de otro día sin volver a correrlas.

**Cómo se usa (flujo de una prueba):**
1. Si la feature es **determinística** (no aprende de los datos), calculala sobre `train` en una celda nueva, igual que en `07`/`11`.
2. Si **aprende de los datos** (encoding, agregados con target, etc.), va como transformer en `transformers_extra`, para que se ajuste solo con el train de cada fold.
3. `probar("nombre", columnas_numericas_extra=[...])` o `probar("nombre", transformers_extra=[...])`.
4. `comparar_pareado("Base", "nombre")`. **Criterio**: mejora real si gana en 5 de 5 folds, o en 4 de 5 con la media de la diferencia mayor que su desvío.
5. Si pasa, pasa a ser parte de la Base (y después, de `features.py`).

**Reglas de siempre:** una cosa por vez, todo lo que aprende de los datos adentro del `Pipeline`, `random_state=42`. Este laboratorio no se entrega.

In [1]:
import sys
sys.path.append("..")

import os
import pandas as pd
import numpy as np
import warnings
import time
from joblib import Memory

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, TargetEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

from features import (
    aplanar_anidadas, agregar_logs, agregar_features_contenido, CapeoPorMoneda,
    VendedorEncoder, PrediccionOOF, armar_modelo_texto, COLUMNAS_CONTENIDO,
)

SEED = 42

train = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)
train = aplanar_anidadas(train)
train = agregar_features_contenido(train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

modelo_fijo = HistGradientBoostingClassifier(
    learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED
)

columnas_categoricas_base = ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]
columnas_numericas_base = [
    "log_price", "log_available_quantity", "automatic_relist",
    "sold_quantity", "initial_quantity",
    "n_fotos", "n_atributos", "n_variaciones", "tiene_video", "tiene_garantia",
    "free_shipping", "local_pick_up", "n_metodos_pago", "accepts_mercadopago",
    "es_tienda_oficial", "tiene_descuento", "tiene_oferta", "tiene_parent_item",
] + COLUMNAS_CONTENIDO

carpeta_lab = ".cache_laboratorio"
os.makedirs(carpeta_lab, exist_ok=True)
memoria_texto = Memory(location=carpeta_lab, verbose=0)
archivo_resultados = os.path.join(carpeta_lab, "resultados.csv")

## Pipeline de la Base y funciones del laboratorio

`armar_pipeline` reproduce la Base (igual que `12_tuning.ipynb` con el modelo fijo) y acepta:
- `columnas_numericas_extra`: columnas ya calculadas en `train` que se suman tal cual (el HistGB no necesita escalado).
- `transformers_extra`: transformers que se agregan al `ColumnTransformer`, como tuplas `(nombre, transformer, columnas)`.
- `sacar`: nombres de transformers de la Base a sacar (ej. `{"tfidf"}`), para probar si algo sobra.

**Cómo funciona la caché del texto.** El transformer `texto` es un `Pipeline` de dos pasos: `PrediccionOOF` y un paso final `"passthrough"` (no hace nada). `Pipeline(memory=...)` solo guarda en disco los pasos que **no son el último**; el `"passthrough"` está para que `PrediccionOOF` quede guardado. La clave del guardado es "mismo transformer, mismos parámetros, mismas filas", así que en cada fold lo encuentra hecho de una prueba anterior. Si cambiás el modelo de texto (otro `C`, otros n-gramas), la clave cambia y se recalcula solo.

In [2]:
def armar_pipeline(columnas_numericas_extra=(), transformers_extra=(), sacar=(), modelo=None):
    texto_cacheado = Pipeline([
        ("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
        ("identidad", "passthrough"),
    ], memory=memoria_texto)

    transformers = [
        ("onehot", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas_base),
        ("num", "passthrough", columnas_numericas_base + list(columnas_numericas_extra)),
        ("categoria", TargetEncoder(target_type="binary", random_state=SEED), ["category_id"]),
        ("tfidf", TfidfVectorizer(max_features=500), "title"),
        ("texto", texto_cacheado, ["title", "warranty_texto"]),
        ("vendedor", VendedorEncoder(incluir_target=True), ["seller_id", "log_price"]),
    ]
    transformers = [t for t in transformers if t[0] not in set(sacar)] + list(transformers_extra)

    return Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", ColumnTransformer(transformers, sparse_threshold=0)),
        ("modelo", modelo if modelo is not None else modelo_fijo),
    ])


def cargar_resultados():
    if os.path.exists(archivo_resultados):
        return pd.read_csv(archivo_resultados).set_index("nombre")
    return pd.DataFrame(columns=["fecha", "accuracy_media", "accuracy_std", "roc_auc_media",
                                 "fold_1", "fold_2", "fold_3", "fold_4", "fold_5", "minutos"]).rename_axis("nombre")


def probar(nombre, columnas_numericas_extra=(), transformers_extra=(), sacar=(), modelo=None, datos=None,
           parametros_fit=None, pipeline=None):
    """Mide la Base + lo que se le pase con la CV fija (folds en paralelo) y guarda
    el resultado en el registro. Si el nombre ya existía, lo pisa.
    parametros_fit: parámetros que se pasan al fit (ej. {"modelo__cat_features": [...]}).
    pipeline: si se pasa, se mide ese pipeline en vez de armar uno con armar_pipeline."""
    inicio = time.time()
    if pipeline is None:
        pipeline = armar_pipeline(columnas_numericas_extra, transformers_extra, sacar, modelo)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        resultados = cross_validate(pipeline, train if datos is None else datos, y, cv=cv,
                                    scoring=["accuracy", "roc_auc"], n_jobs=5, params=parametros_fit)

    accuracy = resultados["test_accuracy"]
    registro = cargar_resultados()
    registro.loc[nombre] = [pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), accuracy.mean(), accuracy.std(),
                            resultados["test_roc_auc"].mean(), *accuracy, (time.time() - inicio) / 60]
    registro.to_csv(archivo_resultados)

    print(nombre, "-> accuracy:", round(accuracy.mean(), 4), "+/-", round(accuracy.std(), 4),
          "| roc_auc:", round(resultados["test_roc_auc"].mean(), 4),
          "| minutos:", round((time.time() - inicio) / 60, 1))


def comparar_pareado(nombre_a, nombre_b):
    """Diferencia de accuracy fold por fold (B - A), sobre los mismos folds."""
    registro = cargar_resultados()
    columnas_folds = ["fold_1", "fold_2", "fold_3", "fold_4", "fold_5"]
    diferencias = registro.loc[nombre_b, columnas_folds].to_numpy(dtype=float) \
        - registro.loc[nombre_a, columnas_folds].to_numpy(dtype=float)
    print(f"{nombre_b}  vs  {nombre_a}")
    print("  diferencia por fold:", np.round(diferencias, 4))
    print("  folds en que gana B:", (diferencias > 0).sum(), "de", len(diferencias))
    print("  media de la diferencia:", round(diferencias.mean(), 4), "| desvío de la diferencia:", round(diferencias.std(), 4))
    gana = (diferencias > 0).sum()
    veredicto = gana == 5 or (gana == 4 and diferencias.mean() > diferencias.std())
    print("  ->", "MEJORA REAL" if veredicto else "no pasa el criterio")

## Base

Tiene que dar **0,9339 ± 0,0018** (igual que `11b`/`12`). La primera vez tarda más porque llena la caché del modelo de texto. Las siguientes, mucho menos.

In [3]:
probar("Base")

Base -> accuracy: 0.9339 +/- 0.0018 | roc_auc: 0.9827 | minutos: 0.7


## Registro de pruebas

Todas las pruebas hechas hasta ahora (de esta sesión y de las anteriores), de mejor a peor.

In [4]:
cargar_resultados().sort_values("accuracy_media", ascending=False).round(4)

,fecha,accuracy_media,accuracy_std,roc_auc_media,fold_1,fold_2,fold_3,fold_4,fold_5,minutos
nombre,,,,,,,,,,
Base,2026-10-05 20:10,0.9339,0.0018,0.9827,0.9332,0.9308,0.9359,0.9349,0.9349,0.6742


## Pruebas

Ejemplos de cómo se escribe una prueba (copiá la que corresponda en una celda nueva):

```python
# 1) Feature determinística: se calcula sobre train y se suma como columna
train["titulo_largo"] = train["title"].str.len()
probar("+ titulo_largo", columnas_numericas_extra=["titulo_largo"])
comparar_pareado("Base", "+ titulo_largo")

# 2) Algo que aprende de los datos: va como transformer (se ajusta en cada fold)
probar("+ TE de seller_state", transformers_extra=[
    ("estado_te", TargetEncoder(target_type="binary", random_state=SEED), ["seller_state"]),
])
comparar_pareado("Base", "+ TE de seller_state")

# 3) Sacar algo de la Base para ver si sobra
probar("Base sin tfidf 500", sacar={"tfidf"})
comparar_pareado("Base", "Base sin tfidf 500")
```

Si la feature nueva sale de una columna que **no** está en `train` todavía (por ejemplo una que hay que aplanar), calculala primero en una celda y después llamá a `probar`.

## Prueba: Gradient Boosting de `train.ipynb` con las features finales

La misma grilla del inciso 3.3.11 de `train.ipynb` (`max_iter` ∈ {100, 200} × `learning_rate` ∈ {0,05, 0,1} × `max_depth` ∈ {None, 5}), medida en el laboratorio con las features finales. Cada combinación queda en el registro. La combinación `max_iter=200`, `learning_rate=0.1`, `max_depth=None` es exactamente la Base (el HistGB fijo).

Diferencia con `13_comparacion_modelos.ipynb`: allá el preprocesador es el de `train.ipynb` (One-Hot con `drop="first"`, escalado e imputación). Para árboles eso casi no cambia nada (un árbol corta igual con la variable escalada o sin escalar), así que estos números deberían ser prácticamente los mismos que dará la 13.

In [5]:
for max_iter in [100, 200]:
    for learning_rate in [0.05, 0.1]:
        for max_depth in [None, 5]:
            nombre = f"HistGB train.ipynb (max_iter={max_iter}, lr={learning_rate}, max_depth={max_depth})"
            probar(nombre, modelo=HistGradientBoostingClassifier(
                max_iter=max_iter, learning_rate=learning_rate, max_depth=max_depth, random_state=SEED))

HistGB train.ipynb (max_iter=100, lr=0.05, max_depth=None) -> accuracy: 0.9317 +/- 0.0015 | roc_auc: 0.9816 | minutos: 0.5


HistGB train.ipynb (max_iter=100, lr=0.05, max_depth=5) -> accuracy: 0.9292 +/- 0.0021 | roc_auc: 0.9806 | minutos: 0.5


HistGB train.ipynb (max_iter=100, lr=0.1, max_depth=None) -> accuracy: 0.9339 +/- 0.0018 | roc_auc: 0.9827 | minutos: 0.5


HistGB train.ipynb (max_iter=100, lr=0.1, max_depth=5) -> accuracy: 0.9313 +/- 0.0021 | roc_auc: 0.982 | minutos: 0.4


HistGB train.ipynb (max_iter=200, lr=0.05, max_depth=None) -> accuracy: 0.9334 +/- 0.0017 | roc_auc: 0.9827 | minutos: 0.7


HistGB train.ipynb (max_iter=200, lr=0.05, max_depth=5) -> accuracy: 0.9312 +/- 0.0019 | roc_auc: 0.982 | minutos: 0.6


HistGB train.ipynb (max_iter=200, lr=0.1, max_depth=None) -> accuracy: 0.9339 +/- 0.0018 | roc_auc: 0.9827 | minutos: 0.6


HistGB train.ipynb (max_iter=200, lr=0.1, max_depth=5) -> accuracy: 0.9319 +/- 0.0022 | roc_auc: 0.9822 | minutos: 0.5


In [6]:
registro = cargar_resultados()
grilla_gb = registro[registro.index.str.startswith("HistGB train.ipynb")].sort_values("accuracy_media", ascending=False)
grilla_gb[["accuracy_media", "accuracy_std", "roc_auc_media"]].round(4)

,accuracy_media,accuracy_std,roc_auc_media
nombre,,,
"HistGB train.ipynb (max_iter=200, lr=0.1, max_depth=None)",0.9339,0.0018,0.9827
"HistGB train.ipynb (max_iter=100, lr=0.1, max_depth=None)",0.9339,0.0018,0.9827
"HistGB train.ipynb (max_iter=200, lr=0.05, max_depth=None)",0.9334,0.0017,0.9827
"HistGB train.ipynb (max_iter=200, lr=0.1, max_depth=5)",0.9319,0.0022,0.9822
"HistGB train.ipynb (max_iter=100, lr=0.05, max_depth=None)",0.9317,0.0015,0.9816
"HistGB train.ipynb (max_iter=100, lr=0.1, max_depth=5)",0.9313,0.0021,0.9820
"HistGB train.ipynb (max_iter=200, lr=0.05, max_depth=5)",0.9312,0.0019,0.9820
"HistGB train.ipynb (max_iter=100, lr=0.05, max_depth=5)",0.9292,0.0021,0.9806


In [7]:
mejor_gb = grilla_gb.index[0]
comparar_pareado("Base", mejor_gb)

HistGB train.ipynb (max_iter=200, lr=0.1, max_depth=None)  vs  Base
  diferencia por fold: [0. 0. 0. 0. 0.]
  folds en que gana B: 0 de 5
  media de la diferencia: 0.0 | desvío de la diferencia: 0.0
  -> no pasa el criterio


# Última ronda de features

## Paso 1 — Publicación "padre" (`parent_item_id`)

Idea: si una publicación es una re-publicación de otra (su "padre"), probablemente tengan la misma condición. Si el padre está en el train, su etiqueta sería una pista muy fuerte para el hijo.

EDA antes de proponer nada:
- qué % de filas tiene `parent_item_id` (train y test);
- de esas, cuántos padres son un `id` que **está en train**;
- si el padre está en train, cuántas veces coincide la condición;
- al revés: cuántas publicaciones de train son "padre" de otra, y cuántos padres se repiten (publicaciones "hermanas" que comparten padre).

In [2]:
test_crudo = pd.read_json("../data/test_data.jsonlines", lines=True)
ids_train = set(train["id"])
condicion_por_id = dict(zip(train["id"], y))

print("¿id único en train?", train["id"].is_unique, "| ids de test que están en train:", test_crudo["id"].isin(ids_train).sum())
print()
for nombre, datos in [("train", train), ("test", test_crudo)]:
    tiene_padre = datos["parent_item_id"].notna()
    padre_en_train = datos["parent_item_id"].isin(ids_train)
    print(f"{nombre}: con parent_item_id = {tiene_padre.sum()} ({tiene_padre.mean():.1%})"
          f" | padre está en train = {padre_en_train.sum()}")

padre_en_train = train["parent_item_id"].isin(ids_train)
coincide = y[padre_en_train] == train.loc[padre_en_train, "parent_item_id"].map(condicion_por_id)
print("\ntrain: filas cuyo padre está en train:", padre_en_train.sum(), "| coincide la condición:", coincide.sum())

print("\nAl revés:")
print("publicaciones de train que son padre de alguna fila de train:", train["id"].isin(set(train["parent_item_id"].dropna())).sum())
print("publicaciones de train que son padre de alguna fila de test:", train["id"].isin(set(test_crudo["parent_item_id"].dropna())).sum())
hijos_por_padre = pd.concat([train["parent_item_id"], test_crudo["parent_item_id"]]).dropna().value_counts()
print("padres distintos (train + test):", len(hijos_por_padre), "| padres con más de un hijo (hermanos):", (hijos_por_padre > 1).sum())

¿id único en train? True | ids de test que están en train: 0

train: con parent_item_id = 53934 (77.0%) | padre está en train = 1
test: con parent_item_id = 23055 (76.8%) | padre está en train = 0

train: filas cuyo padre está en train: 1 | coincide la condición: 1

Al revés:
publicaciones de train que son padre de alguna fila de train: 1
publicaciones de train que son padre de alguna fila de test: 0
padres distintos (train + test): 76989 | padres con más de un hijo (hermanos): 0


**Conclusión del Paso 1: la idea no se puede aplicar, así que no hay transformer que proponer.**

- El 77% de las filas tiene `parent_item_id` (igual en train y en test), pero **el padre casi nunca está en el dataset**: 1 sola fila de train (de 53.934) tiene su padre en train, y **ninguna** fila de test.
- Tampoco sirve "al revés" ni por hermanos: ningún `parent_item_id` se repite entre las 100.000 filas, así que no hay dos publicaciones que compartan padre.
- Lectura: `parent_item_id` apunta a publicaciones **fuera** de esta muestra (la versión anterior de la publicación en MercadoLibre). Como dato, solo sirve que **exista** (que la publicación fue re-publicada), y eso ya está en la Base como `tiene_parent_item`.

Un transformer padre → condición devolvería `NaN` en el 99,99% de las filas: no hay nada que medir.

## Paso 2 — Títulos repetidos

Idea: muchas publicaciones se repiten con el **mismo título** (el mismo producto publicado varias veces). Si las que comparten título suelen tener la misma condición, la condición de las "gemelas" del train sería una pista.

**Normalización** (determinística, sin aprender nada): minúsculas, sin espacios al principio ni al final, espacios múltiples reducidos a uno. Así `"iPhone 5  16GB "` y `"iphone 5 16gb"` cuentan como el mismo título.

EDA:
- cuántos títulos se repiten en train y cuántas filas están en un grupo repetido;
- qué % del test tiene un título idéntico en train;
- coincidencia de condición dentro de cada grupo. Para cada fila con gemelas en train: ¿su condición coincide con la **mayoría de las otras** filas del grupo, sin contarse a sí misma?;
- **qué parte de los repetidos son del mismo vendedor.** Si casi todos lo son, `VendedorEncoder` ya tiene esa señal y esto no agregaría mucho.

In [3]:
def normalizar_titulo(titulos):
    return titulos.str.lower().str.strip().str.replace(r"\s+", " ", regex=True)


titulo_train = normalizar_titulo(train["title"])
titulo_test = normalizar_titulo(test_crudo["title"])
repeticiones_train = titulo_train.map(titulo_train.value_counts())

print("Títulos distintos en train:", titulo_train.nunique(), "de", len(titulo_train))
print("Filas de train con título repetido en train:", (repeticiones_train > 1).sum(), f"({(repeticiones_train > 1).mean():.1%})")
print("Filas de test con título idéntico en train:", titulo_test.isin(set(titulo_train)).sum(),
      f"({titulo_test.isin(set(titulo_train)).mean():.1%})")

# coincidencia: % new de las OTRAS filas del mismo título (sin contarse a sí misma)
suma_new = titulo_train.map(y.groupby(titulo_train).sum())
pct_new_otras = (suma_new - y) / (repeticiones_train - 1)
repetidas = repeticiones_train > 1
mayoria_otras = (pct_new_otras[repetidas] > 0.5).astype(int)
empate = pct_new_otras[repetidas] == 0.5
coincide = (mayoria_otras == y[repetidas])[~empate]
print("\nFilas con gemelas (sin empates):", len(coincide), "| coincide con la mayoría de las otras:", f"{coincide.mean():.1%}")

grupos = pd.DataFrame({"titulo": titulo_train, "seller_id": train["seller_id"]})[repetidas]
vendedores_por_titulo = grupos.groupby("titulo")["seller_id"].nunique()
print("\nGrupos de títulos repetidos:", len(vendedores_por_titulo),
      "| con un solo vendedor:", f"{(vendedores_por_titulo == 1).mean():.1%}")
filas_mismo_vendedor = grupos["titulo"].map(vendedores_por_titulo) == 1
print("Filas repetidas cuyo grupo es de un solo vendedor:", f"{filas_mismo_vendedor.mean():.1%}")

# test: ¿el título idéntico en train es del mismo vendedor?
en_train = titulo_test.isin(set(titulo_train))
pares_train = set(zip(titulo_train, train["seller_id"]))
mismo_vendedor_test = pd.Series([(t, s) in pares_train for t, s in zip(titulo_test, test_crudo["seller_id"])])
print("Test con título en train:", en_train.sum(), "| de esas, con el MISMO vendedor en train:",
      f"{mismo_vendedor_test[en_train.to_numpy()].mean():.1%}")

Títulos distintos en train: 69318 de 70000
Filas de train con título repetido en train: 1063 (1.5%)
Filas de test con título idéntico en train: 457 (1.5%)

Filas con gemelas (sin empates): 1057 | coincide con la mayoría de las otras: 93.3%

Grupos de títulos repetidos: 381 | con un solo vendedor: 57.5%
Filas repetidas cuyo grupo es de un solo vendedor: 50.5%


Test con título en train: 457 | de esas, con el MISMO vendedor en train: 70.5%


**Lectura del EDA.**
- La coincidencia es alta (93% de las filas coincide con la mayoría de sus gemelas), pero alcanza a muy pocas filas: 1,5% del train y 1,5% del test.
- En el 70% de los casos de test, la gemela de train es **del mismo vendedor**, que `VendedorEncoder` ya cubre.
- Techo optimista de ganancia: ≈ +0,001.

Igual se mide, porque es barato.

**Prueba P2.**
- `titulo_norm` (determinístico, afuera del `Pipeline`).
- `TargetEncoder(target_type="binary")` sobre `titulo_norm`: *cross-fitting* + *smoothing*. No es *leakage*: ninguna fila usa su propia etiqueta, y la validación solo ve promedios del train del fold.
- `ConteoOtras`: cuántas **otras** filas del train del fold tienen el mismo título. No usa el target. En el `fit_transform` cada fila se descuenta a sí misma (`- 1`), para que en train y en validación/test el número signifique lo mismo.

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin


class ConteoOtras(BaseEstimator, TransformerMixin):
    """Cantidad de OTRAS filas del train (del fit) con el mismo valor."""

    def fit(self, X, y=None):
        self.conteo_ = X.iloc[:, 0].value_counts()
        return self

    def transform(self, X):
        return X.iloc[:, 0].map(self.conteo_).fillna(0).to_frame().to_numpy()

    def fit_transform(self, X, y=None):
        self.fit(X)
        return (X.iloc[:, 0].map(self.conteo_) - 1).to_frame().to_numpy()


train["titulo_norm"] = normalizar_titulo(train["title"])

probar("P2 - títulos repetidos (TE + conteo)", transformers_extra=[
    ("titulo_te", TargetEncoder(target_type="binary", random_state=SEED), ["titulo_norm"]),
    ("titulo_conteo", ConteoOtras(), ["titulo_norm"]),
])
comparar_pareado("Base", "P2 - títulos repetidos (TE + conteo)")

P2 - títulos repetidos (TE + conteo) -> accuracy: 0.9341 +/- 0.0017 | roc_auc: 0.9827 | minutos: 0.6
P2 - títulos repetidos (TE + conteo)  vs  Base
  diferencia por fold: [-0.0006  0.0009  0.      0.0008  0.    ]
  folds en que gana B: 2 de 5
  media de la diferencia: 0.0002 | desvío de la diferencia: 0.0006
  -> no pasa el criterio


**Conclusión del Paso 2: no pasa.** 0,9341 ± 0,0017 contra 0,9339. Gana en 2 de 5 folds, con media +0,0002 y desvío 0,0006: es ruido. Es lo que anticipaba el EDA: la señal alcanza a solo el 1,5% de las filas, y en la mayoría de esos casos la gemela es del mismo vendedor, que ya está cubierto. **No se suma.**

## Paso 3 — Perfil del vendedor sin target

`PerfilVendedor`: por `seller_id`, el promedio de 5 columnas sobre las publicaciones del vendedor vistas en el `fit` (% con garantía, % de fotos cuadradas, % con envío gratis, cantidad de fotos promedio, % tienda oficial). Vendedores no vistos → `NaN` (el HistGB los maneja solo).

**No es leakage:** no usa `condition`, solo promedia features de las publicaciones, y se ajusta adentro del `Pipeline` con el train de cada fold.

Dos variantes:
- **P3a**: el promedio incluye a la propia fila. En train, un vendedor de 1 publicación tiene un "perfil" que es copia de su fila. En validación/test, un vendedor nuevo tiene `NaN`: el número significa cosas distintas en train y en test.
- **P3b**: en el `fit_transform` cada fila se describe con las **otras** publicaciones de su vendedor (sin otras → `NaN`). Así significa lo mismo en train y en test.

In [4]:
class PerfilVendedor(BaseEstimator, TransformerMixin):
    """Promedio, por seller_id, de columnas de las publicaciones vistas en fit.
    No usa el target. Vendedores no vistos -> NaN.
    excluir_propia=True: en el train, cada fila se describe con las OTRAS
    publicaciones de su vendedor (si no tiene otras -> NaN)."""

    def __init__(self, columnas, excluir_propia=True):
        self.columnas = columnas
        self.excluir_propia = excluir_propia

    def fit(self, X, y=None):
        valores = X[self.columnas].astype(float)
        self.sumas_ = valores.groupby(X["seller_id"]).sum()
        self.cantidades_ = valores.notna().groupby(X["seller_id"]).sum()
        return self

    def transform(self, X):
        medias = self.sumas_ / self.cantidades_.replace(0, np.nan)
        return medias.reindex(X["seller_id"]).to_numpy()

    def fit_transform(self, X, y=None):
        self.fit(X)
        if not self.excluir_propia:
            return self.transform(X)
        valores = X[self.columnas].astype(float)
        sumas = self.sumas_.reindex(X["seller_id"]).to_numpy() - valores.fillna(0).to_numpy()
        cantidades = self.cantidades_.reindex(X["seller_id"]).to_numpy() - valores.notna().to_numpy()
        return sumas / np.where(cantidades > 0, cantidades, np.nan)


columnas_perfil = ["tiene_garantia", "foto_pct_cuadradas", "free_shipping", "n_fotos", "es_tienda_oficial"]

probar("P3a - perfil vendedor (incluye propia)", transformers_extra=[
    ("perfil", PerfilVendedor(columnas_perfil, excluir_propia=False), ["seller_id"] + columnas_perfil),
])
comparar_pareado("Base", "P3a - perfil vendedor (incluye propia)")
print()
probar("P3b - perfil vendedor (sin la propia)", transformers_extra=[
    ("perfil", PerfilVendedor(columnas_perfil, excluir_propia=True), ["seller_id"] + columnas_perfil),
])
comparar_pareado("Base", "P3b - perfil vendedor (sin la propia)")

P3a - perfil vendedor (incluye propia) -> accuracy: 0.9337 +/- 0.0014 | roc_auc: 0.9825 | minutos: 0.6
P3a - perfil vendedor (incluye propia)  vs  Base
  diferencia por fold: [-0.0002  0.0009 -0.0021  0.0009 -0.0006]
  folds en que gana B: 2 de 5
  media de la diferencia: -0.0002 | desvío de la diferencia: 0.0011
  -> no pasa el criterio



P3b - perfil vendedor (sin la propia) -> accuracy: 0.934 +/- 0.0017 | roc_auc: 0.9827 | minutos: 0.6
P3b - perfil vendedor (sin la propia)  vs  Base
  diferencia por fold: [ 0.      0.0004 -0.0016  0.0015 -0.0001]
  folds en que gana B: 2 de 5
  media de la diferencia: 0.0 | desvío de la diferencia: 0.001
  -> no pasa el criterio


## Prueba extra V3 — `VendedorEncoder` sin contarse a sí misma

Hoy, en el `fit_transform` de `VendedorEncoder`:
- `seller_n_listings` cuenta a la propia fila: un vendedor con 1 publicación da log(2) en train, pero log(1) si es nuevo en validación/test;
- `seller_avg_log_price` incluye el precio propio: un vendedor con 1 publicación tiene su propio precio como "promedio" en train, y el promedio global en validación/test.

(`seller_pct_new` ya excluye a la propia fila, con *leave-one-out*.)

`VendedorEncoderV3` aplica la misma idea a las otras dos: en el `fit_transform`, conteo y precio promedio de las **otras** publicaciones (sin otras → promedio global, igual que un vendedor no visto). `transform` no cambia. No usa el target más allá de lo que ya usaba `seller_pct_new`.

In [5]:
class VendedorEncoderV3(VendedorEncoder):
    def fit_transform(self, X, y=None):
        salida = super().fit_transform(X, y)
        seller = X["seller_id"]
        conteo = seller.map(self.conteo_)
        suma_log_price = seller.map(X.groupby(seller)["log_price"].sum())
        salida["seller_n_listings"] = np.log1p(conteo - 1).to_numpy()
        salida["seller_avg_log_price"] = np.where(
            conteo > 1, (suma_log_price - X["log_price"]) / (conteo - 1), self.media_log_price_global_
        )
        return salida


probar("V3 - VendedorEncoder sin contarse a sí misma", sacar={"vendedor"}, transformers_extra=[
    ("vendedor_v3", VendedorEncoderV3(incluir_target=True), ["seller_id", "log_price"]),
])
comparar_pareado("Base", "V3 - VendedorEncoder sin contarse a sí misma")

V3 - VendedorEncoder sin contarse a sí misma -> accuracy: 0.9333 +/- 0.0019 | roc_auc: 0.9828 | minutos: 0.5
V3 - VendedorEncoder sin contarse a sí misma  vs  Base
  diferencia por fold: [ 0.0003 -0.0011 -0.0021 -0.0004  0.0001]
  folds en que gana B: 2 de 5
  media de la diferencia: -0.0006 | desvío de la diferencia: 0.0009
  -> no pasa el criterio


**Conclusión del Paso 3 y de V3: ninguna pasa.**

| Prueba | accuracy | fold por fold vs Base |
|---|---|---|
| P3a: perfil vendedor (incluye propia) | 0,9337 ± 0,0014 | 2/5, media −0,0002 |
| P3b: perfil vendedor (sin la propia) | 0,9340 ± 0,0017 | 2/5, media 0,0000 |
| V3: `VendedorEncoder` sin contarse a sí misma | 0,9333 ± 0,0019 | 2/5, media −0,0006 |

- **Perfil del vendedor:** no agrega nada. Interpretación (no medida): para vendedores con varias publicaciones, `seller_pct_new` ya resume casi todo lo que el perfil podría decir sobre si es "tienda" o "particular". Para los de 1 publicación, P3b da `NaN`, igual que la Base, así que no les aporta información nueva. P3b queda apenas por encima de P3a, en la dirección que predecía el argumento, pero la diferencia es ruido.
- **V3:** el desajuste que describimos existe en teoría, pero **corregirlo no mejora** (y hasta empeora un poco, dentro del ruido). Lo dejamos como está. Para la defensa queda como dato: se probó y no cambia nada.

## Paso 4 — Promediar semillas

**De dónde sale el azar del HistGB en nuestra configuración.** Según la documentación, `random_state` controla:
- el submuestreo para armar los *bins*, que solo se activa con más de 200.000 filas (acá hay 56.000 por fold): **no actúa**;
- el sorteo de features por corte (`max_features`), pero usamos `max_features=1.0` (todas): **no actúa**;
- **el 10% de validación del *early stopping*** (`early_stopping="auto"` se prende con más de 10.000 filas): **sí actúa**. Cambia con qué 90% se entrenan los árboles y en qué iteración se frena.

Promediar las probabilidades de 5 HistGB que solo difieren en `random_state` (42 a 46) es parecido al Bagging de `S8_Ensambles.ipynb`: modelos entrenados con muestras algo distintas, cuyo promedio reduce varianza. Se arma con `VotingClassifier(voting="soft")` (también de S8). El resto de la Base (modelo de texto, encoders) queda con semilla 42.

In [3]:
from sklearn.ensemble import VotingClassifier

promedio_semillas = VotingClassifier(
    estimators=[
        (f"histgb_{semilla}", HistGradientBoostingClassifier(learning_rate=0.1, max_depth=None, max_iter=200,
                                                             random_state=semilla))
        for semilla in range(42, 47)
    ],
    voting="soft",
)

probar("P4 - promedio de 5 semillas (42-46)", modelo=promedio_semillas)
comparar_pareado("Base", "P4 - promedio de 5 semillas (42-46)")

P4 - promedio de 5 semillas (42-46) -> accuracy: 0.934 +/- 0.0016 | roc_auc: 0.9829 | minutos: 1.9
P4 - promedio de 5 semillas (42-46)  vs  Base
  diferencia por fold: [ 0.0011  0.0001 -0.0011  0.0006 -0.0003]
  folds en que gana B: 3 de 5
  media de la diferencia: 0.0001 | desvío de la diferencia: 0.0008
  -> no pasa el criterio


**Conclusión del Paso 4: no pasa.** 0,9340 ± 0,0016 contra 0,9339. Gana en 3 de 5 folds, con media +0,0001 y desvío 0,0008: ruido. La roc_auc sube apenas (0,9827 → 0,9829), que es lo esperable de un promedio (probabilidades un poco más suaves), pero no cambia qué lado del 0,5 queda cada publicación. Con un solo 10% de validación distinto entre semillas, los 5 modelos son casi iguales y promediarlos no reduce casi nada de varianza. Además cuesta 3 veces más tiempo (1,9 min contra 0,6). **No se suma.**

## Paso 5 — Umbral de decisión

Hoy la clase sale de comparar P(`new`) con 0,5 (`S8_Ensambles.ipynb`: "se asigna la clase a partir de un umbral, por ejemplo 0.5"; la curva ROC de `Repaso de Metricas.pdf` muestra qué pasa al moverlo). **Elegir el umbral que maximiza la accuracy no está en el material.**

Procedimiento honesto (el umbral nunca se elige mirando el fold donde se evalúa):
1. Probabilidades *out-of-fold* de la Base con `cross_val_predict` (mismos folds de siempre).
2. Para cada fold k: con los **otros 4 folds** elijo el umbral (entre 0,30 y 0,70, de a 0,005) que maximiza la accuracy; lo aplico al fold k y comparo contra 0,5 en ese mismo fold.
3. Las accuracies del fold k con el umbral elegido se guardan en el registro como "P5", para usar `comparar_pareado` contra la Base.

In [3]:
from sklearn.model_selection import cross_val_predict

inicio = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    prob_oof = cross_val_predict(armar_pipeline(), train, y, cv=cv, method="predict_proba", n_jobs=5)[:, 1]
print("Probabilidades OOF en", round((time.time() - inicio) / 60, 1), "min")

folds = list(cv.split(train, y))
umbrales = np.arange(0.30, 0.7001, 0.005)
y_array = y.to_numpy()

filas_umbral = []
for k, (_, indices_k) in enumerate(folds):
    otros = np.concatenate([indices for j, (_, indices) in enumerate(folds) if j != k])
    accuracy_otros = [((prob_oof[otros] >= u) == y_array[otros]).mean() for u in umbrales]
    umbral_elegido = umbrales[int(np.argmax(accuracy_otros))]
    filas_umbral.append({
        "fold": k + 1,
        "umbral_elegido": round(umbral_elegido, 3),
        "accuracy_con_0.5": ((prob_oof[indices_k] >= 0.5) == y_array[indices_k]).mean(),
        "accuracy_con_umbral": ((prob_oof[indices_k] >= umbral_elegido) == y_array[indices_k]).mean(),
    })

tabla_umbral = pd.DataFrame(filas_umbral).set_index("fold")
tabla_umbral["diferencia"] = tabla_umbral["accuracy_con_umbral"] - tabla_umbral["accuracy_con_0.5"]
tabla_umbral.round(4)

Probabilidades OOF en 1.0 min


,umbral_elegido,accuracy_con_0.5,accuracy_con_umbral,diferencia
fold,,,,
1,0.495,0.9332,0.9334,0.0002
2,0.495,0.9308,0.9309,0.0001
3,0.495,0.9359,0.9359,0.0000
4,0.495,0.9349,0.9351,0.0001
5,0.495,0.9349,0.9347,-0.0001


In [4]:
registro = cargar_resultados()
accuracy_p5 = tabla_umbral["accuracy_con_umbral"].to_numpy()
registro.loc["P5 - umbral elegido con 4 folds"] = [pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), accuracy_p5.mean(),
                                                  accuracy_p5.std(), np.nan, *accuracy_p5, np.nan]
registro.to_csv(archivo_resultados)

print("Chequeo: accuracy con 0,5 por fold == Base del registro ->",
      np.allclose(tabla_umbral["accuracy_con_0.5"].to_numpy(),
                  registro.loc["Base", ["fold_1", "fold_2", "fold_3", "fold_4", "fold_5"]].to_numpy(dtype=float)))
comparar_pareado("Base", "P5 - umbral elegido con 4 folds")

curva = pd.Series([((prob_oof >= u) == y_array).mean() for u in umbrales], index=np.round(umbrales, 3))
print("\nAccuracy OOF total según el umbral (alrededor de 0,5):")
print(curva.loc[0.44:0.56].round(4).to_string())

Chequeo: accuracy con 0,5 por fold == Base del registro -> True
P5 - umbral elegido con 4 folds  vs  Base
  diferencia por fold: [ 0.0002  0.0001  0.      0.0001 -0.0001]
  folds en que gana B: 3 de 5
  media de la diferencia: 0.0001 | desvío de la diferencia: 0.0001
  -> no pasa el criterio

Accuracy OOF total según el umbral (alrededor de 0,5):
0.440    0.9332
0.445    0.9333
0.450    0.9332
0.455    0.9334
0.460    0.9335
0.465    0.9333
0.470    0.9335
0.475    0.9335
0.480    0.9335
0.485    0.9336
0.490    0.9338
0.495    0.9340
0.500    0.9339
0.505    0.9337
0.510    0.9336
0.515    0.9335
0.520    0.9334
0.525    0.9333
0.530    0.9332
0.535    0.9332
0.540    0.9333
0.545    0.9333
0.550    0.9334
0.555    0.9333
0.560    0.9332


**Conclusión del Paso 5: no pasa.** En las 5 rotaciones el umbral elegido con 4 folds fue el mismo, **0,495**: prácticamente 0,5. La ganancia honesta es +0,0001 (3 de 5 folds), ruido. La curva de accuracy alrededor de 0,5 es casi plana (entre 0,9332 y 0,9340 para umbrales de 0,44 a 0,56). Lectura: las probabilidades del HistGB están bien **calibradas** (no hay un corrimiento sistemático hacia `new` o `used`), así que 0,5 ya es el umbral correcto. **Se mantiene 0,5.**

## Paso 6 — CatBoost

Material: `material/Arboles/Árboles.pdf` (slide 68: "Es un Gradient Boosting preparado para variables categóricas. Funciona muy bien, incluso con variables categóricas (sin transformar) y con los parámetros de default"), `material/Arboles/Modelos.ipynb` y el `Catboost.ipynb` del profe (github.com/LCaravaggio/AnalisisPredictivo, 04_Árboles), que usa `CatBoostClassifier()` con parámetros por defecto. Lo que **no** está en el material es cómo codifica las categóricas.

**Ordered target statistics.** CatBoost ordena las filas de train al azar, y codifica la categoría de cada fila con el % de `new` **de las filas anteriores** de esa categoría, mezclado con un *prior* global: (suma de `new` anteriores + a·p) / (cantidad anterior + a).
- La etiqueta de una fila nunca entra en su propia codificación.
- Cada fila de train queda en la misma situación que una fila de test: solo "ve" otras filas.
- Usa varias permutaciones para que las primeras filas de cada categoría (con estadísticas ruidosas) no sesguen.
- Al predecir usa todo el train.

Dentro de nuestra CV va adentro del `Pipeline`, así que solo ve el train de cada fold.

- **6a**: mismas features de la Base, `CatBoostClassifier(random_state=42, verbose=0)` en lugar del HistGB. Todo llega numérico, así que solo cambia el algoritmo de boosting (árboles simétricos, 1.000 iteraciones, `learning_rate` automático).
- **6b**: `seller_id`, `category_id`, `listing_type_id`, `buying_mode`, `shipping_mode` y `seller_state` **crudas** como `cat_features`. Sin `VendedorEncoder`, sin `TargetEncoder` y sin el one-hot; se mantienen `prob_texto_new`, el TF-IDF de 500 y todas las numéricas.

Detalle técnico: `cat_features` se pasa **en el `fit`** (`parametros_fit={"modelo__cat_features": [...]}`), no en el constructor. Con lista en el constructor sklearn no puede clonar el modelo (incompatibilidad conocida de CatBoost con `clone`), y CatBoost no acepta tupla.

In [3]:
from catboost import CatBoostClassifier

probar("P6a - CatBoost (mismas features de la Base)", modelo=CatBoostClassifier(random_state=SEED, verbose=0))
comparar_pareado("Base", "P6a - CatBoost (mismas features de la Base)")

P6a - CatBoost (mismas features de la Base) -> accuracy: 0.9338 +/- 0.002 | roc_auc: 0.9828 | minutos: 0.8
P6a - CatBoost (mismas features de la Base)  vs  Base
  diferencia por fold: [-0.0005  0.0002 -0.0018  0.0022 -0.0007]
  folds en que gana B: 2 de 5
  media de la diferencia: -0.0001 | desvío de la diferencia: 0.0013
  -> no pasa el criterio


In [4]:
columnas_cat = ["seller_id", "category_id", "listing_type_id", "buying_mode", "shipping_mode", "seller_state"]
columnas_cat_crudas = [f"cat_{c}" for c in columnas_cat]
for c in columnas_cat:
    train[f"cat_{c}"] = train[c].astype(str).fillna("desconocido")

# Orden de salida del ColumnTransformer sin onehot/categoria/vendedor:
# num (columnas_numericas_base) -> tfidf (500) -> texto (1) -> categóricas crudas (6, agregadas al final)
inicio_cat = len(columnas_numericas_base) + 500 + 1
indices_cat = list(range(inicio_cat, inicio_cat + len(columnas_cat_crudas)))

# Chequeo de los índices con un ajuste chico (2.000 filas) del preprocesador
chequeo = armar_pipeline(sacar={"onehot", "categoria", "vendedor"},
                         transformers_extra=[("cat_crudas", "passthrough", columnas_cat_crudas)])
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    salida = Pipeline(chequeo.steps[:-1]).fit_transform(train.iloc[:2000], y.iloc[:2000])
print("Columnas de salida:", salida.shape[1], "| índices categóricos:", indices_cat)
print("Valores en esas columnas (fila 0):", list(salida[0, list(indices_cat)]))
assert salida.shape[1] == inicio_cat + len(columnas_cat_crudas)

probar("P6b - CatBoost con categóricas crudas", sacar={"onehot", "categoria", "vendedor"},
       transformers_extra=[("cat_crudas", "passthrough", columnas_cat_crudas)],
       modelo=CatBoostClassifier(random_state=SEED, verbose=0),
       parametros_fit={"modelo__cat_features": indices_cat})
comparar_pareado("Base", "P6b - CatBoost con categóricas crudas")

Columnas de salida: 531 | índices categóricos: [525, 526, 527, 528, 529, 530]
Valores en esas columnas (fila 0): ['4086929269', 'MLA73721', 'bronze', 'buy_it_now', 'not_specified', 'Buenos Aires']


P6b - CatBoost con categóricas crudas -> accuracy: 0.9365 +/- 0.0017 | roc_auc: 0.9837 | minutos: 1.6
P6b - CatBoost con categóricas crudas  vs  Base
  diferencia por fold: [0.004  0.0024 0.0008 0.0032 0.0026]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.0026 | desvío de la diferencia: 0.0011
  -> MEJORA REAL


**Conclusión del Paso 6.**

| Prueba | accuracy | roc_auc | fold por fold vs Base | tiempo |
|---|---|---|---|---|
| Base (HistGB) | 0,9339 ± 0,0018 | 0,9827 | — | 0,7 min |
| P6a: CatBoost, mismas features | 0,9338 ± 0,0020 | 0,9828 | 2/5, media −0,0001 | 0,8 min |
| **P6b: CatBoost, categóricas crudas** | **0,9365 ± 0,0017** | **0,9837** | **5/5, media +0,0026, desvío 0,0011** | 1,6 min |

- **6a**: con las mismas features, CatBoost rinde **igual** que el HistGB. El algoritmo de boosting por sí solo no cambia nada (es coherente con la "superficie plana" de `12_tuning`).
- **6b pasa con claridad**: gana en los 5 folds, por entre +0,0008 y +0,0040, con media 2,4 veces su desvío. Lo que mejora es **cómo se codifican `seller_id` y `category_id`**: los *ordered target statistics* de CatBoost (con varias permutaciones y combinaciones entre categóricas) funcionan mejor que nuestro *leave-one-out* + `TargetEncoder`. Y eso sin `seller_n_listings` ni `seller_avg_log_price`, que se fueron con `VendedorEncoder`.
- **Sesgo de selección**: en esta ronda ya van 8 pruebas sobre los mismos folds. Que alguna gane 5/5 por azar no es imposible (≈3% cada una). Pero 6b no pasa "justo": su ganancia es la más grande de la ronda y la mínima por fold (+0,0008) también es positiva. El chequeo con otra semilla de CV del cierre lo va a confirmar o no.

# Cierre

Solo pasó una prueba (P6b), así que no hay combinación ni *ablation*.

## Chequeo de robustez: otra semilla de CV

Todas las decisiones se tomaron con los mismos 5 folds (`random_state=42`). Para descartar que P6b haya ganado por cómo cayeron esas filas en los folds (sesgo de selección), mido la Base y P6b con **otros** folds: `StratifiedKFold(n_splits=5, shuffle=True, random_state=7)`. Mismo criterio pareado. Si la mejora es real, tiene que mantenerse.

El modelo de texto se vuelve a calcular una vez para estos folds nuevos (la caché depende de qué filas hay en cada fold). Al final vuelvo `cv` a la semilla 42.

In [3]:
from catboost import CatBoostClassifier

columnas_cat = ["seller_id", "category_id", "listing_type_id", "buying_mode", "shipping_mode", "seller_state"]
columnas_cat_crudas = [f"cat_{c}" for c in columnas_cat]
for c in columnas_cat:
    train[f"cat_{c}"] = train[c].astype(str).fillna("desconocido")
inicio_cat = len(columnas_numericas_base) + 500 + 1
indices_cat = list(range(inicio_cat, inicio_cat + len(columnas_cat_crudas)))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

probar("Base (cv semilla 7)")
probar("P6b (cv semilla 7)", sacar={"onehot", "categoria", "vendedor"},
       transformers_extra=[("cat_crudas", "passthrough", columnas_cat_crudas)],
       modelo=CatBoostClassifier(random_state=SEED, verbose=0),
       parametros_fit={"modelo__cat_features": indices_cat})
comparar_pareado("Base (cv semilla 7)", "P6b (cv semilla 7)")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

Base (cv semilla 7) -> accuracy: 0.9332 +/- 0.0012 | roc_auc: 0.9826 | minutos: 1.0


P6b (cv semilla 7) -> accuracy: 0.9359 +/- 0.0019 | roc_auc: 0.9836 | minutos: 1.7
P6b (cv semilla 7)  vs  Base (cv semilla 7)
  diferencia por fold: [0.0038 0.0038 0.0025 0.001  0.0027]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.0028 | desvío de la diferencia: 0.001
  -> MEJORA REAL


**Robustez confirmada.** Con folds distintos (semilla 7): Base 0,9332 ± 0,0012 → P6b 0,9359 ± 0,0019, **gana en 5 de 5 folds, media +0,0028, desvío 0,0010**. Es prácticamente la misma ganancia que con la semilla 42 (+0,0026). La mejora de P6b no depende de cómo cayeron las filas en los folds: **P6b es la configuración final.**

## Verificación de `armar_pipeline_final()` (`features.py`)

La configuración final quedó en `features.py`: `agregar_categoricas_crudas`, `armar_pipeline_final` e `INDICES_CAT`. Para confirmar que es **exactamente** P6b, la mido con la CV de siempre (semilla 42). Tiene que dar las mismas accuracies fold por fold que P6b (diferencia 0).

In [3]:
from features import agregar_categoricas_crudas, armar_pipeline_final, INDICES_CAT

train_final = agregar_categoricas_crudas(train)

inicio = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    resultados_final = cross_validate(armar_pipeline_final(), train_final, y, cv=cv, scoring=["accuracy", "roc_auc"],
                                      n_jobs=5, params={"modelo__cat_features": INDICES_CAT})

accuracy_final = resultados_final["test_accuracy"]
registro = cargar_resultados()
registro.loc["Final (armar_pipeline_final de features.py)"] = [
    pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), accuracy_final.mean(), accuracy_final.std(),
    resultados_final["test_roc_auc"].mean(), *accuracy_final, (time.time() - inicio) / 60]
registro.to_csv(archivo_resultados)

print("Final -> accuracy:", round(accuracy_final.mean(), 4), "+/-", round(accuracy_final.std(), 4),
      "| roc_auc:", round(resultados_final["test_roc_auc"].mean(), 4),
      "| minutos:", round((time.time() - inicio) / 60, 1))
comparar_pareado("P6b - CatBoost con categóricas crudas", "Final (armar_pipeline_final de features.py)")

Final -> accuracy: 0.9365 +/- 0.0017 | roc_auc: 0.9837 | minutos: 2.1
Final (armar_pipeline_final de features.py)  vs  P6b - CatBoost con categóricas crudas
  diferencia por fold: [0. 0. 0. 0. 0.]
  folds en que gana B: 0 de 5
  media de la diferencia: 0.0 | desvío de la diferencia: 0.0
  -> no pasa el criterio


# Ronda corta post-congelamiento

Features congeladas en la configuración final (`armar_pipeline_final()`, CatBoost P6b, **0,9365**). Dos pruebas, contra esa configuración.

## Parte A — Métodos de pago (y `shipping`)

Hoy de `non_mercado_pago_payment_methods` solo se usa **cuántos** métodos hay (`n_metodos_pago`, en `aplanar_anidadas`). Cada método trae `description`, `id` y `type`. EDA:
- qué métodos hay, en qué % de filas aparece cada uno y su % de `new`, en train; y en qué % de filas de test aparece cada uno (el test no tiene etiqueta);
- qué otros campos tiene `shipping` además de `mode`, `free_shipping` y `local_pick_up`.

Si algún método separa claramente y es frecuente, la propuesta es una flag 0/1 por método: un *one-hot* / variables *dummies* como en `material/Variables Categóricas, Datos Faltantes y Feature Engineering/Copia de AP_Categóricas.ipynb` (secciones "One-Hot Encoder" y "Feature Engineering"), igual que los `tag_*` de `aplanar_anidadas`.

In [2]:
from collections import Counter

test_crudo = pd.read_json("../data/test_data.jsonlines", lines=True)


def metodos_de_pago(lista):
    return {m.get("description") for m in lista}


metodos_train = train["non_mercado_pago_payment_methods"].map(metodos_de_pago)
metodos_test = test_crudo["non_mercado_pago_payment_methods"].map(metodos_de_pago)
conteo = Counter(m for conjunto in metodos_train for m in conjunto)

filas = []
for metodo, n in conteo.most_common():
    tiene = metodos_train.map(lambda conjunto, m=metodo: m in conjunto)
    filas.append({
        "metodo": metodo,
        "pct_filas_train": tiene.mean(),
        "pct_new_con": y[tiene].mean(),
        "pct_new_sin": y[~tiene].mean(),
        "pct_filas_test": metodos_test.map(lambda conjunto, m=metodo: m in conjunto).mean(),
    })
tabla_metodos = pd.DataFrame(filas).set_index("metodo")
tabla_metodos["diferencia"] = tabla_metodos["pct_new_con"] - tabla_metodos["pct_new_sin"]
print("Sin ningún método:", round((train["n_metodos_pago"] == 0).mean(), 3), "| % new:", round(y[train["n_metodos_pago"] == 0].mean(), 3))
tabla_metodos.round(3)

Sin ningún método: 0.306 | % new: 0.489


,pct_filas_train,pct_new_con,pct_new_sin,pct_filas_test,diferencia
metodo,,,,,
Efectivo,0.670,0.555,0.501,0.672,0.053
Transferencia bancaria,0.514,0.611,0.459,0.516,0.152
Tarjeta de crédito,0.246,0.708,0.481,0.247,0.227
Acordar con el comprador,0.080,0.453,0.544,0.080,-0.092
Giro postal,0.017,0.220,0.542,0.017,-0.322
MercadoPago,0.007,1.000,0.534,0.008,0.466
Visa,0.007,0.500,0.537,0.007,-0.037
MasterCard,0.006,0.489,0.537,0.007,-0.048
Visa Electron,0.006,0.484,0.537,0.006,-0.053


In [3]:
print("Campos de shipping:", dict(Counter(k for d in train["shipping"] for k in d)))
print()
shipping_tags = train["shipping"].map(lambda d: tuple(d.get("tags") or []))
print("shipping.tags:", shipping_tags.value_counts().to_dict())
print("shipping.dimensions no nulas:", train["shipping"].map(lambda d: d.get("dimensions") is not None).sum())
print("shipping.methods no vacías:", train["shipping"].map(lambda d: bool(d.get("methods"))).sum())

tiene_free_methods = train["shipping"].map(lambda d: d.get("free_methods") is not None)
print("\nshipping.free_methods presente:", tiene_free_methods.sum(), f"({tiene_free_methods.mean():.1%})",
      "| % new:", round(y[tiene_free_methods].mean(), 3))
print("Cruce free_methods presente vs free_shipping:")
print(pd.crosstab(tiene_free_methods, train["free_shipping"], rownames=["free_methods"], colnames=["free_shipping"]))

Campos de shipping: {'local_pick_up': 70000, 'methods': 67892, 'tags': 70000, 'free_shipping': 70000, 'mode': 70000, 'dimensions': 70000, 'free_methods': 2108}

shipping.tags: {(): 69954, ('relisted_with_modifications',): 43, ('free_method_upgraded',): 3}
shipping.dimensions no nulas: 13
shipping.methods no vacías: 0

shipping.free_methods presente: 2108 (3.0%) | % new: 0.933
Cruce free_methods presente vs free_shipping:
free_shipping  False  True 
free_methods               
False          67883      9
True               0   2108


**Lectura del EDA.**
- Separan y son frecuentes: **tarjeta de crédito** (25% de las filas, 71% `new` contra 48% sin) y **transferencia** (51%, +15 puntos). Efectivo y "acordar con el comprador" separan poco. Giro postal separa mucho (−32 puntos) pero está en el 1,7%. MercadoPago, cheque y las marcas de tarjeta son < 1%.
- Train y test tienen las mismas proporciones.
- `shipping`: nada nuevo. `free_methods` coincide con `free_shipping` salvo en 9 filas; `tags`, `dimensions` y `methods` están casi siempre vacíos.

**Prueba PA: 5 flags de método de pago** (los que están en ≥ 1,5% de las filas), *dummies* determinísticas como los `tag_*` de `aplanar_anidadas`, sumadas a las numéricas de la **configuración final**.

`armar_pipeline_final_lab` replica `armar_pipeline_final()` de `features.py` (con la caché del texto del laboratorio) y acepta columnas numéricas extra. Como las columnas extra corren de lugar a las categóricas, devuelve también sus índices. Antes de medir, un chequeo con 2.000 filas confirma que en esos índices están las categóricas.

In [4]:
from catboost import CatBoostClassifier
from features import agregar_categoricas_crudas, COLUMNAS_NUMERICAS, COLUMNAS_CAT_CRUDAS

train = agregar_categoricas_crudas(train)

metodos_flag = {
    "Efectivo": "pago_efectivo",
    "Transferencia bancaria": "pago_transferencia",
    "Tarjeta de crédito": "pago_tarjeta_credito",
    "Acordar con el comprador": "pago_acordar",
    "Giro postal": "pago_giro_postal",
}
for metodo, columna in metodos_flag.items():
    train[columna] = metodos_train.map(lambda conjunto, m=metodo: m in conjunto)


def armar_pipeline_final_lab(columnas_numericas_extra=()):
    """armar_pipeline_final() de features.py, con la caché del texto del laboratorio
    y columnas numéricas extra. Devuelve (pipeline, índices de las categóricas)."""
    numericas = COLUMNAS_NUMERICAS + list(columnas_numericas_extra)
    texto_cacheado = Pipeline([
        ("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
        ("identidad", "passthrough"),
    ], memory=memoria_texto)
    preprocesador = ColumnTransformer([
        ("num", "passthrough", numericas),
        ("tfidf", TfidfVectorizer(max_features=500), "title"),
        ("texto", texto_cacheado, ["title", "warranty_texto"]),
        ("cat_crudas", "passthrough", COLUMNAS_CAT_CRUDAS),
    ], sparse_threshold=0)
    pipeline = Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", preprocesador),
        ("modelo", CatBoostClassifier(random_state=SEED, verbose=0)),
    ])
    inicio = len(numericas) + 500 + 1
    return pipeline, list(range(inicio, inicio + len(COLUMNAS_CAT_CRUDAS)))


columnas_pago = list(metodos_flag.values())
pipeline_pa, indices_pa = armar_pipeline_final_lab(columnas_pago)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    salida = Pipeline(pipeline_pa.steps[:-1]).fit_transform(train.iloc[:2000], y.iloc[:2000])
print("Índices categóricos:", indices_pa, "| fila 0:", list(salida[0, indices_pa]))
assert salida.shape[1] == indices_pa[-1] + 1

probar("PA - Final + 5 flags de método de pago", pipeline=pipeline_pa,
       parametros_fit={"modelo__cat_features": indices_pa})
comparar_pareado("Final (armar_pipeline_final de features.py)", "PA - Final + 5 flags de método de pago")

Índices categóricos: [530, 531, 532, 533, 534, 535] | fila 0: ['4086929269', 'MLA73721', 'bronze', 'buy_it_now', 'not_specified', 'Buenos Aires']


PA - Final + 5 flags de método de pago -> accuracy: 0.9368 +/- 0.0013 | roc_auc: 0.9837 | minutos: 1.7
PA - Final + 5 flags de método de pago  vs  Final (armar_pipeline_final de features.py)
  diferencia por fold: [-0.0001  0.0013  0.0001  0.0003 -0.0003]
  folds en que gana B: 3 de 5
  media de la diferencia: 0.0002 | desvío de la diferencia: 0.0006
  -> no pasa el criterio


**Conclusión de la Parte A: no pasa.** 0,9368 ± 0,0013 contra 0,9365. Gana en 3 de 5 folds, con media +0,0002 y desvío 0,0006: ruido. Lectura (no medida): ofrecer tarjeta o transferencia distingue comercio de particular, y eso ya lo captura CatBoost con `seller_id` (y las features de tienda oficial, envío, etc.). **No se toca nada: las features siguen congeladas.**

# Ronda de robustez sobre CatBoost (C1–C5)

Todo contra la configuración final (CatBoost P6b, registro "Final (armar_pipeline_final de features.py)", **0,9365**), con comparación pareada.

`armar_pipeline_final_lab` (redefinida acá, más general que la de la Parte A) replica `armar_pipeline_final()` de `features.py` con la caché del texto del laboratorio, y permite cambiar el modelo, sumar numéricas o categóricas crudas, o sacar transformers. Devuelve el pipeline y los índices de las categóricas en la salida.

## C1 — Promediar semillas de CatBoost

**Azar de CatBoost** (valores por defecto verificados con `get_all_params()`):
- `bootstrap_type="MVS"`, `subsample=0.8`: cada árbol usa el 80% de las filas, elegidas al azar (favoreciendo las peor predichas);
- `random_strength=1`: ruido aleatorio en el puntaje de cada corte candidato;
- `has_time=False`: permutaciones al azar para los *ordered target statistics* de las categóricas.

Actúa en **cada árbol** y en la codificación de las categóricas, a diferencia del HistGB del Paso 4 (solo cambiaba el 10% del *early stopping*). Promediar 5 semillas es un voto *soft* / *bagging* (`S8_Ensambles.ipynb`). MVS, `random_strength` y las permutaciones no están en el material.

**Medición:**
- probabilidades OOF de CatBoost P6b con las semillas 42 a 46 (la 42 se lee de `oof_ensambles.csv`). Su promedio = voto *soft* de 5 modelos en cada fold;
- en test: las 5 semillas entrenadas con todo el train, para ver cuánto cambian las predicciones solo por la semilla.

Se guardan en `.cache_laboratorio/oof_catboost_semillas.csv` y `test_catboost_semillas.csv`.

In [3]:
from catboost import CatBoostClassifier
from features import (agregar_categoricas_crudas, agregar_features_contenido, COLUMNAS_NUMERICAS,
                      COLUMNAS_CAT_CRUDAS)

train = agregar_categoricas_crudas(train)


def armar_pipeline_final_lab(columnas_numericas_extra=(), columnas_cat_extra=(), sacar=(), modelo=None):
    """armar_pipeline_final() de features.py con la caché del texto del laboratorio.
    Permite sumar numéricas / categóricas crudas, sacar transformers ("tfidf", ...)
    o cambiar el modelo. Devuelve (pipeline, índices de las categóricas)."""
    numericas = COLUMNAS_NUMERICAS + list(columnas_numericas_extra)
    categoricas = COLUMNAS_CAT_CRUDAS + list(columnas_cat_extra)
    texto_cacheado = Pipeline([
        ("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
        ("identidad", "passthrough"),
    ], memory=memoria_texto)
    partes = [
        ("num", "passthrough", numericas, len(numericas)),
        ("tfidf", TfidfVectorizer(max_features=500), "title", 500),
        ("texto", texto_cacheado, ["title", "warranty_texto"], 1),
        ("cat_crudas", "passthrough", categoricas, len(categoricas)),
    ]
    partes = [parte for parte in partes if parte[0] not in set(sacar)]
    inicio_cat = sum(ancho for nombre, _, _, ancho in partes if nombre != "cat_crudas")
    preprocesador = ColumnTransformer([(n, t, c) for n, t, c, _ in partes], sparse_threshold=0)
    pipeline = Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", preprocesador),
        ("modelo", modelo if modelo is not None else CatBoostClassifier(random_state=SEED, verbose=0)),
    ])
    return pipeline, list(range(inicio_cat, inicio_cat + len(categoricas)))


columnas_folds = ["fold_1", "fold_2", "fold_3", "fold_4", "fold_5"]
NOMBRE_FINAL = "Final (armar_pipeline_final de features.py)"
folds = list(cv.split(train, y))
fold_de_cada_fila = np.empty(len(train), dtype=int)
for k, (_, indices_validacion) in enumerate(folds):
    fold_de_cada_fila[indices_validacion] = k + 1


def accuracy_por_fold_desde_prob(probabilidad):
    acierto = (np.asarray(probabilidad) >= 0.5).astype(int) == y.to_numpy()
    return pd.Series(acierto).groupby(fold_de_cada_fila).mean().to_numpy()


def registrar(nombre, accuracies, minutos, roc_auc=np.nan):
    registro = cargar_resultados()
    registro.loc[nombre] = [pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), accuracies.mean(), accuracies.std(),
                            roc_auc, *accuracies, minutos]
    registro.to_csv(archivo_resultados)

In [4]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

semillas = [42, 43, 44, 45, 46]
archivo_semillas = os.path.join(carpeta_lab, "oof_catboost_semillas.csv")
oof_semillas = pd.DataFrame({"fold": fold_de_cada_fila, "y": y.to_numpy()})
oof_semillas["semilla_42"] = pd.read_csv(os.path.join(carpeta_lab, "oof_ensambles.csv"))["catboost"]

inicio_c1 = time.time()
for semilla in semillas[1:]:
    inicio = time.time()
    pipeline, indices = armar_pipeline_final_lab(modelo=CatBoostClassifier(random_state=semilla, verbose=0))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        oof_semillas[f"semilla_{semilla}"] = cross_val_predict(
            pipeline, train, y, cv=cv, method="predict_proba", n_jobs=5,
            params={"modelo__cat_features": indices})[:, 1]
    print(f"semilla {semilla}: {(time.time() - inicio) / 60:.1f} min")
oof_semillas.to_csv(archivo_semillas, index=False)
minutos_cv = (time.time() - inicio_c1) / 60

columnas_semillas = [f"semilla_{s}" for s in semillas]
oof_semillas["promedio"] = oof_semillas[columnas_semillas].mean(axis=1)

tabla_semillas = pd.DataFrame({c: accuracy_por_fold_desde_prob(oof_semillas[c]) for c in columnas_semillas + ["promedio"]},
                              index=columnas_folds).T
tabla_semillas["media"] = tabla_semillas[columnas_folds].mean(axis=1)
tabla_semillas["desvio_entre_folds"] = tabla_semillas[columnas_folds].std(axis=1, ddof=0)
print("Accuracy media de las semillas sueltas: entre", round(tabla_semillas.loc[columnas_semillas, "media"].min(), 4),
      "y", round(tabla_semillas.loc[columnas_semillas, "media"].max(), 4))
tabla_semillas.round(4)

semilla 43: 1.6 min


semilla 44: 1.6 min


semilla 45: 1.6 min


semilla 46: 1.6 min


Accuracy media de las semillas sueltas: entre 0.9364 y 0.9366


,fold_1,fold_2,fold_3,fold_4,fold_5,media,desvio_entre_folds
semilla_42,0.9372,0.9332,0.9366,0.9381,0.9375,0.9365,0.0017
semilla_43,0.9361,0.9341,0.9372,0.9369,0.9376,0.9364,0.0012
semilla_44,0.9361,0.9339,0.9366,0.9379,0.9384,0.9366,0.0016
semilla_45,0.9361,0.9341,0.9363,0.9379,0.9379,0.9365,0.0014
semilla_46,0.9358,0.9340,0.9364,0.9381,0.9374,0.9364,0.0014
promedio,0.9370,0.9344,0.9369,0.9378,0.9377,0.9367,0.0012


In [5]:
accuracy_promedio = tabla_semillas.loc["promedio", columnas_folds].to_numpy(dtype=float)
registrar("C1 - CatBoost promedio 5 semillas (42-46)", accuracy_promedio, minutos_cv,
          roc_auc=np.mean([roc_auc_score(oof_semillas.loc[oof_semillas["fold"] == k, "y"],
                                         oof_semillas.loc[oof_semillas["fold"] == k, "promedio"]) for k in range(1, 6)]))
comparar_pareado(NOMBRE_FINAL, "C1 - CatBoost promedio 5 semillas (42-46)")

clases_oof = {c: (oof_semillas[c] >= 0.5).astype(int) for c in columnas_semillas + ["promedio"]}
cambios_pares = [(clases_oof[a] != clases_oof[b]).mean() for i, a in enumerate(columnas_semillas) for b in columnas_semillas[i + 1:]]
print(f"\nTrain (OOF): entre dos semillas sueltas cambia la clase en {np.mean(cambios_pares):.2%} de las filas "
      f"(rango {min(cambios_pares):.2%}-{max(cambios_pares):.2%})")
print(f"Train (OOF): semilla 42 vs promedio: {(clases_oof['semilla_42'] != clases_oof['promedio']).mean():.2%}")

C1 - CatBoost promedio 5 semillas (42-46)  vs  Final (armar_pipeline_final de features.py)
  diferencia por fold: [-0.0002  0.0011  0.0002 -0.0004  0.0002]
  folds en que gana B: 3 de 5
  media de la diferencia: 0.0002 | desvío de la diferencia: 0.0005
  -> no pasa el criterio

Train (OOF): entre dos semillas sueltas cambia la clase en 0.72% de las filas (rango 0.69%-0.75%)
Train (OOF): semilla 42 vs promedio: 0.47%


In [6]:
# Test: 5 semillas entrenadas con todo el train
test = agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(
    pd.read_json("../data/test_data.jsonlines", lines=True))))
archivo_test_semillas = os.path.join(carpeta_lab, "test_catboost_semillas.csv")

inicio = time.time()
prob_test = pd.DataFrame()
for semilla in semillas:
    pipeline, indices = armar_pipeline_final_lab(modelo=CatBoostClassifier(random_state=semilla, verbose=0))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pipeline.fit(train, y, modelo__cat_features=indices)
    prob_test[f"semilla_{semilla}"] = pipeline.predict_proba(test)[:, 1]
prob_test.to_csv(archivo_test_semillas, index=False)
print(f"Test, 5 semillas: {(time.time() - inicio) / 60:.1f} min")

prob_test["promedio"] = prob_test[columnas_semillas].mean(axis=1)
clases_test = {c: (prob_test[c] >= 0.5).astype(int) for c in columnas_semillas + ["promedio"]}
cambios_test = [(clases_test[a] != clases_test[b]).mean() for i, a in enumerate(columnas_semillas) for b in columnas_semillas[i + 1:]]
print(f"Test: entre dos semillas sueltas cambia la clase en {np.mean(cambios_test):.2%} de las filas "
      f"(rango {min(cambios_test):.2%}-{max(cambios_test):.2%}) -> ~{np.mean(cambios_test) * 30000:.0f} filas")
print(f"Test: semilla 42 vs promedio: {(clases_test['semilla_42'] != clases_test['promedio']).mean():.2%}")
vieja = pd.read_csv("../submission/submission_catboost.csv")
print("Chequeo: semilla 42 == submission_catboost.csv ->",
      ((prob_test["semilla_42"] >= 0.5).map({True: "new", False: "used"}) == vieja["condition"]).all())

Test, 5 semillas: 3.0 min
Test: entre dos semillas sueltas cambia la clase en 0.73% de las filas (rango 0.68%-0.77%) -> ~218 filas
Test: semilla 42 vs promedio: 0.42%
Chequeo: semilla 42 == submission_catboost.csv -> True


**Conclusión de C1: no pasa, pero da información de robustez.**

| | accuracy | desvío entre folds |
|---|---|---|
| Semillas sueltas (42–46) | 0,9364 – 0,9366 | 0,0012 – 0,0017 |
| **Promedio de 5 semillas** | **0,9367** | **0,0012** |

- **Contra P6b** (semilla 42): 3/5 folds, media +0,0002, desvío 0,0005 → **no pasa** el criterio.
- **La semilla 42 no fue "suerte"**: las 5 semillas sueltas dan entre 0,9364 y 0,9366. CatBoost P6b es estable en accuracy.
- **Varianza:** el desvío entre folds del promedio (0,0012) queda en el extremo bajo del de las semillas sueltas (0,0012–0,0017). Baja algo, pero poco.
- **Predicciones:** entre dos semillas sueltas cambia la clase en el **0,73% del test (~218 filas)**. El promedio cambia el 0,42% respecto de la semilla 42. Eso es ≈ 0,002–0,004 de accuracy que puede moverse **solo por la semilla**: cualquier diferencia de ese tamaño en el leaderboard es ruido.
- Chequeo: la semilla 42 entrenada con todo el train reproduce exactamente `submission_catboost.csv`.
- Costo del promedio: 5 veces el tiempo de entrenamiento (~5 min contra ~1 min con todo el train).

## C2 — Tuning de CatBoost

| Hiperparámetro | Qué controla | Valores | Material |
|---|---|---|---|
| `depth` | profundidad de cada árbol **simétrico** (mismo corte en todo un nivel → 2^depth hojas) | 4 · 6 · 8 | `max_depth` (S7, `Árboles.pdf`); árboles simétricos no |
| `learning_rate` | cuánto pesa cada árbol nuevo (por defecto, automático ≈ 0,058 con 56.000 filas) | 0,03 · 0,06 · 0,1 | S8 |
| `iterations` | máximo de árboles: tope 3.000, lo decide el *early stopping* | — | S8 (`n_estimators`) |
| `l2_leaf_reg` | penalización L2 de los valores de las hojas (por defecto 3) | 1 · 3 · 10 | idea de L2 (`Continuación de regresion y Regulización L1 y L2.pdf`) |

**Early stopping sin tocar el fold de validación.** `CatBoostConEarlyStopping` es un envoltorio compatible con sklearn. En su `fit` (que dentro del `Pipeline` solo recibe el **train del fold**) separa al azar el 10% de ese train (estratificado, semilla 42) como validación interna, entrena CatBoost con el 90% restante y frena si en 100 árboles seguidos no mejora la pérdida en ese 10% (`use_best_model=True`). El fold de validación de la CV nunca participa. *Early stopping* no está en el material (misma idea que en `12_tuning`).

`RandomizedSearchCV`: 15 combinaciones de las 27, misma CV, `refit="accuracy"`, `random_state=42`. El mejor de 15 elegido sobre los mismos folds sale algo inflado (sesgo de selección): solo cuenta una ganancia clara.

In [4]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split, RandomizedSearchCV


class CatBoostConEarlyStopping(ClassifierMixin, BaseEstimator):
    """CatBoost con early stopping sobre una validación interna sacada del propio
    train que recibe en fit (nunca del fold de validación de la CV)."""

    def __init__(self, depth=6, learning_rate=None, l2_leaf_reg=3, iterations=3000,
                 early_stopping_rounds=100, validation_fraction=0.1, random_state=42):
        self.depth = depth
        self.learning_rate = learning_rate
        self.l2_leaf_reg = l2_leaf_reg
        self.iterations = iterations
        self.early_stopping_rounds = early_stopping_rounds
        self.validation_fraction = validation_fraction
        self.random_state = random_state

    def fit(self, X, y, cat_features=None):
        X_entrena, X_valida, y_entrena, y_valida = train_test_split(
            X, y, test_size=self.validation_fraction, stratify=y, random_state=self.random_state)
        self.modelo_ = CatBoostClassifier(depth=self.depth, learning_rate=self.learning_rate,
                                          l2_leaf_reg=self.l2_leaf_reg, iterations=self.iterations,
                                          random_state=self.random_state, verbose=0)
        self.modelo_.fit(X_entrena, y_entrena, cat_features=cat_features, eval_set=(X_valida, y_valida),
                         early_stopping_rounds=self.early_stopping_rounds, use_best_model=True)
        self.classes_ = self.modelo_.classes_
        self.n_iter_ = self.modelo_.get_best_iteration() + 1
        return self

    def predict_proba(self, X):
        return self.modelo_.predict_proba(X)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

In [5]:
N_COMBINACIONES = 15
DATOS_C2, Y_C2 = train, y
pipeline_c2, indices_c2 = armar_pipeline_final_lab(modelo=CatBoostConEarlyStopping(random_state=SEED))
grilla_c2 = {
    "modelo__depth": [4, 6, 8],
    "modelo__learning_rate": [0.03, 0.06, 0.1],
    "modelo__l2_leaf_reg": [1, 3, 10],
}
busqueda_c2 = RandomizedSearchCV(pipeline_c2, grilla_c2, n_iter=N_COMBINACIONES, scoring=["accuracy", "roc_auc"],
                                 refit="accuracy", cv=cv, random_state=SEED, n_jobs=5)
inicio = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    busqueda_c2.fit(DATOS_C2, Y_C2, modelo__cat_features=indices_c2)
minutos_c2 = (time.time() - inicio) / 60
print(f"Búsqueda: {minutos_c2:.1f} min")
print("Mejor combinación:", busqueda_c2.best_params_, "| árboles usados al reentrenar con todo:",
      busqueda_c2.best_estimator_.named_steps["modelo"].n_iter_)

resultados_c2 = pd.DataFrame(busqueda_c2.cv_results_)
tabla_c2 = resultados_c2[["param_modelo__depth", "param_modelo__learning_rate", "param_modelo__l2_leaf_reg",
                          "mean_test_accuracy", "std_test_accuracy", "mean_test_roc_auc", "rank_test_accuracy"]].copy()
tabla_c2.columns = ["depth", "learning_rate", "l2_leaf_reg", "accuracy", "accuracy_std", "roc_auc", "rank"]
tabla_c2.sort_values("rank").round(4)

Búsqueda: 53.9 min
Mejor combinación: {'modelo__learning_rate': 0.03, 'modelo__l2_leaf_reg': 3, 'modelo__depth': 8} | árboles usados al reentrenar con todo: 1685


,depth,learning_rate,l2_leaf_reg,accuracy,accuracy_std,roc_auc,rank
3,8,0.03,3,0.9364,0.0013,0.9835,1
6,6,0.06,10,0.9360,0.0012,0.9834,2
14,6,0.03,10,0.9358,0.0012,0.9833,3
2,6,0.03,1,0.9358,0.0017,0.9834,4
8,6,0.03,3,0.9356,0.0011,0.9834,5
7,6,0.10,10,0.9356,0.0013,0.9832,6
9,8,0.03,10,0.9355,0.0011,0.9834,7
5,6,0.10,1,0.9355,0.0016,0.9833,8
1,6,0.06,3,0.9353,0.0011,0.9833,9
11,4,0.06,3,0.9350,0.0016,0.9830,10


In [6]:
mejor_c2 = busqueda_c2.best_index_
accuracy_mejor_c2 = np.array([resultados_c2.loc[mejor_c2, f"split{k}_test_accuracy"] for k in range(5)])
nombre_c2 = "C2 - CatBoost tuneado " + str({k.replace("modelo__", ""): v for k, v in busqueda_c2.best_params_.items()})
registrar(nombre_c2, accuracy_mejor_c2, minutos_c2, roc_auc=resultados_c2.loc[mejor_c2, "mean_test_roc_auc"])
comparar_pareado(NOMBRE_FINAL, nombre_c2)

C2 - CatBoost tuneado {'learning_rate': 0.03, 'l2_leaf_reg': 3, 'depth': 8}  vs  Final (armar_pipeline_final de features.py)
  diferencia por fold: [-0.0002  0.0005  0.0003 -0.001  -0.0004]
  folds en que gana B: 2 de 5
  media de la diferencia: -0.0002 | desvío de la diferencia: 0.0005
  -> no pasa el criterio


**Conclusión de C2: no pasa. CatBoost por defecto es lo mejor.** (Tiempo: 54 minutos.)

- Mejor de 15: `depth=8`, `learning_rate=0,03`, `l2_leaf_reg=3` (1.685 árboles al reentrenar con todo) = **0,9364 ± 0,0013**, contra 0,9365 de P6b. 2/5 folds, media −0,0002.
- **Superficie plana**, como en `12_tuning`: las 15 combinaciones están entre 0,9346 y 0,9364. Patrón: árboles más profundos (6–8) > 4, y `learning_rate` chico un poco mejor.
- **Costo del *early stopping*:** la combinación más parecida a los valores por defecto (`depth=6`, `learning_rate=0,06`, `l2=3`) da 0,9353, unos 0,001 por debajo de P6b. Esa diferencia es el precio de entrenar con el 90% del train (el otro 10% se usa para frenar). Ninguna combinación recupera ese costo.
- Coherente con la slide 68 de `Árboles.pdf`: CatBoost "funciona muy bien… con los parámetros de default". **Se mantienen los parámetros por defecto.**

## C3 — Más categóricas crudas (EDA)

Candidatas a sumar como `cat_features` crudas:
- `currency_id` (moneda);
- `status` (estado de la publicación);
- `official_store_id` (tienda oficial; nulo → "ninguna"). Hoy solo se usa la flag `es_tienda_oficial`;
- `tags` como un solo texto ordenado (ej. `"dragged_bids_and_visits|good_quality_thumbnail"`). Hoy entran como 5 flags `tag_*`, que en `02_features` midieron ruido (bloque F).

EDA: cardinalidad, valores más frecuentes con su % del total y % de `new`, y si la distribución es parecida en test.

In [2]:
test_crudo = pd.read_json("../data/test_data.jsonlines", lines=True)


def categoricas_extra(df):
    return pd.DataFrame({
        "currency_id": df["currency_id"].astype(str),
        "status": df["status"].astype(str),
        "official_store_id": df["official_store_id"].map(lambda v: "ninguna" if pd.isna(v) else str(int(v))),
        "tags": df["tags"].map(lambda lista: "|".join(sorted(lista)) if len(lista) else "sin_tags"),
    }, index=df.index)


extra_train = categoricas_extra(train)
extra_test = categoricas_extra(test_crudo)

for columna in extra_train.columns:
    print(f"=== {columna}: {extra_train[columna].nunique()} valores distintos en train, "
          f"{extra_test[columna].nunique()} en test; "
          f"valores de test que no están en train: {(~extra_test[columna].isin(set(extra_train[columna]))).mean():.2%} de las filas")
    resumen = pd.DataFrame({
        "pct_train": extra_train[columna].value_counts(normalize=True),
        "pct_new": y.groupby(extra_train[columna]).mean(),
        "pct_test": extra_test[columna].value_counts(normalize=True),
    }).sort_values("pct_train", ascending=False).head(8)
    print(resumen.round(3).to_string())
    print()

=== currency_id: 2 valores distintos en train, 2 en test; valores de test que no están en train: 0.00% de las filas
             pct_train  pct_new  pct_test
currency_id                              
ARS              0.994    0.538     0.994
USD              0.006    0.381     0.006

=== status: 4 valores distintos en train, 3 en test; valores de test que no están en train: 0.00% de las filas
                pct_train  pct_new  pct_test
status                                      
active              0.958    0.530     0.955
paused              0.042    0.692     0.045
closed              0.000    0.400     0.000
not_yet_active      0.000    1.000       NaN

=== official_store_id: 165 valores distintos en train, 112 en test; valores de test que no están en train: 0.13% de las filas
                   pct_train  pct_new  pct_test
official_store_id                              
ninguna                0.992    0.533     0.992
1580                   0.000    1.000     0.000
6448           

**Lectura del EDA.** Señal chica: `official_store_id` ya está cubierta por `es_tienda_oficial` y por `seller_id`; `currency_id` alcanza a muy pocas filas. Lo más prometedor es `status = paused` (4,2%, 69% `new` contra 53%), que hoy no se usa en ninguna feature, y los tags de calidad de foto.

**Prueba C3:** las 4 columnas como categóricas crudas extra de P6b (`cat_currency_id`, `cat_status`, `cat_official_store_id`, `cat_tags`). Son determinísticas y sin target (afuera del `Pipeline`). CatBoost las codifica con sus *ordered target statistics*. Si pasa, *ablation* (sacar de a una).

In [5]:
for columna in extra_train.columns:
    train[f"cat_{columna}"] = extra_train[columna]
columnas_cat_c3 = [f"cat_{c}" for c in extra_train.columns]

pipeline_c3, indices_c3 = armar_pipeline_final_lab(columnas_cat_extra=columnas_cat_c3)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    salida = Pipeline(pipeline_c3.steps[:-1]).fit_transform(train.iloc[:2000], y.iloc[:2000])
print("Índices categóricos:", indices_c3, "| fila 0:", list(salida[0, indices_c3]))
assert salida.shape[1] == indices_c3[-1] + 1

probar("C3 - P6b + 4 categóricas crudas (moneda, status, tienda, tags)", pipeline=pipeline_c3,
       parametros_fit={"modelo__cat_features": indices_c3})
comparar_pareado(NOMBRE_FINAL, "C3 - P6b + 4 categóricas crudas (moneda, status, tienda, tags)")

Índices categóricos: [525, 526, 527, 528, 529, 530, 531, 532, 533, 534] | fila 0: ['4086929269', 'MLA73721', 'bronze', 'buy_it_now', 'not_specified', 'Buenos Aires', 'ARS', 'active', 'ninguna', 'dragged_bids_and_visits']


C3 - P6b + 4 categóricas crudas (moneda, status, tienda, tags) -> accuracy: 0.936 +/- 0.0016 | roc_auc: 0.9837 | minutos: 2.0
C3 - P6b + 4 categóricas crudas (moneda, status, tienda, tags)  vs  Final (armar_pipeline_final de features.py)
  diferencia por fold: [-0.0013 -0.0001 -0.0004 -0.0005 -0.0003]
  folds en que gana B: 0 de 5
  media de la diferencia: -0.0005 | desvío de la diferencia: 0.0004
  -> no pasa el criterio


**Conclusión de C3: no pasa, empeora un poco.** 0,9360 ± 0,0016 contra 0,9365: **pierde en los 5 folds** (media −0,0005, desvío 0,0004). Como no pasa, no hay *ablation*. Lectura (no medida): estas categóricas aportan poca señal nueva (tienda oficial ≈ vendedor, moneda en el 0,6% de las filas), y cada categórica extra le agrega a CatBoost más estadísticas por permutación y combinaciones de categóricas, o sea más ruido para elegir cortes. **No se suman.**

## C4 — Texto

**`text_features` de CatBoost (no está en el material).** CatBoost puede recibir columnas de texto. Por defecto (verificado con `get_all_params()`):
1. separa en palabras por espacios (**sin** pasar a minúsculas);
2. arma diccionarios de palabras y de pares de palabras (las que aparecen ≥ 5 veces, hasta 50.000);
3. las convierte en features numéricas: **BoW** (0/1 por palabra o par) y **Naive Bayes** (estimación de P(`new` | palabras)). Según la documentación, Naive Bayes usa el target con el mismo esquema ordenado que las categóricas, así que cada fila de train no ve su propia etiqueta. Naive Bayes como modelo está en `Presentación_NB_LDA_QDA.pdf`.

Le paso el título en minúsculas (`titulo_minusculas`, determinístico), igual que hace nuestro TF-IDF.

- **C4a**: P6b sin el TF-IDF de 500 (¿es redundante con `prob_texto_new`?).
- **C4b1**: P6b + título como `text_features` (con TF-IDF 500).
- **C4b2**: P6b + título como `text_features`, sin TF-IDF 500.

`prob_texto_new` se mantiene en las tres.

In [4]:
train["titulo_minusculas"] = train["title"].str.lower()


def con_texto_crudo(pipeline, indices_cat):
    """Agrega titulo_minusculas al final del ColumnTransformer como columna de texto
    para CatBoost. Devuelve (pipeline, parámetros del fit)."""
    pipeline.named_steps["preprocesador"].transformers.append(("texto_crudo", "passthrough", ["titulo_minusculas"]))
    indice_texto = indices_cat[-1] + 1
    return pipeline, {"modelo__cat_features": indices_cat, "modelo__text_features": [indice_texto]}


# C4a
pipeline_c4a, indices_c4a = armar_pipeline_final_lab(sacar={"tfidf"})
probar("C4a - P6b sin TF-IDF 500", pipeline=pipeline_c4a, parametros_fit={"modelo__cat_features": indices_c4a})
comparar_pareado(NOMBRE_FINAL, "C4a - P6b sin TF-IDF 500")
print()

# C4b1
pipeline_c4b1, parametros_c4b1 = con_texto_crudo(*armar_pipeline_final_lab())
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    salida = Pipeline(pipeline_c4b1.steps[:-1]).fit_transform(train.iloc[:2000], y.iloc[:2000])
print("Chequeo C4b1: categóricas", parametros_c4b1["modelo__cat_features"], "-> fila 0:",
      list(salida[0, parametros_c4b1["modelo__cat_features"]]), "| texto:", salida[0, parametros_c4b1["modelo__text_features"][0]])
probar("C4b1 - P6b + título como text_features", pipeline=pipeline_c4b1, parametros_fit=parametros_c4b1)
comparar_pareado(NOMBRE_FINAL, "C4b1 - P6b + título como text_features")
print()

# C4b2
pipeline_c4b2, parametros_c4b2 = con_texto_crudo(*armar_pipeline_final_lab(sacar={"tfidf"}))
probar("C4b2 - P6b + título como text_features, sin TF-IDF 500", pipeline=pipeline_c4b2, parametros_fit=parametros_c4b2)
comparar_pareado(NOMBRE_FINAL, "C4b2 - P6b + título como text_features, sin TF-IDF 500")

C4a - P6b sin TF-IDF 500 -> accuracy: 0.9365 +/- 0.0011 | roc_auc: 0.9838 | minutos: 1.0
C4a - P6b sin TF-IDF 500  vs  Final (armar_pipeline_final de features.py)
  diferencia por fold: [-0.0006  0.0014  0.0013 -0.0019 -0.0005]
  folds en que gana B: 2 de 5
  media de la diferencia: -0.0001 | desvío de la diferencia: 0.0012
  -> no pasa el criterio

Chequeo C4b1: categóricas [525, 526, 527, 528, 529, 530] -> fila 0: ['4086929269', 'MLA73721', 'bronze', 'buy_it_now', 'not_specified', 'Buenos Aires'] | texto: pizarron 20x30 + imagen + bolsita artesanal- yk42artesanias


C4b1 - P6b + título como text_features -> accuracy: 0.937 +/- 0.0013 | roc_auc: 0.9836 | minutos: 4.4
C4b1 - P6b + título como text_features  vs  Final (armar_pipeline_final de features.py)
  diferencia por fold: [-0.0001  0.0015  0.0001 -0.0005  0.0011]
  folds en que gana B: 3 de 5
  media de la diferencia: 0.0004 | desvío de la diferencia: 0.0008
  -> no pasa el criterio



C4b2 - P6b + título como text_features, sin TF-IDF 500 -> accuracy: 0.9368 +/- 0.0011 | roc_auc: 0.9838 | minutos: 3.5
C4b2 - P6b + título como text_features, sin TF-IDF 500  vs  Final (armar_pipeline_final de features.py)
  diferencia por fold: [-0.0007  0.0016  0.0004 -0.0011  0.0009]
  folds en que gana B: 3 de 5
  media de la diferencia: 0.0002 | desvío de la diferencia: 0.001
  -> no pasa el criterio


**Conclusión de C4: ninguna pasa.**

| Prueba | accuracy | folds vs P6b | tiempo |
|---|---|---|---|
| C4a: sin TF-IDF 500 | 0,9365 ± 0,0011 | 2/5, media −0,0001 | 1,0 min |
| C4b1: + título como `text_features` | 0,9370 ± 0,0013 | 3/5, media +0,0004, desvío 0,0008 | 4,4 min |
| C4b2: + `text_features`, sin TF-IDF 500 | 0,9368 ± 0,0011 | 3/5, media +0,0002 | 3,5 min |

- **C4a: el TF-IDF de 500 es redundante** con `prob_texto_new`. Sacarlo da la misma accuracy (0,9365) y entrena más rápido. No es una "mejora" (no pasa el criterio), pero sí una **simplificación posible**, a decidir en el cierre.
- **C4b (`text_features`)**: mejoran un poco en promedio, pero solo en 3 de 5 folds: ruido. El texto ya está bien aprovechado por el modelo de texto (`prob_texto_new`), y CatBoost no le saca mucho más.

## C5 — Vendedor simple sobre CatBoost

CatBoost ya codifica `seller_id` (estadísticas ordenadas + contador de frecuencia), pero perdió el precio promedio del vendedor al sacar `VendedorEncoder`. Dos columnas numéricas, **sin target**:
- `seller_avg_log_price`: precio promedio (log) de las publicaciones del vendedor, aprendido en el `fit` (adentro del `Pipeline`). `PrecioPromedioVendedor` reusa `VendedorEncoder(incluir_target=False)` y se queda solo con esa columna;
- `seller_n_total`: log(1 + publicaciones del vendedor en train + test) = V2 de `10_vendedor_v2` (determinística, afuera del `Pipeline`). Limitación: depende del lote de test, no serviría para una publicación nueva que llega sola.

Material: frequency encoder y *feature engineering* por grupos en `AP_Categóricas.ipynb`. Si las dos juntas pasan, *ablation*.

In [4]:
class PrecioPromedioVendedor(VendedorEncoder):
    """Solo seller_avg_log_price de VendedorEncoder (sin target)."""

    def __init__(self):
        super().__init__(incluir_target=False)

    def transform(self, X):
        return super().transform(X)[["seller_avg_log_price"]]

    def fit_transform(self, X, y=None):
        return super().fit_transform(X, y)[["seller_avg_log_price"]]


test_crudo = pd.read_json("../data/test_data.jsonlines", lines=True)
conteo_total = pd.concat([train["seller_id"], test_crudo["seller_id"]]).value_counts()
train["seller_n_total"] = np.log1p(train["seller_id"].map(conteo_total))


def armar_c5(con_precio=True, con_conteo=True):
    pipeline, indices = armar_pipeline_final_lab(columnas_numericas_extra=["seller_n_total"] if con_conteo else ())
    if con_precio:
        pipeline.named_steps["preprocesador"].transformers.append(
            ("precio_vendedor", PrecioPromedioVendedor(), ["seller_id", "log_price"]))
    return pipeline, {"modelo__cat_features": indices}


pipeline_c5, parametros_c5 = armar_c5()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    salida = Pipeline(pipeline_c5.steps[:-1]).fit_transform(train.iloc[:2000], y.iloc[:2000])
print("Chequeo: categóricas", parametros_c5["modelo__cat_features"], "->",
      list(salida[0, parametros_c5["modelo__cat_features"]]), "| última columna (precio vendedor):", salida[0, -1])

probar("C5 - P6b + precio promedio vendedor + conteo train+test", pipeline=pipeline_c5, parametros_fit=parametros_c5)
comparar_pareado(NOMBRE_FINAL, "C5 - P6b + precio promedio vendedor + conteo train+test")

Chequeo: categóricas [526, 527, 528, 529, 530, 531] -> ['4086929269', 'MLA73721', 'bronze', 'buy_it_now', 'not_specified', 'Buenos Aires'] | última columna (precio vendedor): 2.7093826463359885


C5 - P6b + precio promedio vendedor + conteo train+test -> accuracy: 0.9366 +/- 0.0018 | roc_auc: 0.9838 | minutos: 1.8
C5 - P6b + precio promedio vendedor + conteo train+test  vs  Final (armar_pipeline_final de features.py)
  diferencia por fold: [ 0.0003  0.0002 -0.0008 -0.0003  0.0011]
  folds en que gana B: 3 de 5
  media de la diferencia: 0.0001 | desvío de la diferencia: 0.0006
  -> no pasa el criterio


**Conclusión de C5: no pasa.** 0,9366 ± 0,0018 contra 0,9365: 3/5 folds, media +0,0001, desvío 0,0006. Ruido, así que no hay *ablation*. CatBoost ya resume al vendedor con `seller_id` crudo, y el precio promedio y el conteo con test no le agregan nada.

## Cierre de la ronda C1–C5

| Prueba | accuracy | folds vs P6b | tiempo | veredicto |
|---|---|---|---|---|
| **P6b (configuración final)** | **0,9365 ± 0,0017** | — | 1,6 min | — |
| C1: promedio de 5 semillas | 0,9367 ± 0,0012 | 3/5 | 6,4 min CV + 3 min test | no pasa; baja un poco el desvío |
| C2: tuning (mejor de 15) | 0,9364 ± 0,0013 | 2/5 | 54 min | no pasa: los valores por defecto ganan |
| C3: + 4 categóricas crudas | 0,9360 ± 0,0016 | 0/5 | 2,0 min | empeora |
| C4a: sin TF-IDF 500 | 0,9365 ± 0,0011 | 2/5 | 1,0 min | igual: el TF-IDF es redundante |
| C4b1 / C4b2: título como `text_features` | 0,9370 / 0,9368 | 3/5 | 4,4 / 3,5 min | no pasa |
| C5: + precio vendedor + conteo train+test | 0,9366 ± 0,0018 | 3/5 | 1,8 min | no pasa |

**Nada pasó el criterio, así que no hay combinación, *ablation* ni chequeo de robustez. La configuración final no cambia** (no se toca `features.py` y no hay `submission_catboost_v2.csv`).

**Robustez de P6b**, lo que esta ronda sí confirma:
- la semilla 42 no fue suerte (las semillas 42–46 dan 0,9364–0,9366);
- los hiperparámetros por defecto son tan buenos como cualquier combinación de la grilla;
- entre semillas cambia el 0,73% de las predicciones del test (~218 filas), así que diferencias de ~0,002–0,004 en el leaderboard pueden ser solo azar.

**Simplificación posible (no aplicada):** C4a muestra que el TF-IDF de 500 no aporta nada sobre `prob_texto_new`. Sacarlo daría un pipeline más simple con la misma accuracy. No se aplica sin decisión explícita, porque implicaría revalidar, cambiar `features.py` y regenerar la submission.

# Ronda P7 — Combinaciones de categóricas

`agregar_combos` (en `features.py`, código propio) arma 19 columnas `c_*`: combinaciones de categóricas como texto (vendedor × categoría, vendedor × tipo de publicación, categoría × bin de precio, vendedor × categoría × tipo, …) y algunas categóricas sueltas (moneda, status, tienda oficial, tags, métodos de pago, ids de atributos). No usa el target y es determinística: va afuera del `Pipeline`, después de `aplanar_anidadas`.

**Por qué combinar ayuda.** Un vendedor que vende celulares nuevos y ropa usada tiene, en total, 50% `new`. Con vendedor y categoría por separado, el modelo ve algo tibio. `c_seller_cat` le dice "este vendedor **en esta categoría**", y su codificación da 100% para celulares y 0% para ropa. Con muchos valores únicos (51.407 en `c_seller_cat`), la mayoría de los pares tiene 1 o 2 filas, y el *smoothing* los lleva al promedio. La ganancia viene de los pares con varias publicaciones. Material: target encoder y *feature engineering* (`AP_Categóricas.ipynb`).

**Por qué el `TargetEncoder` con *cross-fitting* no es leakage.** En el train del fold, cada fila se codifica con el % de `new` de su categoría en los **otros** 4/5 del train (nunca con su propia etiqueta), mezclado con el promedio global según cuántas filas tenga la categoría. Validación y test usan las estadísticas de todo el train del fold, y las etiquetas de validación nunca participan. Va adentro del `Pipeline`, así que se reajusta en cada fold.

## P7a — HistGB con `TargetEncoder` sobre vendedor, categoría y combinaciones

La Base HistGB (`armar_pipeline()`), sacando `VendedorEncoder` con target y el `TargetEncoder` de `category_id`, y sumando:
- `TargetEncoder(target_type="binary", random_state=42)` sobre `seller_id`, `category_id` y las 19 `c_*`;
- `VendedorEncoder(incluir_target=False)`: conteo y precio promedio del vendedor, sin target.

In [3]:
from features import agregar_combos, COLUMNAS_COMBOS

train = agregar_combos(train)
NOMBRE_FINAL = "Final (armar_pipeline_final de features.py)"

probar("P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target",
       sacar={"vendedor", "categoria"},
       transformers_extra=[
           ("te_combos", TargetEncoder(target_type="binary", random_state=SEED), ["seller_id", "category_id"] + COLUMNAS_COMBOS),
           ("vendedor_simple", VendedorEncoder(incluir_target=False), ["seller_id", "log_price"]),
       ])
comparar_pareado("Base", "P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target")
print()
comparar_pareado(NOMBRE_FINAL, "P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target")

P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target -> accuracy: 0.9371 +/- 0.0015 | roc_auc: 0.9838 | minutos: 0.7
P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target  vs  Base
  diferencia por fold: [0.0051 0.0036 0.0024 0.0019 0.0029]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.0032 | desvío de la diferencia: 0.0011
  -> MEJORA REAL

P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target  vs  Final (armar_pipeline_final de features.py)
  diferencia por fold: [ 0.0011  0.0011  0.0016 -0.0013  0.0002]
  folds en que gana B: 4 de 5
  media de la diferencia: 0.0006 | desvío de la diferencia: 0.001
  -> no pasa el criterio


**Conclusión de P7a.** 0,9371 ± 0,0015 en 0,7 minutos.
- **Contra la Base HistGB (0,9339): mejora real**, 5/5 folds, media +0,0032, desvío 0,0011. Con el mismo modelo, las combinaciones + `TargetEncoder` para el vendedor son mucho mejores que el *leave-one-out* de `VendedorEncoder`.
- **Contra CatBoost P6b (0,9365): no pasa.** 4/5 folds, media +0,0006 < desvío 0,0010. El HistGB con combinaciones **empata** a CatBoost, y entrena más del doble de rápido.
- Lectura (no medida): las combinaciones le dan al HistGB las interacciones vendedor × categoría que CatBoost arma solo con sus combinaciones internas de categóricas. Por eso ahora rinden parecido, y P7c (promediarlos) tiene más sentido que antes: codifican distinto.

## P7b — CatBoost P6b + las 19 `c_*` como categóricas crudas

6 + 19 = 25 `cat_features`. `armar_pipeline_final_lab(columnas_cat_extra=...)` recalcula los índices, y un chequeo con 2.000 filas confirma que en esas posiciones están las categóricas. Expectativa: puede empeorar, porque CatBoost ya combina categóricas internamente y 4 de las sueltas (moneda, status, tienda, tags) empeoraron en C3.

In [5]:
pipeline_p7b, indices_p7b = armar_pipeline_final_lab(columnas_cat_extra=COLUMNAS_COMBOS)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    salida = Pipeline(pipeline_p7b.steps[:-1]).fit_transform(train.iloc[:2000], y.iloc[:2000])
assert salida.shape[1] == indices_p7b[-1] + 1
print(len(indices_p7b), "categóricas, índices", indices_p7b[0], "a", indices_p7b[-1])
print("Fila 0, primeras 8:", list(salida[0, indices_p7b[:8]]))

probar("P7b - CatBoost P6b + 19 combos como categóricas crudas", pipeline=pipeline_p7b,
       parametros_fit={"modelo__cat_features": indices_p7b})
comparar_pareado(NOMBRE_FINAL, "P7b - CatBoost P6b + 19 combos como categóricas crudas")

25 categóricas, índices 525 a 549
Fila 0, primeras 8: ['4086929269', 'MLA73721', 'bronze', 'buy_it_now', 'not_specified', 'Buenos Aires', '4086929269_MLA73721', '4086929269_bronze']


P7b - CatBoost P6b + 19 combos como categóricas crudas -> accuracy: 0.9366 +/- 0.001 | roc_auc: 0.984 | minutos: 3.6
P7b - CatBoost P6b + 19 combos como categóricas crudas  vs  Final (armar_pipeline_final de features.py)
  diferencia por fold: [-0.0011  0.002  -0.0005 -0.0004  0.0002]
  folds en que gana B: 2 de 5
  media de la diferencia: 0.0001 | desvío de la diferencia: 0.0011
  -> no pasa el criterio


**Conclusión de P7b: no pasa.** 0,9366 ± 0,0010 (3,6 min), contra 0,9365. 2/5 folds, media +0,0001: ruido. No empeora (las 6 sueltas no le hicieron el daño que hicieron en C3), pero tampoco suma: CatBoost ya arma por dentro combinaciones de sus categóricas, así que las `c_*` le resultan redundantes. Como no empeoró, no hace falta P7b' (solo las 13 combinaciones).

## P7c — Promedio ponderado CatBoost P6b + HistGB P7a

Ahora rinden igual y **codifican distinto** (CatBoost: *ordered target statistics* sobre categóricas crudas; HistGB: `TargetEncoder` con *cross-fitting* sobre combinaciones explícitas). Pueden equivocarse en publicaciones distintas, que es lo que un ensamble necesita (`S8_Ensambles.ipynb`).

1. Probabilidades OOF: CatBoost P6b de `oof_ensambles.csv` (mismos folds); HistGB P7a con `cross_val_predict` (se guarda en `oof_p7a.csv`). Chequeo: sus accuracies por fold tienen que coincidir con el registro.
2. Correlación y % de desacuerdo entre los dos.
3. P(`new`) = w · CatBoost + (1 − w) · HistGB, con **w elegido en 4 folds** (grilla 0 a 1 de a 0,05; ante empate, el más alto) y **evaluado en el quinto**, rotando. Referencia: promedio fijo 50/50.
4. Comparación pareada contra CatBoost P6b solo.

In [5]:
from sklearn.model_selection import cross_val_predict

nombre_p7a = "P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target"
pipeline_p7a = armar_pipeline(sacar={"vendedor", "categoria"}, transformers_extra=[
    ("te_combos", TargetEncoder(target_type="binary", random_state=SEED), ["seller_id", "category_id"] + COLUMNAS_COMBOS),
    ("vendedor_simple", VendedorEncoder(incluir_target=False), ["seller_id", "log_price"]),
])
inicio = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    prob_p7a = cross_val_predict(pipeline_p7a, train, y, cv=cv, method="predict_proba", n_jobs=5)[:, 1]
print(f"OOF de P7a: {(time.time() - inicio) / 60:.1f} min")

oof_p7 = pd.read_csv(os.path.join(carpeta_lab, "oof_ensambles.csv"))[["fold", "y", "catboost"]]
oof_p7["histgb_p7a"] = prob_p7a
assert (oof_p7["y"].to_numpy() == y.to_numpy()).all()
oof_p7.to_csv(os.path.join(carpeta_lab, "oof_p7a.csv"), index=False)

registro = cargar_resultados()
for columna, nombre in [("catboost", NOMBRE_FINAL), ("histgb_p7a", nombre_p7a)]:
    por_fold = ((oof_p7[columna] >= 0.5).astype(int) == oof_p7["y"]).groupby(oof_p7["fold"]).mean().to_numpy()
    print(columna, "coincide con el registro:", np.allclose(por_fold, registro.loc[nombre, columnas_folds].to_numpy(dtype=float)))

clase_cat = (oof_p7["catboost"] >= 0.5).astype(int)
clase_hgb = (oof_p7["histgb_p7a"] >= 0.5).astype(int)
print("\nCorrelación de probabilidades CatBoost vs HistGB P7a:", round(oof_p7[["catboost", "histgb_p7a"]].corr().iloc[0, 1], 3),
      "(antes, con el HistGB de la Base: 0,991)")
print(f"Desacuerdo de clase: {(clase_cat != clase_hgb).mean():.1%} (antes: 2,0%)")
error_cat = clase_cat != oof_p7["y"]
print(f"De los {error_cat.sum()} errores de CatBoost, el HistGB P7a acierta el {(clase_hgb == oof_p7['y'])[error_cat].mean():.1%} (antes: 14,0%)")

OOF de P7a: 0.6 min
catboost coincide con el registro: True
histgb_p7a coincide con el registro: True

Correlación de probabilidades CatBoost vs HistGB P7a: 0.993 (antes, con el HistGB de la Base: 0,991)
Desacuerdo de clase: 1.9% (antes: 2,0%)
De los 4442 errores de CatBoost, el HistGB P7a acierta el 15.1% (antes: 14,0%)


In [6]:
grilla_w = np.round(np.arange(1.0, -0.001, -0.05), 2)
accuracy_catboost = registro.loc[NOMBRE_FINAL, columnas_folds].to_numpy(dtype=float)


def combinar(datos, w):
    return w * datos["catboost"] + (1 - w) * datos["histgb_p7a"]


filas_w, accuracy_ponderado, accuracy_50 = [], [], []
for k in range(1, 6):
    entrenamiento, prueba = oof_p7[oof_p7["fold"] != k], oof_p7[oof_p7["fold"] == k]
    accuracy_por_w = [((combinar(entrenamiento, w) >= 0.5).astype(int) == entrenamiento["y"]).mean() for w in grilla_w]
    w = grilla_w[int(np.argmax(accuracy_por_w))]
    accuracy_ponderado.append(((combinar(prueba, w) >= 0.5).astype(int) == prueba["y"]).mean())
    accuracy_50.append(((combinar(prueba, 0.5) >= 0.5).astype(int) == prueba["y"]).mean())
    filas_w.append({"fold": k, "w_catboost_elegido": w})
print("w de CatBoost elegido con los otros 4 folds:", [f["w_catboost_elegido"] for f in filas_w])

registrar("P7c - CatBoost P6b + HistGB P7a ponderado (w en 4 folds)", np.array(accuracy_ponderado), np.nan)
registrar("P7c (ref) - CatBoost P6b + HistGB P7a 50/50", np.array(accuracy_50), np.nan)
comparar_pareado(NOMBRE_FINAL, "P7c - CatBoost P6b + HistGB P7a ponderado (w en 4 folds)")
print()
comparar_pareado(NOMBRE_FINAL, "P7c (ref) - CatBoost P6b + HistGB P7a 50/50")

w de CatBoost elegido con los otros 4 folds: [np.float64(0.35), np.float64(0.55), np.float64(0.4), np.float64(0.35), np.float64(0.4)]
P7c - CatBoost P6b + HistGB P7a ponderado (w en 4 folds)  vs  Final (armar_pipeline_final de features.py)
  diferencia por fold: [0.0017 0.0016 0.0005 0.0001 0.0011]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.001 | desvío de la diferencia: 0.0006
  -> MEJORA REAL

P7c (ref) - CatBoost P6b + HistGB P7a 50/50  vs  Final (armar_pipeline_final de features.py)
  diferencia por fold: [0.0011 0.0017 0.0005 0.0006 0.0014]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.0011 | desvío de la diferencia: 0.0005
  -> MEJORA REAL


**P7c pasa** (semilla de CV 42): el ponderado y el 50/50 ganan en 5/5 folds (+0,0010 y +0,0011). El w elegido para CatBoost ronda 0,35–0,55, o sea casi mitad y mitad.

### Robustez de P7c: otra semilla de CV (`random_state=7`)

Mismo procedimiento con otros folds. Probabilidades OOF de CatBoost P6b y de HistGB P7a con `StratifiedKFold(5, shuffle=True, random_state=7)`; el ponderado elige w con 4 de esos folds y evalúa en el quinto. Comparación pareada contra CatBoost P6b con esa misma semilla (registro "P6b (cv semilla 7)", 0,9359).

In [6]:
cv7 = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
fold7 = np.empty(len(train), dtype=int)
for k, (_, indices_validacion) in enumerate(cv7.split(train, y)):
    fold7[indices_validacion] = k + 1

pipeline_cat7, indices_cat7 = armar_pipeline_final_lab()
inicio = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    prob_cat7 = cross_val_predict(pipeline_cat7, train, y, cv=cv7, method="predict_proba", n_jobs=5,
                                  params={"modelo__cat_features": indices_cat7})[:, 1]
    prob_hgb7 = cross_val_predict(pipeline_p7a, train, y, cv=cv7, method="predict_proba", n_jobs=5)[:, 1]
print(f"OOF con semilla 7: {(time.time() - inicio) / 60:.1f} min")

oof7 = pd.DataFrame({"fold": fold7, "y": y.to_numpy(), "catboost": prob_cat7, "histgb_p7a": prob_hgb7})
oof7.to_csv(os.path.join(carpeta_lab, "oof_p7_semilla7.csv"), index=False)


def accuracy_folds(datos, probabilidad):
    return ((probabilidad >= 0.5).astype(int) == datos["y"]).groupby(datos["fold"]).mean().to_numpy()


acc_cat7 = accuracy_folds(oof7, oof7["catboost"])
print("CatBoost semilla 7 coincide con el registro:",
      np.allclose(acc_cat7, cargar_resultados().loc["P6b (cv semilla 7)", columnas_folds].to_numpy(dtype=float)))

acc_pond7, w7 = [], []
for k in range(1, 6):
    entrenamiento, prueba = oof7[oof7["fold"] != k], oof7[oof7["fold"] == k]
    accuracy_por_w = [((combinar(entrenamiento, w) >= 0.5).astype(int) == entrenamiento["y"]).mean() for w in grilla_w]
    w = grilla_w[int(np.argmax(accuracy_por_w))]
    w7.append(w)
    acc_pond7.append(((combinar(prueba, w) >= 0.5).astype(int) == prueba["y"]).mean())
print("w de CatBoost elegido (semilla 7):", w7)

registrar("P7c ponderado (cv semilla 7)", np.array(acc_pond7), np.nan)
registrar("P7c 50/50 (cv semilla 7)", accuracy_folds(oof7, combinar(oof7, 0.5)), np.nan)
registrar("P7a HistGB (cv semilla 7)", accuracy_folds(oof7, oof7["histgb_p7a"]), np.nan)
for nombre in ["P7c ponderado (cv semilla 7)", "P7c 50/50 (cv semilla 7)", "P7a HistGB (cv semilla 7)"]:
    comparar_pareado("P6b (cv semilla 7)", nombre)
    print()

OOF con semilla 7: 2.8 min
CatBoost semilla 7 coincide con el registro: True
w de CatBoost elegido (semilla 7): [np.float64(0.55), np.float64(0.4), np.float64(0.6), np.float64(0.6), np.float64(0.6)]
P7c ponderado (cv semilla 7)  vs  P6b (cv semilla 7)
  diferencia por fold: [ 0.0005 -0.0006  0.0015  0.0016  0.0019]
  folds en que gana B: 4 de 5
  media de la diferencia: 0.001 | desvío de la diferencia: 0.0009
  -> MEJORA REAL

P7c 50/50 (cv semilla 7)  vs  P6b (cv semilla 7)
  diferencia por fold: [0.0007 0.0004 0.0018 0.0019 0.0019]
  folds en que gana B: 5 de 5
  media de la diferencia: 0.0013 | desvío de la diferencia: 0.0006
  -> MEJORA REAL

P7a HistGB (cv semilla 7)  vs  P6b (cv semilla 7)
  diferencia por fold: [-0.0004 -0.0014  0.0023  0.      0.0017]
  folds en que gana B: 2 de 5
  media de la diferencia: 0.0004 | desvío de la diferencia: 0.0014
  -> no pasa el criterio



**Conclusión de la ronda P7.**

| Prueba | CV semilla 42 | folds vs P6b | CV semilla 7 | folds vs P6b (semilla 7) |
|---|---|---|---|---|
| CatBoost P6b (referencia) | 0,9365 | — | 0,9359 | — |
| P7a: HistGB + `TargetEncoder`(vendedor, categoría, combos) | 0,9371 | 4/5, no pasa | 0,9364 | 2/5, no pasa |
| P7b: CatBoost + combos crudas | 0,9366 | 2/5, no pasa | — | — |
| P7c ponderado (w en 4 folds) | 0,9376 | **5/5** | 0,9369 | 4/5, media > desvío |
| **P7c 50/50** | **0,9376** | **5/5, +0,0011** | **0,9373** | **5/5, +0,0013** |

- **P7c 50/50 pasa con las dos semillas de CV.** Es la opción preferida: no tiene ningún parámetro elegido mirando datos (el w elegido igual quedaba en 0,35–0,6, casi mitad y mitad).
- La ganancia es chica: +0,001, unas 14 publicaciones por fold. Pero es consistente: 10 de 10 folds entre las dos semillas.
- Dato curioso: la correlación entre los dos modelos sigue alta (0,993, contra 0,991 con el HistGB viejo). Lo que cambió es que **ahora los dos son igual de buenos**. Antes el HistGB era peor y promediar diluía; ahora sus errores se compensan un poco.
- **Sesgo de selección:** en esta ronda se compararon 4 variantes contra P6b, pero P7c pasó también con folds nuevos (semilla 7). No pasa "justo".
- **Submission generada (sin subir):** `submission/submission_catboost_histgb.csv` (`notebooks/17_submission_p7c.ipynb`): 266 predicciones (0,9%) distintas de `submission_catboost.csv`. La parte CatBoost de esa notebook reproduce exactamente `submission_catboost.csv`.

# Ronda P8–P9 (referencia: P7c 50/50 = 0,9376; semilla 7: 0,9373)

Cada feature nueva se mide primero en el HistGB P7a (rápido). Si suma, se confirma en CatBoost P6b y en el promedio 50/50 contra P7c.

## P9 — Embeddings del título

**Qué es un embedding.** `paraphrase-multilingual-MiniLM-L12-v2` es una red neuronal ya entrenada (por otros, con millones de pares de frases en más de 50 idiomas) que convierte cada frase en 384 números, de modo que frases con **significado** parecido den vectores parecidos aunque usen palabras distintas ("celular nuevo en caja" ≈ "smartphone sellado"). Nosotros no la entrenamos: solo leemos su salida (*transfer learning*). `normalize_embeddings=True` deja cada vector con largo 1.

**Por qué no es leakage.** La red nunca vio nuestras etiquetas, y el embedding de cada título depende solo de ese título (determinístico, fila por fila). Por eso se calcula una vez, para train y test, afuera del `Pipeline`. Lo que aprende de nuestras etiquetas es la regresión logística de arriba, que va en `PrediccionOOF` (*cross-fitting*, solo con el train de cada fold).

**Material:** redes neuronales y embeddings **no** están en `material/` (solo aparece `MLPClassifier` comentado en `Arboles/Modelos.ipynb`).

Paso 1: verificar MPS y calcular los embeddings una vez (orden original de las filas), guardados en `.cache_laboratorio/`.

In [4]:
import torch
from sentence_transformers import SentenceTransformer

print("MPS disponible:", torch.backends.mps.is_available())
NOMBRE_MODELO_EMB = "paraphrase-multilingual-MiniLM-L12-v2"
archivo_emb_train = os.path.join(carpeta_lab, "emb_minilm_train.npy")
archivo_emb_test = os.path.join(carpeta_lab, "emb_minilm_test.npy")

if os.path.exists(archivo_emb_train) and os.path.exists(archivo_emb_test):
    emb_train, emb_test = np.load(archivo_emb_train), np.load(archivo_emb_test)
    print("Leídos de disco")
else:
    modelo_emb = SentenceTransformer(NOMBRE_MODELO_EMB, device="mps")
    titulos_test = pd.read_json("../data/test_data.jsonlines", lines=True)["title"].tolist()
    inicio = time.time()
    emb_train = modelo_emb.encode(train["title"].tolist(), batch_size=256, normalize_embeddings=True, show_progress_bar=False)
    emb_test = modelo_emb.encode(titulos_test, batch_size=256, normalize_embeddings=True, show_progress_bar=False)
    print(f"Embeddings de {len(emb_train) + len(emb_test)} títulos en {(time.time() - inicio) / 60:.1f} min")
    np.save(archivo_emb_train, emb_train)
    np.save(archivo_emb_test, emb_test)

print("train:", emb_train.shape, "| test:", emb_test.shape, "| norma de la fila 0:", round(float(np.linalg.norm(emb_train[0])), 4))

/Users/nicolasscravaglieri/Downloads/kaggle-ap-2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MPS disponible: True


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19437.52it/s]

Embeddings de 100000 títulos en 0.6 min
train: (70000, 384) | test: (30000, 384) | norma de la fila 0: 1.0


Ejemplo de que "significado parecido = vectores parecidos": similitud coseno (producto escalar, porque están normalizados) entre algunos títulos reales.

In [5]:
titulos_ejemplo = train["title"].iloc[:2000]
i = 0
similitudes = emb_train[:2000] @ emb_train[i]
orden = np.argsort(-similitudes)[1:4]
print("Título:", titulos_ejemplo.iloc[i])
for j in orden:
    print(f"  {similitudes[j]:.2f}  {titulos_ejemplo.iloc[j]}")
print(f"  {similitudes[np.argmin(similitudes)]:.2f}  (el menos parecido) {titulos_ejemplo.iloc[np.argmin(similitudes)]}")

Título: Pizarron 20x30 + Imagen + Bolsita Artesanal- Yk42artesanias
  0.67  Cubre Optica Zanella Zb G1
  0.63  Album Figuritas Bandana Vivir Intentando Faltan 39 Cromos
  0.63  Actua. Simp Ef. Gnp24/s4-f03 Genebre 15800-124 Art 5130001
  -0.00  (el menos parecido) Reglamento Internacional Para Prev Abordajes - 


### P9a, P9b, P9c en el HistGB P7a

- **P9a**: `prob_embed_new` = regresión logística (`C=1`) sobre los 384 números, en `PrediccionOOF`. Se suma a `prob_texto_new`.
- **P9b**: la regresión logística del modelo de texto (`C=2`) recibe TF-IDF (título palabras y caracteres + garantía) **y** los 384 números juntos. Su probabilidad **reemplaza** a `prob_texto_new`.
- **P9c**: los 384 números como columnas directas (referencia).

Cada una se compara contra el HistGB P7a solo (0,9371).

In [6]:
columnas_emb = [f"emb_{i}" for i in range(emb_train.shape[1])]
train = pd.concat([train, pd.DataFrame(emb_train, columns=columnas_emb, index=train.index)], axis=1)

nombre_p7a = "P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target"
extras_p7a = [
    ("te_combos", TargetEncoder(target_type="binary", random_state=SEED), ["seller_id", "category_id"] + COLUMNAS_COMBOS),
    ("vendedor_simple", VendedorEncoder(incluir_target=False), ["seller_id", "log_price"]),
]

# P9a
probar("P9a - HistGB P7a + prob_embed_new (logística sobre embeddings)", sacar={"vendedor", "categoria"},
       transformers_extra=extras_p7a + [
           ("embed", PrediccionOOF(LogisticRegression(C=1, max_iter=1000, random_state=SEED), random_state=SEED), columnas_emb),
       ])
comparar_pareado(nombre_p7a, "P9a - HistGB P7a + prob_embed_new (logística sobre embeddings)")
print()

# P9b
modelo_texto_y_embed = Pipeline([
    ("vectores", ColumnTransformer([
        ("palabras", TfidfVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode"), "title"),
        ("caracteres", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, strip_accents="unicode"), "title"),
        ("garantia", TfidfVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode"), "warranty_texto"),
        ("embeddings", "passthrough", columnas_emb),
    ])),
    ("modelo", LogisticRegression(C=2, max_iter=1000, random_state=SEED)),
])
probar("P9b - HistGB P7a con texto = TF-IDF + embeddings (reemplaza prob_texto_new)", sacar={"vendedor", "categoria", "texto"},
       transformers_extra=extras_p7a + [
           ("texto_embed", PrediccionOOF(modelo_texto_y_embed, random_state=SEED), ["title", "warranty_texto"] + columnas_emb),
       ])
comparar_pareado(nombre_p7a, "P9b - HistGB P7a con texto = TF-IDF + embeddings (reemplaza prob_texto_new)")
print()

# P9c
probar("P9c - HistGB P7a + 384 embeddings como columnas", sacar={"vendedor", "categoria"},
       transformers_extra=extras_p7a, columnas_numericas_extra=columnas_emb)
comparar_pareado(nombre_p7a, "P9c - HistGB P7a + 384 embeddings como columnas")

P9a - HistGB P7a + prob_embed_new (logística sobre embeddings) -> accuracy: 0.9367 +/- 0.0014 | roc_auc: 0.9837 | minutos: 0.7
P9a - HistGB P7a + prob_embed_new (logística sobre embeddings)  vs  P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target
  diferencia por fold: [ 0.0001  0.0003 -0.0005 -0.0012 -0.0009]
  folds en que gana B: 2 de 5
  media de la diferencia: -0.0004 | desvío de la diferencia: 0.0006
  -> no pasa el criterio



P9b - HistGB P7a con texto = TF-IDF + embeddings (reemplaza prob_texto_new) -> accuracy: 0.9361 +/- 0.0016 | roc_auc: 0.9837 | minutos: 1.6
P9b - HistGB P7a con texto = TF-IDF + embeddings (reemplaza prob_texto_new)  vs  P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target
  diferencia por fold: [-0.0025 -0.001  -0.0019 -0.0001  0.0007]
  folds en que gana B: 1 de 5
  media de la diferencia: -0.001 | desvío de la diferencia: 0.0012
  -> no pasa el criterio



P9c - HistGB P7a + 384 embeddings como columnas -> accuracy: 0.9358 +/- 0.0017 | roc_auc: 0.983 | minutos: 0.9
P9c - HistGB P7a + 384 embeddings como columnas  vs  P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target
  diferencia por fold: [-0.002  -0.0019 -0.0014 -0.0004 -0.0007]
  folds en que gana B: 0 de 5
  media de la diferencia: -0.0013 | desvío de la diferencia: 0.0006
  -> no pasa el criterio


**Diagnóstico:** ¿cuánto sabe la regresión logística sobre los embeddings **sola**, comparada con el modelo de texto actual sobre TF-IDF (0,8414 en `09_texto`, T3-ref)?

In [4]:
emb_solo = cross_validate(LogisticRegression(C=1, max_iter=1000, random_state=SEED), emb_train, y, cv=cv,
                          scoring="accuracy", n_jobs=5)["test_score"]
print("Regresión logística sobre embeddings sola:", round(emb_solo.mean(), 4), "+/-", round(emb_solo.std(), 4))

Regresión logística sobre embeddings sola: 0.7247 +/- 0.0046


**Conclusión de P9: ninguna variante suma. Los embeddings empeoran al HistGB P7a.**

| Prueba | accuracy | folds vs HistGB P7a (0,9371) | tiempo |
|---|---|---|---|
| P9a: + `prob_embed_new` | 0,9367 ± 0,0014 | 2/5, media −0,0004 | 0,7 min |
| P9b: texto = TF-IDF + embeddings (reemplaza `prob_texto_new`) | 0,9361 ± 0,0016 | 1/5, media −0,0010 | 1,6 min |
| P9c: 384 embeddings como columnas | 0,9358 ± 0,0017 | **0/5**, media −0,0013 | 0,9 min |

Cálculo de los embeddings de los 100.000 títulos en la GPU del M4: **0,6 minutos**.

- **Diagnóstico:** una regresión logística sobre los embeddings **sola** llega a **0,7247**, muy por debajo de la regresión logística sobre TF-IDF (0,8227 solo con el título, 0,8414 con título + garantía). Los embeddings de paráfrasis resumen **de qué producto se trata** (el "tema" del título), pero diluyen justo las palabras que separan `new`/`used` ("nuevo", "usado", "sin uso", "impecable") y los códigos de modelo. El ejemplo de vecinos más parecidos lo muestra: los "vecinos" de un pizarrón artesanal son una óptica de moto y un álbum de figuritas, títulos sin relación.
- P9c es la peor: 384 columnas densas y ruidosas le hacen elegir peores cortes al HistGB.
- **Como ninguna sumó, no se prueban los modelos más grandes** (mpnet, e5), que era el paso 4 condicional. **P9 no avanza a CatBoost ni al promedio.**

## P8 — URL de la miniatura

Las fotos ya no existen (29 de 30 URLs al azar redirigen a "imagen no disponible"), pero el **texto** de la URL de `thumbnail` trae información: en `.../653901-MLA1133955308_102015-I.jpg` la regex `/(\d+)-(MLA\d+)_(\d+)-` extrae número de imagen, MLA de la publicación dueña de la foto y un sufijo (¿MMYYYY = mes y año de subida?).

**No es leakage:** es partir un texto que ya viene en cada fila, sin usar `condition`. Es determinístico y va afuera del `Pipeline`. La fecha de subida de la foto es anterior a la publicación (no es información del futuro). Material: *feature engineering* (`AP_Categóricas.ipynb`); las expresiones regulares no están en el material.

### EDA

In [4]:
test_crudo = pd.read_json("../data/test_data.jsonlines", lines=True)


def partes_url(df):
    partes = df["thumbnail"].str.extract(r"/(\d+)-(MLA\d+)_(\d+)-")
    partes.columns = ["numero_imagen", "mla_foto", "sufijo"]
    return partes


url_train, url_test = partes_url(train), partes_url(test_crudo)
print(f"Matchean la regex: train {url_train['sufijo'].notna().mean():.2%} | test {url_test['sufijo'].notna().mean():.2%}")

largo_train = url_train["sufijo"].str.len()
print("\n% de filas y % new por largo del sufijo (train), y % de filas en test:")
print(pd.DataFrame({"pct_train": largo_train.value_counts(normalize=True, dropna=False),
                    "pct_new": y.groupby(largo_train.fillna(0)).mean().rename(index={0: np.nan}),
                    "pct_test": url_test["sufijo"].str.len().value_counts(normalize=True, dropna=False)}).round(3))

seis = largo_train == 6
mes = url_train.loc[seis, "sufijo"].str[:2].astype(int)
anio = url_train.loc[seis, "sufijo"].str[2:].astype(int)
print("\nSufijos de 6 dígitos: meses entre", mes.min(), "y", mes.max(), "| años entre", anio.min(), "y", anio.max())
print("% new por año de la foto:")
print(pd.DataFrame({"n": anio.value_counts().sort_index(), "pct_new": y[seis].groupby(anio).mean().round(3)}))

de_otra = url_train["mla_foto"].notna() & (url_train["mla_foto"] != train["id"])
print(f"\nFoto de OTRA publicación (MLA de la URL != id): {de_otra.mean():.2%} de las filas | % new: {y[de_otra].mean():.3f} "
      f"(resto: {y[~de_otra].mean():.3f}) | en test: {(url_test['mla_foto'].notna() & (url_test['mla_foto'] != test_crudo['id'])).mean():.2%}")

Matchean la regex: train 98.98% | test 98.93%

% de filas y % new por largo del sufijo (train), y % de filas en test:
        pct_train  pct_new  pct_test
sufijo                              
1.0         0.000    0.000     0.000
2.0         0.001    0.312     0.001
3.0         0.007    0.358     0.006
4.0         0.067    0.360     0.071
6.0         0.915    0.554     0.911
NaN         0.010    0.315     0.011

Sufijos de 6 dígitos: meses entre 1 y 12 | años entre 2011 y 2015
% new por año de la foto:
            n  pct_new
sufijo                
2011        4    1.000
2012     3295    0.421
2013     6837    0.479
2014    15281    0.572
2015    38659    0.571

Foto de OTRA publicación (MLA de la URL != id): 0.00% de las filas | % new: nan (resto: 0.537) | en test: 0.00%


### Las 4 features y la prueba en el HistGB P7a

- `url_largo_sufijo`: largo del sufijo (`NaN` si la URL no matchea);
- `url_mes_foto`, `url_anio_foto`: mes y año, solo si el sufijo tiene 6 dígitos;
- `url_meses_hasta_inicio`: meses entre la fecha de la foto y `start_time`;
- `url_foto_de_otra`: el MLA de la URL es distinto del `id` de la publicación.

Se suman como numéricas al HistGB P7a y se comparan contra el HistGB P7a solo.

In [5]:
def agregar_features_url(df):
    df = df.copy()
    partes = partes_url(df)
    largo = partes["sufijo"].str.len()
    seis = largo == 6
    df["url_largo_sufijo"] = largo
    df["url_mes_foto"] = np.where(seis, pd.to_numeric(partes["sufijo"].str[:2], errors="coerce"), np.nan)
    df["url_anio_foto"] = np.where(seis, pd.to_numeric(partes["sufijo"].str[2:], errors="coerce"), np.nan)
    meses_inicio = df["start_time"].dt.year * 12 + df["start_time"].dt.month
    df["url_meses_hasta_inicio"] = meses_inicio - (df["url_anio_foto"] * 12 + df["url_mes_foto"])
    df["url_foto_de_otra"] = (partes["mla_foto"].notna() & (partes["mla_foto"] != df["id"])).astype(int)
    return df


columnas_url = ["url_largo_sufijo", "url_mes_foto", "url_anio_foto", "url_meses_hasta_inicio", "url_foto_de_otra"]
train = agregar_features_url(train)
print(train[columnas_url].describe().round(2).to_string())

nombre_p7a = "P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target"
extras_p7a = [
    ("te_combos", TargetEncoder(target_type="binary", random_state=SEED), ["seller_id", "category_id"] + COLUMNAS_COMBOS),
    ("vendedor_simple", VendedorEncoder(incluir_target=False), ["seller_id", "log_price"]),
]
probar("P8 - HistGB P7a + features de la URL de la miniatura", sacar={"vendedor", "categoria"},
       transformers_extra=extras_p7a, columnas_numericas_extra=columnas_url)
comparar_pareado(nombre_p7a, "P8 - HistGB P7a + features de la URL de la miniatura")

       url_largo_sufijo  url_mes_foto  url_anio_foto  url_meses_hasta_inicio  url_foto_de_otra
count          69288.00      64076.00       64076.00                64076.00           70000.0
mean               5.84          6.92        2014.39                    9.38               0.0
std                0.57          2.89           0.87                   10.51               0.0
min                1.00          1.00        2011.00                  -26.00               0.0
25%                6.00          5.00        2014.00                    1.00               0.0
50%                6.00          7.00        2015.00                    5.00               0.0
75%                6.00          9.00        2015.00                   15.00               0.0
max                6.00         12.00        2015.00                   50.00               0.0


P8 - HistGB P7a + features de la URL de la miniatura -> accuracy: 0.9371 +/- 0.0018 | roc_auc: 0.9837 | minutos: 0.7
P8 - HistGB P7a + features de la URL de la miniatura  vs  P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target
  diferencia por fold: [-0.0001 -0.0005 -0.0014  0.0003  0.0016]
  folds en que gana B: 2 de 5
  media de la diferencia: -0.0 | desvío de la diferencia: 0.001
  -> no pasa el criterio


**Conclusión de P8: no suma.** 0,9371 ± 0,0018, igual que el HistGB P7a: 2/5 folds, media 0,0000.

- **EDA confirmado:**
  - el 99% de las URLs matchea la regex;
  - el sufijo de 6 dígitos es MMYYYY (meses 1–12, años 2011–2015);
  - los sufijos de 4 dígitos (6,7% de las filas) tienen **36% `new`** contra 55% de los de 6 dígitos;
  - por año de la foto, 2012 tiene 42% `new` y 2014–2015 57%.
- **La flag "foto de otra publicación" quedó siempre en 0:** en la URL de `thumbnail`, el MLA coincide **siempre** con el `id`. El ~1% con ~31% `new` que esperábamos corresponde, en realidad, a las filas cuya URL **no matchea** la regex (1,0% de las filas, 31,5% `new`). Son casi todas publicaciones sin fotos, que el modelo ya identifica con `n_fotos = 0` y las features de fotos en `NaN`.
- Lectura (no medida): la antigüedad de la foto ya está capturada por otras features (fotos de `11_anidadas`, `gap_actualizacion`, vendedor), así que no agrega información nueva.

**Cierre de la ronda P8–P9: nada pasó. El modelo final queda en P7c 50/50 (CatBoost P6b + HistGB P7a).**

### Ejemplo para la presentación: por qué los embeddings de paráfrasis no sirven acá

Similitud coseno entre títulos **ilustrativos** (inventados para el ejemplo), con el mismo modelo.

In [3]:
modelo_emb = SentenceTransformer(NOMBRE_MODELO_EMB, device="mps")
ejemplos = [
    "Iphone 5 16gb Nuevo En Caja Sellado",
    "Iphone 5 16gb Usado Con Detalles",
    "Zapatillas Nike Air Nuevas En Caja",
    "Zapatillas Nike Air Usadas Con Detalles",
]
vectores = modelo_emb.encode(ejemplos, normalize_embeddings=True)
pd.DataFrame(vectores @ vectores.T, index=ejemplos, columns=["iphone nuevo", "iphone usado", "nike nuevas", "nike usadas"]).round(2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8024.87it/s]

,iphone nuevo,iphone usado,nike nuevas,nike usadas
Iphone 5 16gb Nuevo En Caja Sellado,1.00,0.82,0.25,0.13
Iphone 5 16gb Usado Con Detalles,0.82,1.00,0.11,0.19
Zapatillas Nike Air Nuevas En Caja,0.25,0.11,1.00,0.85
Zapatillas Nike Air Usadas Con Detalles,0.13,0.19,0.85,1.00


**N1 no se completó.** La corrida (MLP sobre las 238.008 columnas del TF-IDF, ~15 millones de parámetros, 6 ajustes por fold) superó el límite de 2 horas sin terminar ni el MLP de texto solo. No hay resultados. Se decidió pasar a N2, con una entrada mucho más chica.

## N2 — MLP como tercer modelo del ensamble (Keras, como en S10 de SIA)

**Entrada:** las features del HistGB P7a (~560 columnas densas). Para la red se escalan adentro del `Pipeline`: numéricas con imputación por mediana + indicador de faltante (la red no acepta `NaN`) y `StandardScaler`; conteo y precio promedio del vendedor con `StandardScaler`. Los *target encodings*, el TF-IDF y `prob_texto_new` ya están entre 0 y 1.

**Red** (S10): `Dense(128, relu)` → `Dropout(0.3)` → `Dense(64, relu)` → `Dropout(0.3)` → `Dense(1, sigmoid)`, `binary_crossentropy`, `AdamW(learning_rate=1e-3)`, `batch_size=128`, `EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)` con una validación del 10% sacada **del propio train que recibe**. `MLPKeras` la envuelve con interfaz de sklearn (`fit` / `predict_proba`) para que vaya adentro del `Pipeline`. Keras corre en CPU (`tensorflow-metal` no es compatible con TensorFlow 2.21).

**No es leakage:** todo lo que aprende (escalado, imputación, `TargetEncoder` con *cross-fitting*, `PrediccionOOF`, la red) se ajusta con el train de cada fold. El *early stopping* usa una validación sacada de ese train. Los pesos del ensamble se eligen con 4 folds y se evalúan en el quinto.

Prueba previa: un ajuste en un fold tarda 0,2 min (14 épocas).

In [4]:
import os as _os
_os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer


class MLPKeras(ClassifierMixin, BaseEstimator):
    """MLP de Keras (como en S10 de SIA) con interfaz de sklearn.
    Dense(128, relu) -> Dropout(0.3) -> Dense(64, relu) -> Dropout(0.3) -> Dense(1, sigmoid),
    AdamW(lr=1e-3), binary cross-entropy, early stopping (patience=10, restore_best_weights)
    sobre una validación sacada del propio train que recibe en fit."""

    def __init__(self, capas=(128, 64), dropout=0.3, learning_rate=1e-3, batch_size=128, epocas=100,
                 paciencia=10, validation_fraction=0.1, random_state=42):
        self.capas = capas
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epocas = epocas
        self.paciencia = paciencia
        self.validation_fraction = validation_fraction
        self.random_state = random_state

    def fit(self, X, y):
        import tensorflow as tf
        import keras
        from keras import layers
        tf.config.threading.set_intra_op_parallelism_threads(2)
        keras.utils.set_random_seed(self.random_state)
        X = np.asarray(X, dtype="float32")
        y = np.asarray(y, dtype="float32")
        X_entrena, X_valida, y_entrena, y_valida = train_test_split(
            X, y, test_size=self.validation_fraction, stratify=y, random_state=self.random_state)
        capas = [layers.Input(shape=(X.shape[1],))]
        for neuronas in self.capas:
            capas += [layers.Dense(neuronas, activation="relu"), layers.Dropout(self.dropout)]
        capas.append(layers.Dense(1, activation="sigmoid"))
        self.modelo_ = keras.Sequential(capas)
        self.modelo_.compile(optimizer=keras.optimizers.AdamW(learning_rate=self.learning_rate),
                             loss="binary_crossentropy", metrics=["accuracy"])
        corte = keras.callbacks.EarlyStopping(monitor="val_loss", patience=self.paciencia, restore_best_weights=True)
        historia = self.modelo_.fit(X_entrena, y_entrena, epochs=self.epocas, batch_size=self.batch_size,
                                    validation_data=(X_valida, y_valida), callbacks=[corte], verbose=0)
        self.epocas_entrenadas_ = len(historia.history["val_loss"])
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        p = self.modelo_.predict(np.asarray(X, dtype="float32"), verbose=0, batch_size=4096).ravel()
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def armar_pipeline_mlp():
    """Features del HistGB P7a, escaladas para el MLP (adentro del Pipeline)."""
    preprocesador = ColumnTransformer([
        ("onehot", OneHotEncoder(handle_unknown="ignore"), ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]),
        ("num", Pipeline([("imputacion", SimpleImputer(strategy="median", add_indicator=True)),
                          ("escala", StandardScaler())]), COLUMNAS_NUMERICAS),
        ("tfidf", TfidfVectorizer(max_features=500), "title"),
        ("texto", Pipeline([("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
                            ("identidad", "passthrough")], memory=memoria_texto), ["title", "warranty_texto"]),
        ("te_combos", TargetEncoder(target_type="binary", random_state=SEED), ["seller_id", "category_id"] + COLUMNAS_COMBOS),
        ("vendedor_simple", Pipeline([("vendedor", VendedorEncoder(incluir_target=False)), ("escala", StandardScaler())]),
         ["seller_id", "log_price"]),
    ], sparse_threshold=0)
    return Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", preprocesador),
        ("modelo", MLPKeras(random_state=SEED)),
    ])

In [5]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

inicio = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    prob_mlp = cross_val_predict(armar_pipeline_mlp(), train, y, cv=cv, method="predict_proba", n_jobs=5)[:, 1]
minutos_mlp = (time.time() - inicio) / 60

oof_n2 = pd.read_csv(os.path.join(carpeta_lab, "oof_p7a.csv"))
assert (oof_n2["y"].to_numpy() == y.to_numpy()).all()
oof_n2["mlp"] = prob_mlp
oof_n2.to_csv(os.path.join(carpeta_lab, "oof_n2.csv"), index=False)


def acc_folds(probabilidad):
    return ((np.asarray(probabilidad) >= 0.5).astype(int) == oof_n2["y"]).groupby(oof_n2["fold"]).mean().to_numpy()


acc_mlp = acc_folds(oof_n2["mlp"])
registro = cargar_resultados()
registro.loc["N2 - MLP Keras solo (features HistGB P7a)"] = [
    pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), acc_mlp.mean(), acc_mlp.std(),
    np.mean([roc_auc_score(oof_n2.loc[oof_n2["fold"] == k, "y"], oof_n2.loc[oof_n2["fold"] == k, "mlp"]) for k in range(1, 6)]),
    *acc_mlp, minutos_mlp]
registro.to_csv(archivo_resultados)
print(f"MLP solo: accuracy {acc_mlp.mean():.4f} ± {acc_mlp.std():.4f} | {minutos_mlp:.1f} min")
print("Correlación de probabilidades:")
print(oof_n2[["catboost", "histgb_p7a", "mlp"]].corr().round(3))
clases = {c: (oof_n2[c] >= 0.5).astype(int) for c in ["catboost", "histgb_p7a", "mlp"]}
print(f"Desacuerdo MLP vs CatBoost: {(clases['mlp'] != clases['catboost']).mean():.1%} | "
      f"MLP vs HistGB P7a: {(clases['mlp'] != clases['histgb_p7a']).mean():.1%} | "
      f"CatBoost vs HistGB P7a: {(clases['catboost'] != clases['histgb_p7a']).mean():.1%}")

MLP solo: accuracy 0.9302 ± 0.0020 | 0.4 min
Correlación de probabilidades:
            catboost  histgb_p7a    mlp
catboost       1.000       0.993  0.986
histgb_p7a     0.993       1.000  0.984
mlp            0.986       0.984  1.000
Desacuerdo MLP vs CatBoost: 2.7% | MLP vs HistGB P7a: 2.9% | CatBoost vs HistGB P7a: 1.9%


### Ensambles de 3 modelos contra P7c (50/50 CatBoost + HistGB P7a)

- **Pesos iguales** (1/3 cada uno): no elige nada.
- **Pesos elegidos con 4 folds y evaluados en el quinto:** grilla de pesos de a 0,05 que suman 1 (231 combinaciones). En cada rotación se elige la de mayor accuracy en los otros 4 folds; ante empate, la más cercana a P7c (MLP con peso 0, mitad y mitad).

Comparación pareada contra P7c.

In [6]:
NOMBRE_P7C = "P7c (ref) - CatBoost P6b + HistGB P7a 50/50"
modelos3 = ["catboost", "histgb_p7a", "mlp"]
P = oof_n2[modelos3].to_numpy()

acc_iguales = acc_folds(P @ np.array([1 / 3, 1 / 3, 1 / 3]))

grilla = [np.array([a, b, round(1 - a - b, 2)]) for a in np.arange(0, 1.001, 0.05) for b in np.arange(0, 1.001 - a, 0.05)]
grilla = [np.round(w, 2) for w in grilla if w[2] >= -1e-9]
referencia = np.array([0.5, 0.5, 0.0])
grilla.sort(key=lambda w: np.abs(w - referencia).sum())

acc_elegidos, pesos_elegidos = [], []
for k in range(1, 6):
    entrena = (oof_n2["fold"] != k).to_numpy()
    prueba = ~entrena
    y_entrena, y_prueba = oof_n2["y"].to_numpy()[entrena], oof_n2["y"].to_numpy()[prueba]
    mejores = [((P[entrena] @ w >= 0.5).astype(int) == y_entrena).mean() for w in grilla]
    w = grilla[int(np.argmax(mejores))]
    pesos_elegidos.append(dict(zip(modelos3, w)))
    acc_elegidos.append(((P[prueba] @ w >= 0.5).astype(int) == y_prueba).mean())

registrar("N2 - promedio 3 modelos, pesos iguales", acc_iguales, np.nan)
registrar("N2 - 3 modelos, pesos elegidos en 4 folds", np.array(acc_elegidos), np.nan)
print("Pesos elegidos (CatBoost, HistGB P7a, MLP) por rotación:")
for pesos in pesos_elegidos:
    print("   ", pesos)
print()
for nombre in ["N2 - MLP Keras solo (features HistGB P7a)", "N2 - promedio 3 modelos, pesos iguales",
               "N2 - 3 modelos, pesos elegidos en 4 folds"]:
    comparar_pareado(NOMBRE_P7C, nombre)
    print()

Pesos elegidos (CatBoost, HistGB P7a, MLP) por rotación:
    {'catboost': np.float64(0.5), 'histgb_p7a': np.float64(0.45), 'mlp': np.float64(0.05)}
    {'catboost': np.float64(0.5), 'histgb_p7a': np.float64(0.45), 'mlp': np.float64(0.05)}
    {'catboost': np.float64(0.4), 'histgb_p7a': np.float64(0.55), 'mlp': np.float64(0.05)}
    {'catboost': np.float64(0.35), 'histgb_p7a': np.float64(0.65), 'mlp': np.float64(-0.0)}
    {'catboost': np.float64(0.4), 'histgb_p7a': np.float64(0.6), 'mlp': np.float64(-0.0)}

N2 - MLP Keras solo (features HistGB P7a)  vs  P7c (ref) - CatBoost P6b + HistGB P7a 50/50
  diferencia por fold: [-0.0067 -0.0074 -0.0088 -0.0081 -0.0059]
  folds en que gana B: 0 de 5
  media de la diferencia: -0.0074 | desvío de la diferencia: 0.001
  -> no pasa el criterio

N2 - promedio 3 modelos, pesos iguales  vs  P7c (ref) - CatBoost P6b + HistGB P7a 50/50
  diferencia por fold: [ 0.0001 -0.0007 -0.0014 -0.0001 -0.0005]
  folds en que gana B: 1 de 5
  media de la diferencia:

**Conclusión de N2: no pasa. El MLP no mejora al ensamble.**

| Prueba | accuracy | folds vs P7c (0,9376) | tiempo |
|---|---|---|---|
| MLP Keras solo (features HistGB P7a) | 0,9302 ± 0,0020 | 0/5, −0,0074 | 0,4 min |
| Promedio de 3 modelos, pesos iguales | 0,9371 | 1/5, −0,0005 | — |
| 3 modelos, pesos elegidos en 4 folds | 0,9375 | 1/5, −0,0001 | — |

- El MLP solo queda ~0,007 por debajo de los modelos de árboles. Es lo que suele pasar con datos tabulares: los *boosting* de árboles manejan mejor las features heterogéneas (conteos, flags, codificaciones) que una red chica.
- Es algo más "distinto" que los árboles (correlación 0,986 con CatBoost contra 0,993 entre CatBoost y HistGB; 2,7% de desacuerdo), pero es más flojo. Con pesos iguales diluye el ensamble. Cuando se eligen los pesos con 4 folds, le dan **0–5%** (prácticamente lo descarta) y queda igual que P7c.
- Mismo patrón que en `16_ensambles` y en la slide de S8: un modelo más débil no aporta aunque sea diverso.

**Cierre de la ronda N: nada pasó. El modelo final sigue siendo P7c 50/50.** N1 (MLP sobre TF-IDF de 238.008 columnas) no llegó a correr en el límite de 2 horas.

# Ronda N (segunda versión) — MLP y TabNet con PyTorch

Fuente principal: **AP**, `material/redes predictivo/Intro a Redes para datos estructurados.pdf`; SIA S10 como complemento.
- Slide 3: en datos tabulares, los árboles (XGBoost, HistGB, Random Forest) les ganan a las redes.
- Neurona z = w·x + b, y = f(z) (slide 5); red *fully connected* (slides 6–7).
- *Cross-entropy* (slide 9); descenso por gradiente, mini-batch y épocas (slides 10–14); *learning rate* (slide 16).
- **Adam** (slides 19–21: *"para la mayoría de nuestros casos, usaremos Adam"*; `SparseAdam` para tensores sparse).
- Normalización (slide 23) y dropout (slide 24); TabNet (slides 26–32).

Referencia: P7c 50/50 (0,9376).

## N1 — MLP de texto en PyTorch

Misma entrada que el modelo de texto actual (TF-IDF de título y garantía, 238.008 columnas sparse). La capa oculta es un `nn.EmbeddingBag` en modo suma con los valores del TF-IDF como pesos: matemáticamente es una capa *fully connected* sobre la matriz sparse, pero solo calcula las ~100 columnas no nulas de cada fila. Después: 64 neuronas → ReLU → Dropout 0,3 → 1 neurona (sigmoide en la pérdida, `BCEWithLogitsLoss`). Optimizadores: `SparseAdam` (capa sparse, slide 21) + Adam (resto), `lr=1e-3`, batch 256, *early stopping* con paciencia 5 sobre el 10% del train que recibe (se queda con los mejores pesos).

**No es leakage:** va adentro de `PrediccionOOF` (cada fila de train recibe la probabilidad de un MLP que no la vio), y la validación del *early stopping* sale del train que recibe el MLP, nunca del fold de validación de la CV.

**Prueba de tiempo previa:** un ajuste en el fold 1 tarda 0,4 min tanto en `mps` como en `cpu` (12 épocas). Se usa CPU, porque así los 5 folds corren en paralelo (la GPU no se comparte bien entre 5 procesos).

In [4]:
import torch
from torch import nn
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split


class _RedTexto(nn.Module):
    """Entrada sparse (TF-IDF) -> capa oculta (EmbeddingBag en modo suma = capa densa
    sobre las columnas no nulas) -> ReLU -> Dropout -> 1 neurona (logit)."""

    def __init__(self, n_columnas, neuronas, dropout):
        super().__init__()
        self.capa_oculta = nn.EmbeddingBag(n_columnas, neuronas, mode="sum", sparse=True)
        self.sesgo = nn.Parameter(torch.zeros(neuronas))
        self.dropout = nn.Dropout(dropout)
        self.salida = nn.Linear(neuronas, 1)

    def forward(self, indices, offsets, pesos):
        h = torch.relu(self.capa_oculta(indices, offsets, per_sample_weights=pesos) + self.sesgo)
        return self.salida(self.dropout(h)).squeeze(1)


class MLPTextoTorch(ClassifierMixin, BaseEstimator):
    """MLP de PyTorch para matrices sparse (TF-IDF). Adam (SparseAdam para la capa sparse),
    binary cross-entropy, early stopping sobre una validación sacada del propio train."""

    def __init__(self, neuronas=64, dropout=0.3, learning_rate=1e-3, batch_size=256, epocas=100,
                 paciencia=5, validation_fraction=0.1, device="cpu", random_state=42):
        self.neuronas = neuronas
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epocas = epocas
        self.paciencia = paciencia
        self.validation_fraction = validation_fraction
        self.device = device
        self.random_state = random_state

    def _lote(self, X):
        X = X.tocsr()
        return (torch.from_numpy(X.indices.astype(np.int64)).to(self.device),
                torch.from_numpy(X.indptr[:-1].astype(np.int64)).to(self.device),
                torch.from_numpy(X.data.astype(np.float32)).to(self.device))

    def _logits(self, X):
        self.red_.eval()
        salidas = []
        with torch.no_grad():
            for inicio in range(0, X.shape[0], 4096):
                salidas.append(self.red_(*self._lote(X[inicio:inicio + 4096])).cpu())
        return torch.cat(salidas)

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        torch.set_num_threads(2)
        y = np.asarray(y, dtype=np.float32)
        X_entrena, X_valida, y_entrena, y_valida = train_test_split(
            X, y, test_size=self.validation_fraction, stratify=y, random_state=self.random_state)
        self.red_ = _RedTexto(X.shape[1], self.neuronas, self.dropout).to(self.device)
        optimizador_sparse = torch.optim.SparseAdam(list(self.red_.capa_oculta.parameters()), lr=self.learning_rate)
        optimizador = torch.optim.Adam([self.red_.sesgo] + list(self.red_.salida.parameters()), lr=self.learning_rate)
        perdida = nn.BCEWithLogitsLoss()
        y_valida_t = torch.from_numpy(y_valida)
        generador = np.random.default_rng(self.random_state)
        mejor, mejor_estado, sin_mejora = np.inf, None, 0
        for epoca in range(self.epocas):
            self.red_.train()
            orden = generador.permutation(X_entrena.shape[0])
            for inicio in range(0, len(orden), self.batch_size):
                filas = orden[inicio:inicio + self.batch_size]
                logits = self.red_(*self._lote(X_entrena[filas]))
                valor = perdida(logits, torch.from_numpy(y_entrena[filas]).to(self.device))
                optimizador_sparse.zero_grad()
                optimizador.zero_grad()
                valor.backward()
                optimizador_sparse.step()
                optimizador.step()
            perdida_valida = perdida(self._logits(X_valida), y_valida_t).item()
            if perdida_valida < mejor - 1e-5:
                mejor, sin_mejora = perdida_valida, 0
                mejor_estado = {k: v.detach().clone() for k, v in self.red_.state_dict().items()}
            else:
                sin_mejora += 1
                if sin_mejora >= self.paciencia:
                    break
        self.red_.load_state_dict(mejor_estado)
        self.epocas_entrenadas_ = epoca + 1
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        p = torch.sigmoid(self._logits(X)).numpy()
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def armar_mlp_texto_torch(device="cpu"):
    return Pipeline([
        ("vectores", armar_modelo_texto().named_steps["vectores"]),
        ("modelo", MLPTextoTorch(device=device, random_state=SEED)),
    ])

In [5]:
inicio = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    solo = cross_validate(armar_mlp_texto_torch(), train, y, cv=cv, scoring=["accuracy", "roc_auc"], n_jobs=5)
print(f"MLP de texto solo: accuracy {solo['test_accuracy'].mean():.4f} ± {solo['test_accuracy'].std():.4f} | "
      f"roc_auc {solo['test_roc_auc'].mean():.4f} | {(time.time() - inicio) / 60:.1f} min "
      "(regresión logística sobre el mismo TF-IDF: 0,8414)")

MLP de texto solo: accuracy 0.8179 ± 0.0013 | roc_auc 0.8993 | 1.1 min (regresión logística sobre el mismo TF-IDF: 0,8414)


### N1a y N1b en el HistGB P7a

- **N1a**: `prob_mlp_texto` **reemplaza** a `prob_texto_new`.
- **N1b**: `prob_mlp_texto` se **suma** como feature extra.

Sin caché: la caché de joblib necesita *picklear* la clase `MLPTextoTorch`, y como está definida en la notebook (`__main__`) no puede (falló la primera corrida). Cada variante recalcula sus ajustes.

In [5]:
nombre_p7a = "P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target"
extras_p7a = [
    ("te_combos", TargetEncoder(target_type="binary", random_state=SEED), ["seller_id", "category_id"] + COLUMNAS_COMBOS),
    ("vendedor_simple", VendedorEncoder(incluir_target=False), ["seller_id", "log_price"]),
]


def mlp_torch_cacheado():
    return PrediccionOOF(armar_mlp_texto_torch(), random_state=SEED)


probar("N1a (torch) - HistGB P7a con prob_mlp_texto en lugar de prob_texto_new", sacar={"vendedor", "categoria", "texto"},
       transformers_extra=extras_p7a + [("texto_mlp", mlp_torch_cacheado(), ["title", "warranty_texto"])])
comparar_pareado(nombre_p7a, "N1a (torch) - HistGB P7a con prob_mlp_texto en lugar de prob_texto_new")
print()
probar("N1b (torch) - HistGB P7a + prob_mlp_texto como feature extra", sacar={"vendedor", "categoria"},
       transformers_extra=extras_p7a + [("texto_mlp", mlp_torch_cacheado(), ["title", "warranty_texto"])])
comparar_pareado(nombre_p7a, "N1b (torch) - HistGB P7a + prob_mlp_texto como feature extra")

N1a (torch) - HistGB P7a con prob_mlp_texto en lugar de prob_texto_new -> accuracy: 0.9348 +/- 0.0015 | roc_auc: 0.983 | minutos: 4.2
N1a (torch) - HistGB P7a con prob_mlp_texto en lugar de prob_texto_new  vs  P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target
  diferencia por fold: [-0.0026 -0.0019 -0.0034 -0.0026 -0.0009]
  folds en que gana B: 0 de 5
  media de la diferencia: -0.0023 | desvío de la diferencia: 0.0008
  -> no pasa el criterio



N1b (torch) - HistGB P7a + prob_mlp_texto como feature extra -> accuracy: 0.9367 +/- 0.0019 | roc_auc: 0.9836 | minutos: 4.4
N1b (torch) - HistGB P7a + prob_mlp_texto como feature extra  vs  P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target
  diferencia por fold: [-0.0008 -0.0013 -0.0012  0.0009  0.0005]
  folds en que gana B: 2 de 5
  media de la diferencia: -0.0004 | desvío de la diferencia: 0.0009
  -> no pasa el criterio


**Conclusión de N1 (PyTorch): no suma.**

| Prueba | accuracy | folds vs HistGB P7a (0,9371) | tiempo |
|---|---|---|---|
| MLP de texto solo | 0,8179 ± 0,0013 | (regresión logística sobre el mismo TF-IDF: 0,8414) | 1,1 min |
| N1a: `prob_mlp_texto` en lugar de `prob_texto_new` | 0,9348 ± 0,0015 | **0/5**, −0,0023 | 4,2 min |
| N1b: `prob_mlp_texto` como feature extra | 0,9367 ± 0,0019 | 2/5, −0,0004 | 4,4 min |

Sobre el mismo TF-IDF, el MLP rinde **menos** que la regresión logística. Con 15 millones de parámetros y 56.000 filas, sobreajusta antes de aprender combinaciones útiles (frena en ~12 épocas). Reemplazar el modelo de texto empeora, y sumarlo no agrega nada.

## N2 — TabNet como tercer modelo (slides 26–32)

Entrada: las features del HistGB P7a, normalizadas (slide 23) e imputadas adentro del `Pipeline`. Hiperparámetros sin tuning (slides 29–32): `n_d = n_a = 16`, `n_steps = 4`, `gamma = 1,5`, `lambda_sparse = 1e-4`, Adam `lr = 0,02`, `max_epochs = 100`, `patience = 10`, `batch_size = 1024` (`virtual_batch_size = 128`). *Early stopping* sobre el 10% del train que recibe.

**No es leakage:** escalado, imputación, `TargetEncoder` (*cross-fitting*), `PrediccionOOF` y TabNet se ajustan con el train de cada fold. El *early stopping* usa una validación de ese train. Los pesos del ensamble se eligen con 4 folds y se evalúan en el quinto.

**Prueba de tiempo previa (fold 1):** `mps` 2,1 min (35 épocas, accuracy 0,9260); `cpu` 3,5 min. Se usa MPS, con los folds de a uno.

In [4]:
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer


class TabNetEnvuelto(ClassifierMixin, BaseEstimator):
    """TabNet (pytorch-tabnet) con interfaz de sklearn. Early stopping (patience) sobre una
    validación sacada del propio train que recibe en fit. Adam como optimizador."""

    def __init__(self, n_d=16, n_steps=4, gamma=1.5, lambda_sparse=1e-4, lr=2e-2, max_epochs=100,
                 patience=10, batch_size=1024, virtual_batch_size=128, validation_fraction=0.1,
                 device="cpu", random_state=42):
        self.n_d = n_d
        self.n_steps = n_steps
        self.gamma = gamma
        self.lambda_sparse = lambda_sparse
        self.lr = lr
        self.max_epochs = max_epochs
        self.patience = patience
        self.batch_size = batch_size
        self.virtual_batch_size = virtual_batch_size
        self.validation_fraction = validation_fraction
        self.device = device
        self.random_state = random_state

    def fit(self, X, y):
        torch.set_num_threads(2)
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y).astype(int)
        X_entrena, X_valida, y_entrena, y_valida = train_test_split(
            X, y, test_size=self.validation_fraction, stratify=y, random_state=self.random_state)
        self.modelo_ = TabNetClassifier(n_d=self.n_d, n_a=self.n_d, n_steps=self.n_steps, gamma=self.gamma,
                                        lambda_sparse=self.lambda_sparse, optimizer_fn=torch.optim.Adam,
                                        optimizer_params={"lr": self.lr}, seed=self.random_state, verbose=0,
                                        device_name=self.device)
        self.modelo_.fit(X_entrena, y_entrena, eval_set=[(X_valida, y_valida)], eval_metric=["logloss"],
                         max_epochs=self.max_epochs, patience=self.patience, batch_size=self.batch_size,
                         virtual_batch_size=self.virtual_batch_size)
        self.epocas_entrenadas_ = len(self.modelo_.history["loss"])
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        return self.modelo_.predict_proba(np.asarray(X, dtype=np.float32))

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def armar_pipeline_tabnet(device="cpu"):
    """Features del HistGB P7a, escaladas (normalización, slide 23) e imputadas dentro del Pipeline."""
    preprocesador = ColumnTransformer([
        ("onehot", OneHotEncoder(handle_unknown="ignore"), ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]),
        ("num", Pipeline([("imputacion", SimpleImputer(strategy="median", add_indicator=True)),
                          ("escala", StandardScaler())]), COLUMNAS_NUMERICAS),
        ("tfidf", TfidfVectorizer(max_features=500), "title"),
        ("texto", Pipeline([("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
                            ("identidad", "passthrough")], memory=memoria_texto), ["title", "warranty_texto"]),
        ("te_combos", TargetEncoder(target_type="binary", random_state=SEED), ["seller_id", "category_id"] + COLUMNAS_COMBOS),
        ("vendedor_simple", Pipeline([("vendedor", VendedorEncoder(incluir_target=False)), ("escala", StandardScaler())]),
         ["seller_id", "log_price"]),
    ], sparse_threshold=0)
    return Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", preprocesador),
        ("modelo", TabNetEnvuelto(device=device, random_state=SEED)),
    ])

In [5]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

inicio = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    prob_tabnet = cross_val_predict(armar_pipeline_tabnet("mps"), train, y, cv=cv, method="predict_proba", n_jobs=1)[:, 1]
minutos_tabnet = (time.time() - inicio) / 60

oof_tab = pd.read_csv(os.path.join(carpeta_lab, "oof_p7a.csv"))
assert (oof_tab["y"].to_numpy() == y.to_numpy()).all()
oof_tab["tabnet"] = prob_tabnet
oof_tab.to_csv(os.path.join(carpeta_lab, "oof_tabnet.csv"), index=False)


def acc_folds(probabilidad):
    return ((np.asarray(probabilidad) >= 0.5).astype(int) == oof_tab["y"]).groupby(oof_tab["fold"]).mean().to_numpy()


acc_tab = acc_folds(oof_tab["tabnet"])
registrar("N2 - TabNet solo (features HistGB P7a)", acc_tab, minutos_tabnet,
          roc_auc=np.mean([roc_auc_score(oof_tab.loc[oof_tab["fold"] == k, "y"], oof_tab.loc[oof_tab["fold"] == k, "tabnet"]) for k in range(1, 6)]))
print(f"TabNet solo: accuracy {acc_tab.mean():.4f} ± {acc_tab.std():.4f} | {minutos_tabnet:.1f} min")
print(oof_tab[["catboost", "histgb_p7a", "tabnet"]].corr().round(3))

NOMBRE_P7C = "P7c (ref) - CatBoost P6b + HistGB P7a 50/50"
modelos3 = ["catboost", "histgb_p7a", "tabnet"]
P = oof_tab[modelos3].to_numpy()
registrar("N2 - CatBoost + HistGB P7a + TabNet, pesos iguales", acc_folds(P @ np.array([1 / 3, 1 / 3, 1 / 3])), np.nan)

grilla = [np.round(np.array([a, b, 1 - a - b]), 2) for a in np.arange(0, 1.001, 0.05) for b in np.arange(0, 1.001 - a, 0.05)]
grilla = [w for w in grilla if w[2] >= -1e-9]
grilla.sort(key=lambda w: np.abs(w - np.array([0.5, 0.5, 0.0])).sum())
acc_elegidos, pesos_elegidos = [], []
for k in range(1, 6):
    entrena = (oof_tab["fold"] != k).to_numpy()
    y_arr = oof_tab["y"].to_numpy()
    w = grilla[int(np.argmax([((P[entrena] @ w >= 0.5).astype(int) == y_arr[entrena]).mean() for w in grilla]))]
    pesos_elegidos.append(dict(zip(modelos3, w)))
    acc_elegidos.append(((P[~entrena] @ w >= 0.5).astype(int) == y_arr[~entrena]).mean())
registrar("N2 - CatBoost + HistGB P7a + TabNet, pesos elegidos en 4 folds", np.array(acc_elegidos), np.nan)
print("Pesos elegidos (CatBoost, HistGB P7a, TabNet):", pesos_elegidos)
for nombre in ["N2 - TabNet solo (features HistGB P7a)", "N2 - CatBoost + HistGB P7a + TabNet, pesos iguales",
               "N2 - CatBoost + HistGB P7a + TabNet, pesos elegidos en 4 folds"]:
    comparar_pareado(NOMBRE_P7C, nombre)
    print()


Early stopping occurred at epoch 34 with best_epoch = 24 and best_val_0_logloss = 0.20141



Early stopping occurred at epoch 33 with best_epoch = 23 and best_val_0_logloss = 0.17978



Early stopping occurred at epoch 29 with best_epoch = 19 and best_val_0_logloss = 0.19657



Early stopping occurred at epoch 27 with best_epoch = 17 and best_val_0_logloss = 0.19893



Early stopping occurred at epoch 33 with best_epoch = 23 and best_val_0_logloss = 0.19886


TabNet solo: accuracy 0.9254 ± 0.0006 | 9.7 min
            catboost  histgb_p7a  tabnet
catboost       1.000       0.993   0.977
histgb_p7a     0.993       1.000   0.975
tabnet         0.977       0.975   1.000


Pesos elegidos (CatBoost, HistGB P7a, TabNet): [{'catboost': np.float64(0.45), 'histgb_p7a': np.float64(0.5), 'tabnet': np.float64(0.05)}, {'catboost': np.float64(0.45), 'histgb_p7a': np.float64(0.5), 'tabnet': np.float64(0.05)}, {'catboost': np.float64(0.35), 'histgb_p7a': np.float64(0.6), 'tabnet': np.float64(0.05)}, {'catboost': np.float64(0.35), 'histgb_p7a': np.float64(0.65), 'tabnet': np.float64(-0.0)}, {'catboost': np.float64(0.35), 'histgb_p7a': np.float64(0.6), 'tabnet': np.float64(0.05)}]
N2 - TabNet solo (features HistGB P7a)  vs  P7c (ref) - CatBoost P6b + HistGB P7a 50/50
  diferencia por fold: [-0.0124 -0.0091 -0.0129 -0.0136 -0.0131]
  folds en que gana B: 0 de 5
  media de la diferencia: -0.0122 | desvío de la diferencia: 0.0016
  -> no pasa el criterio

N2 - CatBoost + HistGB P7a + TabNet, pesos iguales  vs  P7c (ref) - CatBoost P6b + HistGB P7a 50/50
  diferencia por fold: [-0.0009 -0.0011 -0.0002 -0.0013 -0.0014]
  folds en que gana B: 0 de 5
  media de la diferencia

**Conclusión de N2 (TabNet): no pasa.** TabNet solo: **0,9254 ± 0,0006** (9,7 min en MPS), 0/5 contra P7c (−0,0122). Correlación con CatBoost 0,977. Con pesos iguales, el ensamble de 3 empeora (0/5, −0,0010). Con pesos elegidos en 4 folds, TabNet recibe 0–5% y el resultado queda igual que P7c (3/5, media 0,0000). Coherente con la slide 3 del PDF de AP: en tabulares, los árboles les ganan a las redes.

## N3 — MLP simple de PyTorch como tercer modelo (porque N2 rindió mal)

Mismas features escaladas que TabNet. `Linear(128)` → ReLU → `Dropout(0,3)` → `Linear(64)` → ReLU → `Dropout(0,3)` → `Linear(1)`, Adam `lr=1e-3` (slides 19–21), batch 128, *early stopping* con paciencia 10 sobre el 10% del train que recibe. Mismo esquema de ensamble. (Equivale al MLP de Keras de la primera versión de la ronda, que dio 0,9302.) No es leakage por lo mismo que N2.

Prueba de tiempo previa (fold 1, CPU): 0,1 min, 17 épocas, accuracy 0,9330.

In [4]:
import torch
from torch import nn
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer


class MLPTorch(ClassifierMixin, BaseEstimator):
    """MLP denso de PyTorch: Linear(128) -> ReLU -> Dropout(0,3) -> Linear(64) -> ReLU -> Dropout(0,3) -> Linear(1).
    Adam, binary cross-entropy, early stopping sobre una validación sacada del propio train."""

    def __init__(self, capas=(128, 64), dropout=0.3, learning_rate=1e-3, batch_size=128, epocas=100,
                 paciencia=10, validation_fraction=0.1, device="cpu", random_state=42):
        self.capas = capas
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epocas = epocas
        self.paciencia = paciencia
        self.validation_fraction = validation_fraction
        self.device = device
        self.random_state = random_state

    def _probabilidad(self, X):
        self.red_.eval()
        with torch.no_grad():
            return torch.sigmoid(self.red_(torch.from_numpy(X).to(self.device)).squeeze(1)).cpu().numpy()

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        torch.set_num_threads(2)
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        X_entrena, X_valida, y_entrena, y_valida = train_test_split(
            X, y, test_size=self.validation_fraction, stratify=y, random_state=self.random_state)
        capas, entrada = [], X.shape[1]
        for neuronas in self.capas:
            capas += [nn.Linear(entrada, neuronas), nn.ReLU(), nn.Dropout(self.dropout)]
            entrada = neuronas
        capas.append(nn.Linear(entrada, 1))
        self.red_ = nn.Sequential(*capas).to(self.device)
        optimizador = torch.optim.Adam(self.red_.parameters(), lr=self.learning_rate)
        perdida = nn.BCEWithLogitsLoss()
        Xe, ye = torch.from_numpy(X_entrena).to(self.device), torch.from_numpy(y_entrena).to(self.device)
        Xv, yv = torch.from_numpy(X_valida).to(self.device), torch.from_numpy(y_valida).to(self.device)
        generador = torch.Generator().manual_seed(self.random_state)
        mejor, mejor_estado, sin_mejora = np.inf, None, 0
        for epoca in range(self.epocas):
            self.red_.train()
            orden = torch.randperm(len(Xe), generator=generador)
            for inicio in range(0, len(orden), self.batch_size):
                filas = orden[inicio:inicio + self.batch_size]
                optimizador.zero_grad()
                valor = perdida(self.red_(Xe[filas]).squeeze(1), ye[filas])
                valor.backward()
                optimizador.step()
            self.red_.eval()
            with torch.no_grad():
                perdida_valida = perdida(self.red_(Xv).squeeze(1), yv).item()
            if perdida_valida < mejor - 1e-5:
                mejor, sin_mejora = perdida_valida, 0
                mejor_estado = {k: v.detach().clone() for k, v in self.red_.state_dict().items()}
            else:
                sin_mejora += 1
                if sin_mejora >= self.paciencia:
                    break
        self.red_.load_state_dict(mejor_estado)
        self.epocas_entrenadas_ = epoca + 1
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        p = self._probabilidad(np.asarray(X, dtype=np.float32))
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def armar_pipeline_mlp_torch():
    preprocesador = ColumnTransformer([
        ("onehot", OneHotEncoder(handle_unknown="ignore"), ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]),
        ("num", Pipeline([("imputacion", SimpleImputer(strategy="median", add_indicator=True)),
                          ("escala", StandardScaler())]), COLUMNAS_NUMERICAS),
        ("tfidf", TfidfVectorizer(max_features=500), "title"),
        ("texto", Pipeline([("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
                            ("identidad", "passthrough")], memory=memoria_texto), ["title", "warranty_texto"]),
        ("te_combos", TargetEncoder(target_type="binary", random_state=SEED), ["seller_id", "category_id"] + COLUMNAS_COMBOS),
        ("vendedor_simple", Pipeline([("vendedor", VendedorEncoder(incluir_target=False)), ("escala", StandardScaler())]),
         ["seller_id", "log_price"]),
    ], sparse_threshold=0)
    return Pipeline([
        ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
        ("logs", FunctionTransformer(agregar_logs)),
        ("preprocesador", preprocesador),
        ("modelo", MLPTorch(random_state=SEED)),
    ])

In [5]:
inicio = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    prob_mlp_torch = cross_val_predict(armar_pipeline_mlp_torch(), train, y, cv=cv, method="predict_proba", n_jobs=5)[:, 1]
minutos_mlp_torch = (time.time() - inicio) / 60
oof_tab["mlp_torch"] = prob_mlp_torch
oof_tab.to_csv(os.path.join(carpeta_lab, "oof_tabnet.csv"), index=False)

acc_mlp_torch = acc_folds(oof_tab["mlp_torch"])
registrar("N3 - MLP PyTorch solo (features HistGB P7a)", acc_mlp_torch, minutos_mlp_torch)
print(f"MLP PyTorch solo: accuracy {acc_mlp_torch.mean():.4f} ± {acc_mlp_torch.std():.4f} | {minutos_mlp_torch:.1f} min")

modelos3 = ["catboost", "histgb_p7a", "mlp_torch"]
P = oof_tab[modelos3].to_numpy()
registrar("N3 - CatBoost + HistGB P7a + MLP PyTorch, pesos iguales", acc_folds(P @ np.array([1 / 3, 1 / 3, 1 / 3])), np.nan)
acc_elegidos, pesos_elegidos = [], []
y_arr = oof_tab["y"].to_numpy()
for k in range(1, 6):
    entrena = (oof_tab["fold"] != k).to_numpy()
    w = grilla[int(np.argmax([((P[entrena] @ w >= 0.5).astype(int) == y_arr[entrena]).mean() for w in grilla]))]
    pesos_elegidos.append(dict(zip(modelos3, w)))
    acc_elegidos.append(((P[~entrena] @ w >= 0.5).astype(int) == y_arr[~entrena]).mean())
registrar("N3 - CatBoost + HistGB P7a + MLP PyTorch, pesos elegidos en 4 folds", np.array(acc_elegidos), np.nan)
print("Pesos elegidos (CatBoost, HistGB P7a, MLP):", pesos_elegidos)
for nombre in ["N3 - MLP PyTorch solo (features HistGB P7a)", "N3 - CatBoost + HistGB P7a + MLP PyTorch, pesos iguales",
               "N3 - CatBoost + HistGB P7a + MLP PyTorch, pesos elegidos en 4 folds"]:
    comparar_pareado(NOMBRE_P7C, nombre)
    print()

MLP PyTorch solo: accuracy 0.9313 ± 0.0018 | 0.3 min


Pesos elegidos (CatBoost, HistGB P7a, MLP): [{'catboost': np.float64(0.25), 'histgb_p7a': np.float64(0.6), 'mlp_torch': np.float64(0.15)}, {'catboost': np.float64(0.35), 'histgb_p7a': np.float64(0.5), 'mlp_torch': np.float64(0.15)}, {'catboost': np.float64(0.35), 'histgb_p7a': np.float64(0.5), 'mlp_torch': np.float64(0.15)}, {'catboost': np.float64(0.35), 'histgb_p7a': np.float64(0.65), 'mlp_torch': np.float64(-0.0)}, {'catboost': np.float64(0.35), 'histgb_p7a': np.float64(0.5), 'mlp_torch': np.float64(0.15)}]
N3 - MLP PyTorch solo (features HistGB P7a)  vs  P7c (ref) - CatBoost P6b + HistGB P7a 50/50
  diferencia por fold: [-0.0054 -0.007  -0.0062 -0.0067 -0.0061]
  folds en que gana B: 0 de 5
  media de la diferencia: -0.0063 | desvío de la diferencia: 0.0006
  -> no pasa el criterio

N3 - CatBoost + HistGB P7a + MLP PyTorch, pesos iguales  vs  P7c (ref) - CatBoost P6b + HistGB P7a 50/50
  diferencia por fold: [ 0.0009 -0.0001 -0.0009 -0.0002 -0.0004]
  folds en que gana B: 1 de 5
  

**Conclusión de N3: no pasa.** MLP PyTorch solo: 0,9313 ± 0,0018 (0,3 min), 0/5 contra P7c (−0,0063). Ensamble de 3 con pesos iguales: 0,9375 (1/5). Con pesos elegidos en 4 folds: 0,9377 (3/5, media +0,0001, desvío 0,0004; el MLP recibe 0–15%) → ruido.

## Cierre de la ronda N (redes neuronales): nada pasó. El modelo final sigue siendo P7c 50/50.

| Prueba | accuracy | vs referencia | tiempo |
|---|---|---|---|
| MLP de texto solo (PyTorch, sparse) | 0,8179 | regresión logística sobre el mismo TF-IDF: 0,8414 | 1,1 min |
| N1a: MLP de texto en lugar de `prob_texto_new` (HistGB P7a) | 0,9348 | 0/5 vs P7a | 4,2 min |
| N1b: MLP de texto como feature extra (HistGB P7a) | 0,9367 | 2/5 vs P7a | 4,4 min |
| TabNet solo | 0,9254 | 0/5 vs P7c | 9,7 min (MPS) |
| CatBoost + HistGB + TabNet (pesos elegidos) | 0,9376 | 3/5, ruido | — |
| MLP PyTorch solo | 0,9313 | 0/5 vs P7c | 0,3 min |
| CatBoost + HistGB + MLP (pesos elegidos) | 0,9377 | 3/5, ruido | — |

Las redes quedan entre 0,6 y 1,2 puntos por debajo de los árboles, y como terceras en el ensamble reciben pesos de 0–15% sin mejorar a P7c. Es exactamente lo que muestra la slide 3 del PDF de AP (*"¿Redes para datos tabulares?"*): en clasificación con datos tabulares, XGBoost, HistGradientBoosting y Random Forest les ganan a las redes. Buen material para la parte de "selección de modelos" de la presentación.

# Ronda final de ideas (I1–I4), contra P7c 50/50 (0,9376)

## I1 — kNN de títulos

Para cada publicación: **% de `new` entre los k títulos más parecidos del train** (k = 10 y 30, dos columnas). Es la versión de regresión de kNN (`KNeighborsRegressor`, `material/Knn/Copia de AP_kNN.ipynb`: promediar el target de los vecinos) usada como feature.
- **Parecido = similitud coseno** sobre el TF-IDF del título (palabras y pares, `min_df=2`). El coseno no está en el material, pero como el TF-IDF tiene largo 1, ordenar por coseno da **los mismos vecinos que la distancia euclídea** del material (distancia² = 2 − 2·coseno).
- **(a)** vecinos en todo el train; **(b)** solo entre publicaciones de la misma `category_id` (sin vecinos en la categoría → `NaN`, ~6–8% de las filas).

**No es leakage:** usa etiquetas de vecinos, así que va con *cross-fitting* como `PrediccionOOF`. En el train del fold, cada fila busca vecinos solo en los **otros** 4 pedazos internos (nunca ve su etiqueta). Validación y test buscan vecinos en todo el train del fold. Va adentro del `Pipeline`.

**Prueba previa (fold 1):** (a) 0,4 min, la feature sola acierta 0,774 (k=10) / 0,786 (k=30); (b) 0,9 min, 0,796 / 0,790. Más flojo que el modelo de texto (0,84): se mide si **suma**, no si gana solo.

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold as _SKF


class KNNTitulosOOF(BaseEstimator, TransformerMixin):
    """% de new entre los k títulos más parecidos (similitud coseno sobre TF-IDF del título),
    para cada k en ks. Recibe X con columnas [title, category_id] (category_id solo si por_categoria).
    fit_transform (train): cross-fitting, cada fila ve solo vecinos de OTROS folds internos.
    transform (validación/test): vecinos entre todas las filas del fit.
    Sin vecinos válidos (ej. categoría sin otras filas) -> NaN."""

    def __init__(self, ks=(10, 30), por_categoria=False, n_splits=5, tamano_bloque=1000, random_state=42):
        self.ks = ks
        self.por_categoria = por_categoria
        self.n_splits = n_splits
        self.tamano_bloque = tamano_bloque
        self.random_state = random_state

    def _vecinos(self, V_consulta, cat_consulta, V_base, y_base, cat_base):
        k_max = max(self.ks)
        salida = np.full((V_consulta.shape[0], len(self.ks)), np.nan)
        for inicio in range(0, V_consulta.shape[0], self.tamano_bloque):
            fin = inicio + self.tamano_bloque
            sim = (V_consulta[inicio:fin] @ V_base.T).toarray()
            if self.por_categoria:
                sim[cat_consulta[inicio:fin, None] != cat_base[None, :]] = -np.inf
            k_efectivo = min(k_max, sim.shape[1])
            top = np.argpartition(-sim, k_efectivo - 1, axis=1)[:, :k_efectivo]
            sim_top = np.take_along_axis(sim, top, axis=1)
            orden = np.argsort(-sim_top, axis=1)
            top, sim_top = np.take_along_axis(top, orden, axis=1), np.take_along_axis(sim_top, orden, axis=1)
            validos = np.isfinite(sim_top)
            etiquetas = np.where(validos, y_base[top], np.nan)
            for j, k in enumerate(self.ks):
                with np.errstate(invalid="ignore"):
                    salida[inicio:fin, j] = np.nanmean(etiquetas[:, :k], axis=1) if validos[:, :k].any() else np.nan
        return salida

    def fit(self, X, y):
        titulos = X.iloc[:, 0].astype(str)
        self.vectorizador_ = TfidfVectorizer(ngram_range=(1, 2), min_df=2, strip_accents="unicode").fit(titulos)
        self.V_ = self.vectorizador_.transform(titulos).tocsr()
        self.y_ = np.asarray(y, dtype=float)
        self.cat_ = X.iloc[:, 1].astype(str).to_numpy() if self.por_categoria else np.zeros(len(X), dtype=object)
        return self

    def transform(self, X):
        V = self.vectorizador_.transform(X.iloc[:, 0].astype(str)).tocsr()
        cat = X.iloc[:, 1].astype(str).to_numpy() if self.por_categoria else np.zeros(len(X), dtype=object)
        return self._vecinos(V, cat, self.V_, self.y_, self.cat_)

    def fit_transform(self, X, y=None):
        self.fit(X, y)
        salida = np.full((len(X), len(self.ks)), np.nan)
        cv_interna = _SKF(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        for entrena, prueba in cv_interna.split(self.V_, self.y_):
            salida[prueba] = self._vecinos(self.V_[prueba], self.cat_[prueba], self.V_[entrena], self.y_[entrena],
                                           self.cat_[entrena])
        return salida

In [5]:
nombre_p7a = "P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target"
extras_p7a = [
    ("te_combos", TargetEncoder(target_type="binary", random_state=SEED), ["seller_id", "category_id"] + COLUMNAS_COMBOS),
    ("vendedor_simple", VendedorEncoder(incluir_target=False), ["seller_id", "log_price"]),
]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for por_cat, letra in [(False, "a"), (True, "b")]:
        nombre = f"I1{letra} - HistGB P7a + kNN de títulos (k=10, 30){' misma categoría' if por_cat else ''}"
        probar(nombre, sacar={"vendedor", "categoria"},
               transformers_extra=extras_p7a + [("knn_titulos", KNNTitulosOOF(por_categoria=por_cat, random_state=SEED),
                                                 ["title", "category_id"])])
        comparar_pareado(nombre_p7a, nombre)
        print()

I1a - HistGB P7a + kNN de títulos (k=10, 30) -> accuracy: 0.9365 +/- 0.0016 | roc_auc: 0.9837 | minutos: 1.9
I1a - HistGB P7a + kNN de títulos (k=10, 30)  vs  P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target
  diferencia por fold: [-0.0001 -0.0008 -0.0011 -0.0005 -0.0005]
  folds en que gana B: 0 de 5
  media de la diferencia: -0.0006 | desvío de la diferencia: 0.0003
  -> no pasa el criterio



I1b - HistGB P7a + kNN de títulos (k=10, 30) misma categoría -> accuracy: 0.9365 +/- 0.0008 | roc_auc: 0.9837 | minutos: 1.9
I1b - HistGB P7a + kNN de títulos (k=10, 30) misma categoría  vs  P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target
  diferencia por fold: [-0.0015  0.001  -0.0016 -0.0006  0.    ]
  folds en que gana B: 1 de 5
  media de la diferencia: -0.0006 | desvío de la diferencia: 0.001
  -> no pasa el criterio



**Conclusión de I1: no suma.** (a) 0,9365 ± 0,0016, 0/5 contra el HistGB P7a (−0,0006); (b) misma categoría: 0,9365 ± 0,0008, 1/5 (−0,0006). 1,9 min cada una. El "voto" de los títulos parecidos ya está cubierto por el modelo de texto (`prob_texto_new`), que con 238.008 columnas y una regresión logística resume mejor el título que un promedio de 10–30 vecinos.

## I2 — Stacking con contexto

Un **meta-modelo** combina las probabilidades OOF de CatBoost P6b y del HistGB P7a (guardadas en `oof_p7a.csv`) junto con contexto de la publicación: log(1 + publicaciones del vendedor en train), `tiene_garantia`, `log_price`, `listing_type_id` (one-hot) y `es_tienda_oficial`. Puede aprender, por ejemplo, a confiar más en uno de los dos según el vendedor. Dos meta-modelos: regresión logística (entradas escaladas) y un HistGB chico (`max_iter=100`, `max_leaf_nodes=15`, `learning_rate=0.05`).

**No es leakage:** las probabilidades de entrada son *out-of-fold*; el meta-modelo se entrena con 4 folds y se evalúa en el quinto; el contexto no usa el target. Sesgo chico conocido (igual que en `16_ensambles` y P7c): las probabilidades de los 4 folds de entrenamiento vienen de modelos base que sí vieron el quinto, así que la estimación queda levemente optimista. Material: Stacking en `Árboles.pdf` (slide 56) y S8 de SIA (*"no forma parte de la versión actual de la clase"*).

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

oof = pd.read_csv(os.path.join(carpeta_lab, "oof_p7a.csv"))
assert (oof["y"].to_numpy() == y.to_numpy()).all()
contexto = pd.DataFrame({
    "log_publicaciones_vendedor": np.log1p(train["seller_id"].map(train["seller_id"].value_counts())).to_numpy(),
    "tiene_garantia": train["tiene_garantia"].astype(int).to_numpy(),
    "log_price": np.log(train["price"].clip(lower=1)).to_numpy(),
    "es_tienda_oficial": train["es_tienda_oficial"].astype(int).to_numpy(),
})
contexto = pd.concat([contexto, pd.get_dummies(train["listing_type_id"], prefix="tipo").astype(int).reset_index(drop=True)], axis=1)
X_meta = pd.concat([oof[["catboost", "histgb_p7a"]], contexto], axis=1).to_numpy(dtype=float)
y_arr = oof["y"].to_numpy()

meta_modelos = {
    "regresión logística": lambda: Pipeline([("escala", StandardScaler()),
                                             ("modelo", LogisticRegression(max_iter=1000, random_state=SEED))]),
    "HistGB chico": lambda: HistGradientBoostingClassifier(max_iter=100, max_leaf_nodes=15, learning_rate=0.05,
                                                            random_state=SEED),
}
NOMBRE_P7C = "P7c (ref) - CatBoost P6b + HistGB P7a 50/50"
for nombre_meta, crear in meta_modelos.items():
    acc = []
    for k in range(1, 6):
        entrena = (oof["fold"] != k).to_numpy()
        meta = crear().fit(X_meta[entrena], y_arr[entrena])
        acc.append((meta.predict(X_meta[~entrena]) == y_arr[~entrena]).mean())
    nombre = f"I2 - Stacking con contexto ({nombre_meta})"
    registro = cargar_resultados()
    registro.loc[nombre] = [pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), np.mean(acc), np.std(acc), np.nan, *acc, np.nan]
    registro.to_csv(archivo_resultados)
    print(f"{nombre}: accuracy {np.mean(acc):.4f} ± {np.std(acc):.4f}")
    comparar_pareado(NOMBRE_P7C, nombre)
    print()

I2 - Stacking con contexto (regresión logística): accuracy 0.9373 ± 0.0017
I2 - Stacking con contexto (regresión logística)  vs  P7c (ref) - CatBoost P6b + HistGB P7a 50/50
  diferencia por fold: [-0.0004 -0.0009  0.0001  0.     -0.0004]
  folds en que gana B: 1 de 5
  media de la diferencia: -0.0003 | desvío de la diferencia: 0.0003
  -> no pasa el criterio



I2 - Stacking con contexto (HistGB chico): accuracy 0.9377 ± 0.0019
I2 - Stacking con contexto (HistGB chico)  vs  P7c (ref) - CatBoost P6b + HistGB P7a 50/50
  diferencia por fold: [ 0.0004 -0.0006 -0.0003  0.0005  0.0005]
  folds en que gana B: 3 de 5
  media de la diferencia: 0.0001 | desvío de la diferencia: 0.0004
  -> no pasa el criterio



**Conclusión de I2: no pasa.** Meta-modelo regresión logística: 0,9373, 1/5 contra P7c (−0,0003). HistGB chico: 0,9377, 3/5 (+0,0001, desvío 0,0004) → ruido. El contexto (vendedor, garantía, precio, tipo, tienda oficial) no ayuda a decidir cuándo confiar más en cada modelo: los dos ya usan esas mismas features y se equivocan en publicaciones parecidas.

## I3 — Pseudo-labeling (HistGB P7a)

En cada fold k:
1. se entrena el HistGB P7a con los otros 4 folds;
2. se predicen el fold k y el test **sin usar sus etiquetas**;
3. las filas con probabilidad > 0,97 o < 0,03 se suman al train con la etiqueta **predicha**;
4. se reentrena y se evalúa en el fold k.

**No es leakage:** las etiquetas del fold k nunca se usan. Usar sus features y las del test sin etiqueta reproduce lo que pasaría en la entrega (el test real está disponible sin etiquetas y se le haría lo mismo). El *pseudo-labeling* no está en el material.

Se reporta cuántas filas se agregan, qué tan correctas son las pseudo-etiquetas del fold k (medido con sus etiquetas reales, solo para el reporte) y qué pasa con los vendedores **sin historial** (sin publicaciones en el train del fold). Prueba previa: 1,3 min por fold.

In [4]:
from features import agregar_features_contenido, armar_pipeline_histgb_combos

test_i3 = agregar_combos(agregar_features_contenido(aplanar_anidadas(pd.read_json("../data/test_data.jsonlines", lines=True))))


def pseudo_labeling_fold(k, umbral_alto=0.97, umbral_bajo=0.03):
    """Para el fold k: entrena HistGB P7a con los otros 4 folds, agrega como pseudo-etiquetas las filas del
    fold k y del test con probabilidad > umbral_alto o < umbral_bajo, reentrena y evalúa en el fold k."""
    entrena, valida = folds[k]
    X_entrena, y_entrena = train.iloc[entrena], y.iloc[entrena]
    X_valida, y_valida = train.iloc[valida], y.iloc[valida].to_numpy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        modelo = armar_pipeline_histgb_combos().fit(X_entrena, y_entrena)
        prob_valida = modelo.predict_proba(X_valida)[:, 1]
        prob_test = modelo.predict_proba(test_i3)[:, 1]
        seguras_valida = (prob_valida > umbral_alto) | (prob_valida < umbral_bajo)
        seguras_test = (prob_test > umbral_alto) | (prob_test < umbral_bajo)
        X_nuevo = pd.concat([X_entrena, X_valida[seguras_valida], test_i3[seguras_test]], ignore_index=True)
        y_nuevo = pd.Series(np.concatenate([y_entrena.to_numpy(), (prob_valida[seguras_valida] > 0.5).astype(int),
                                            (prob_test[seguras_test] > 0.5).astype(int)]))
        modelo_pl = armar_pipeline_histgb_combos().fit(X_nuevo, y_nuevo)
        pred_pl = modelo_pl.predict(X_valida)
    sin_historial = ~X_valida["seller_id"].isin(set(X_entrena["seller_id"])).to_numpy()
    pred_base = (prob_valida >= 0.5).astype(int)
    return {
        "fold": k + 1,
        "agregadas_fold": int(seguras_valida.sum()),
        "agregadas_test": int(seguras_test.sum()),
        "acc_base": (pred_base == y_valida).mean(),
        "acc_pseudo": (pred_pl == y_valida).mean(),
        "acc_base_sin_historial": (pred_base == y_valida)[sin_historial].mean(),
        "acc_pseudo_sin_historial": (pred_pl == y_valida)[sin_historial].mean(),
        "pct_sin_historial": sin_historial.mean(),
        "precision_pseudo_etiquetas_fold": ((prob_valida[seguras_valida] > 0.5).astype(int) == y_valida[seguras_valida]).mean(),
    }

In [5]:
inicio = time.time()
filas_i3 = [pseudo_labeling_fold(k) for k in range(5)]
tabla_i3 = pd.DataFrame(filas_i3).set_index("fold")
minutos_i3 = (time.time() - inicio) / 60
print(f"{minutos_i3:.1f} min")

acc_pl = tabla_i3["acc_pseudo"].to_numpy()
registro = cargar_resultados()
registro.loc["I3 - HistGB P7a con pseudo-labeling (0,97/0,03)"] = [
    pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), acc_pl.mean(), acc_pl.std(), np.nan, *acc_pl, minutos_i3]
registro.to_csv(archivo_resultados)
comparar_pareado("P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target",
                 "I3 - HistGB P7a con pseudo-labeling (0,97/0,03)")
tabla_i3.round(4)

6.5 min
I3 - HistGB P7a con pseudo-labeling (0,97/0,03)  vs  P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target
  diferencia por fold: [-0.0004  0.0011 -0.002   0.     -0.0014]
  folds en que gana B: 1 de 5
  media de la diferencia: -0.0006 | desvío de la diferencia: 0.0011
  -> no pasa el criterio


,agregadas_fold,agregadas_test,acc_base,acc_pseudo,acc_base_sin_historial,acc_pseudo_sin_historial,pct_sin_historial,precision_pseudo_etiquetas_fold
fold,,,,,,,,
1,9067,19304,0.9384,0.9379,0.8846,0.8877,0.3021,0.9946
2,8928,19154,0.9344,0.9354,0.8800,0.8838,0.3161,0.9941
3,8762,18783,0.9382,0.9362,0.8825,0.8767,0.3069,0.9949
4,9070,19335,0.9369,0.9369,0.8818,0.8823,0.3021,0.9937
5,9163,19537,0.9377,0.9363,0.8806,0.8792,0.3094,0.9931


**Conclusión de I3: no pasa.** 0,9365 en promedio, 1/5 contra el HistGB P7a (−0,0006), 6,5 min. Se agregan ~9.000 filas del fold y ~19.000 del test por fold, y las pseudo-etiquetas son correctas en el **99,3–99,5%**. Justamente por eso no enseñan nada nuevo: son las publicaciones que el modelo ya clasifica con seguridad. En los vendedores sin historial (31% de cada fold) la accuracy sube en 3 folds y baja en 2: no hay una mejora consistente.

## I4 — Especialista en vendedores sin historial

En cada fold, un segundo HistGB P7a (`armar_pipeline_histgb_combos`) se entrena **solo con las filas cuyo vendedor tiene ≤ 1 publicación** en el train del fold (el grupo que concentra la mitad de los errores, ver `08_errores`). Para las filas de validación de ese grupo (vendedor con 0 o 1 publicación en el train del fold): probabilidad = w · especialista + (1 − w) · P7c, con w ∈ {0; 0,1; …; 1} elegido con 4 folds y evaluado en el quinto. Al resto se le aplica P7c tal cual.

**No es leakage:** el especialista se entrena con el train del fold; el grupo se define contando publicaciones (sin target) en el train del fold; P7c sale de las probabilidades OOF guardadas; el peso se elige rotando.

In [4]:
from features import armar_pipeline_histgb_combos

oof = pd.read_csv(os.path.join(carpeta_lab, "oof_p7a.csv"))
assert (oof["y"].to_numpy() == y.to_numpy()).all()
p7c = 0.5 * oof["catboost"].to_numpy() + 0.5 * oof["histgb_p7a"].to_numpy()
prob_especialista = np.full(len(train), np.nan)
en_grupo = np.zeros(len(train), dtype=bool)

inicio = time.time()
for k, (entrena, valida) in enumerate(folds):
    conteo = train.iloc[entrena]["seller_id"].value_counts()
    chico_entrena = (train.iloc[entrena]["seller_id"].map(conteo) <= 1).to_numpy()
    chico_valida = (train.iloc[valida]["seller_id"].map(conteo).fillna(0) <= 1).to_numpy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        especialista = armar_pipeline_histgb_combos().fit(train.iloc[entrena][chico_entrena], y.iloc[entrena][chico_entrena])
        prob_especialista[valida[chico_valida]] = especialista.predict_proba(train.iloc[valida][chico_valida])[:, 1]
    en_grupo[valida[chico_valida]] = True
    print(f"fold {k + 1}: especialista entrenado con {chico_entrena.sum()} filas; grupo en validación: {chico_valida.sum()} filas")
minutos_i4 = (time.time() - inicio) / 60
print(f"{minutos_i4:.1f} min")

y_arr = y.to_numpy()
fold_fila = oof["fold"].to_numpy()
pesos = np.round(np.arange(0, 1.001, 0.1), 1)


def combinar(w):
    final = p7c.copy()
    final[en_grupo] = w * prob_especialista[en_grupo] + (1 - w) * p7c[en_grupo]
    return final


acc_i4, w_elegidos, acc_grupo = [], [], []
for k in range(1, 6):
    otros = (fold_fila != k) & en_grupo
    w = pesos[int(np.argmax([((combinar(w)[otros] >= 0.5).astype(int) == y_arr[otros]).mean() for w in pesos]))]
    w_elegidos.append(w)
    este = fold_fila == k
    acc_i4.append(((combinar(w)[este] >= 0.5).astype(int) == y_arr[este]).mean())
    g = este & en_grupo
    acc_grupo.append((((combinar(w)[g] >= 0.5).astype(int) == y_arr[g]).mean(), ((p7c[g] >= 0.5).astype(int) == y_arr[g]).mean()))
print("w del especialista elegido en cada rotación:", w_elegidos)
print("Accuracy en el grupo (con especialista, P7c solo):", [tuple(round(a, 4) for a in par) for par in acc_grupo])
print(f"Especialista solo en el grupo: {((prob_especialista[en_grupo] >= 0.5).astype(int) == y_arr[en_grupo]).mean():.4f} | "
      f"P7c en el grupo: {((p7c[en_grupo] >= 0.5).astype(int) == y_arr[en_grupo]).mean():.4f}")

registro = cargar_resultados()
registro.loc["I4 - P7c + especialista en vendedores con <= 1 publicación"] = [
    pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), np.mean(acc_i4), np.std(acc_i4), np.nan, *acc_i4, minutos_i4]
registro.to_csv(archivo_resultados)
comparar_pareado("P7c (ref) - CatBoost P6b + HistGB P7a 50/50", "I4 - P7c + especialista en vendedores con <= 1 publicación")

fold 1: especialista entrenado con 17312 filas; grupo en validación: 5711 filas


fold 2: especialista entrenado con 17106 filas; grupo en validación: 5869 filas


fold 3: especialista entrenado con 17224 filas; grupo en validación: 5757 filas


fold 4: especialista entrenado con 17274 filas; grupo en validación: 5706 filas


fold 5: especialista entrenado con 17230 filas; grupo en validación: 5832 filas
0.7 min
w del especialista elegido en cada rotación: [np.float64(0.2), np.float64(0.2), np.float64(0.1), np.float64(0.1), np.float64(0.1)]
Accuracy en el grupo (con especialista, P7c solo): [(np.float64(0.8993), np.float64(0.899)), (np.float64(0.8915), np.float64(0.8915)), (np.float64(0.8975), np.float64(0.8975)), (np.float64(0.8994), np.float64(0.8973)), (np.float64(0.8981), np.float64(0.8985))]
Especialista solo en el grupo: 0.8825 | P7c en el grupo: 0.8967
I4 - P7c + especialista en vendedores con <= 1 publicación  vs  P7c (ref) - CatBoost P6b + HistGB P7a 50/50
  diferencia por fold: [ 0.0001  0.      0.      0.0009 -0.0001]
  folds en que gana B: 2 de 5
  media de la diferencia: 0.0002 | desvío de la diferencia: 0.0004
  -> no pasa el criterio


**Conclusión de I4: no pasa.** 0,9378 en promedio, 2/5 contra P7c (+0,0002, desvío 0,0004) → ruido. 0,7 min. El especialista se entrena con ~17.200 filas por fold, y el grupo es ~5.800 filas de validación por fold. **Solo, el especialista es peor que P7c en su propio grupo (0,8825 contra 0,8967)**: entrenar únicamente con vendedores sin historial le saca los ejemplos de todos los demás, que también enseñan. Por eso el peso elegido es chico (0,1–0,2) y el resultado queda igual que P7c.

## Cierre de la ronda final (I1–I4): nada pasó. **Modelo final: P7c 50/50.**

| Prueba | accuracy | folds vs referencia | tiempo |
|---|---|---|---|
| I1a: kNN de títulos (todo el train), en HistGB P7a | 0,9365 | 0/5 vs P7a | 1,9 min |
| I1b: kNN de títulos (misma categoría), en HistGB P7a | 0,9365 | 1/5 vs P7a | 1,9 min |
| I2: stacking con contexto, meta-modelo regresión logística | 0,9373 | 1/5 vs P7c | segundos |
| I2: stacking con contexto, meta-modelo HistGB chico | 0,9377 | 3/5 vs P7c (ruido) | segundos |
| I3: pseudo-labeling (0,97/0,03), HistGB P7a | 0,9365 | 1/5 vs P7a | 6,5 min |
| I4: P7c + especialista en vendedores sin historial | 0,9378 | 2/5 vs P7c (ruido) | 0,7 min |

Ninguna idea pasa el criterio, así que no hay combinación, *ablation*, robustez ni submission nueva.

# Última prueba — Modelo en cascada (*two-stage*)

```
publicación → Etapa 1: P7c → P(new)
   ├─ P(new) < 0,2 ó > 0,8  (seguro)  → queda la respuesta de P7c
   └─ 0,2 ≤ P(new) ≤ 0,8    (duda)    → Etapa 2: especialista entrenado SOLO con la zona de duda
                                         (a) HistGB sobre features P7a + P(new) de P7c
                                         (b) regresión logística sobre TF-IDF grande + P(new) de P7c
                                         → decide el especialista, o se promedia con P7c
```

**No es leakage:**
- la zona se define con las probabilidades **OOF** de P7c;
- para evaluar el fold k, el especialista se entrena solo con la zona de los otros 4 folds, con features de un preprocesamiento ajustado con esos 4 folds (el fold k solo se transforma);
- umbrales (0,2/0,8 o 0,3/0,7), especialista (a/b) y modo (reemplazar o promediar) se eligen con 4 folds y se evalúan en el quinto.

Sesgo chico conocido (igual que en los ensambles): las probabilidades OOF de P7c de las filas de entrenamiento vienen de modelos que vieron el quinto fold.

Material: umbral de decisión (S8 de SIA; curva ROC en `Repaso de Metricas.pdf`); regresión logística (`Presentación_Logística.pdf`); Gradient Boosting (`Árboles.pdf`, S8). La cascada no está en el material. Sin oversampling ni SMOTE (`datasets desbalanceados`): se verifica que dentro de la zona las clases estén casi balanceadas.

### 1. La zona de duda

In [4]:
oof = pd.read_csv(os.path.join(carpeta_lab, "oof_p7a.csv"))
assert (oof["y"].to_numpy() == y.to_numpy()).all()
p7c = 0.5 * oof["catboost"].to_numpy() + 0.5 * oof["histgb_p7a"].to_numpy()
y_arr = oof["y"].to_numpy()
fold_fila = oof["fold"].to_numpy()
acierto_p7c = (p7c >= 0.5).astype(int) == y_arr

umbrales = {"0,2/0,8": (0.2, 0.8), "0,3/0,7": (0.3, 0.7)}
for nombre, (bajo, alto) in umbrales.items():
    zona = (p7c >= bajo) & (p7c <= alto)
    print(f"Zona {nombre}: {zona.sum()} filas ({zona.mean():.1%}) | accuracy de P7c en la zona: {acierto_p7c[zona].mean():.4f}"
          f" | % new en la zona: {y_arr[zona].mean():.1%} | errores de P7c dentro de la zona: "
          f"{(~acierto_p7c & zona).sum()} de {(~acierto_p7c).sum()} ({(~acierto_p7c & zona).sum() / (~acierto_p7c).sum():.1%})")
print(f"Accuracy de P7c fuera de la zona 0,2/0,8: {acierto_p7c[~((p7c >= 0.2) & (p7c <= 0.8))].mean():.4f}")

Zona 0,2/0,8: 7679 filas (11.0%) | accuracy de P7c en la zona: 0.6567 | % new en la zona: 46.6% | errores de P7c dentro de la zona: 2636 de 4368 (60.3%)
Zona 0,3/0,7: 4492 filas (6.4%) | accuracy de P7c en la zona: 0.6013 | % new en la zona: 48.0% | errores de P7c dentro de la zona: 1791 de 4368 (41.0%)
Accuracy de P7c fuera de la zona 0,2/0,8: 0.9722


### 2. Probabilidades OOF de los especialistas

Para cada fold j y cada par de umbrales: el especialista se entrena con las filas de la zona de los otros 4 folds y predice las filas de la zona del fold j.
- **(a) HistGB**: el preprocesamiento del HistGB P7a (`armar_pipeline_histgb_combos` sin el modelo) se ajusta con los 4 folds (`fit_transform`, con *cross-fitting* adentro) y transforma el fold j. Al resultado se le suma la probabilidad de P7c. HistGB con los hiperparámetros fijos de siempre.
- **(b) Regresión logística**: TF-IDF grande del modelo de texto (título palabras y caracteres + garantía, ajustado con los 4 folds) + logit de la probabilidad de P7c como columna extra, `C=2`.

In [5]:
from features import armar_pipeline_histgb_combos, armar_modelo_texto
from scipy import sparse
from sklearn.linear_model import LogisticRegression


def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


prob_especialista = {(t, e): np.full(len(train), np.nan) for t in umbrales for e in ["a", "b"]}
inicio = time.time()
for j, (entrena, valida) in enumerate(folds):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        preprocesador = Pipeline(armar_pipeline_histgb_combos().steps[:-1])
        Xa_entrena = preprocesador.fit_transform(train.iloc[entrena], y.iloc[entrena])
        Xa_valida = preprocesador.transform(train.iloc[valida])
        vectores = armar_modelo_texto().named_steps["vectores"]
        Xb_entrena = vectores.fit_transform(train.iloc[entrena])
        Xb_valida = vectores.transform(train.iloc[valida])
    for t, (bajo, alto) in umbrales.items():
        z_e = (p7c[entrena] >= bajo) & (p7c[entrena] <= alto)
        z_v = (p7c[valida] >= bajo) & (p7c[valida] <= alto)
        extra_e, extra_v = p7c[entrena][z_e][:, None], p7c[valida][z_v][:, None]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            esp_a = HistGradientBoostingClassifier(learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED)
            esp_a.fit(np.hstack([Xa_entrena[z_e], extra_e]), y_arr[entrena][z_e])
            prob_especialista[(t, "a")][valida[z_v]] = esp_a.predict_proba(np.hstack([Xa_valida[z_v], extra_v]))[:, 1]
            esp_b = LogisticRegression(C=2, max_iter=2000, random_state=SEED)
            esp_b.fit(sparse.hstack([Xb_entrena[z_e], logit(extra_e)]).tocsr(), y_arr[entrena][z_e])
            prob_especialista[(t, "b")][valida[z_v]] = esp_b.predict_proba(sparse.hstack([Xb_valida[z_v], logit(extra_v)]).tocsr())[:, 1]
    print(f"fold {j + 1} listo ({(time.time() - inicio) / 60:.1f} min)")
minutos_cascada = (time.time() - inicio) / 60

for (t, e), prob in prob_especialista.items():
    zona = ~np.isnan(prob)
    print(f"Especialista ({e}), zona {t}: accuracy en la zona {((prob[zona] >= 0.5) == y_arr[zona]).mean():.4f} "
          f"(P7c en la misma zona: {acierto_p7c[zona].mean():.4f})")

fold 1 listo (0.5 min)


fold 2 listo (1.1 min)


fold 3 listo (1.7 min)


fold 4 listo (2.2 min)


fold 5 listo (2.8 min)
Especialista (a), zona 0,2/0,8: accuracy en la zona 0.6317 (P7c en la misma zona: 0.6567)
Especialista (b), zona 0,2/0,8: accuracy en la zona 0.6197 (P7c en la misma zona: 0.6567)
Especialista (a), zona 0,3/0,7: accuracy en la zona 0.5628 (P7c en la misma zona: 0.6013)
Especialista (b), zona 0,3/0,7: accuracy en la zona 0.5715 (P7c en la misma zona: 0.6013)


### 3. Configuración elegida con 4 folds y evaluada en el quinto

Configuraciones: umbrales (0,2/0,8, 0,3/0,7) × especialista (a, b) × modo (el especialista **reemplaza** a P7c en la zona, o se **promedia** 50/50 con P7c). En cada rotación se elige la de mayor accuracy en los otros 4 folds (ante empate, la primera de la lista) y se mide en el quinto.

In [6]:
def prediccion_final(t, e, modo):
    final = p7c.copy()
    zona = ~np.isnan(prob_especialista[(t, e)])
    final[zona] = prob_especialista[(t, e)][zona] if modo == "reemplaza" else 0.5 * (prob_especialista[(t, e)][zona] + p7c[zona])
    return final


configuraciones = [(t, e, m) for t in umbrales for e in ["a", "b"] for m in ["reemplaza", "promedia"]]
aciertos = {c: (prediccion_final(*c) >= 0.5).astype(int) == y_arr for c in configuraciones}
print("Accuracy OOF total de cada configuración (solo descriptivo, no se elige con esto):")
for c in configuraciones:
    print(f"   {c}: {aciertos[c].mean():.4f}")

acc_cascada, elegidas = [], []
for k in range(1, 6):
    otros = fold_fila != k
    c = max(configuraciones, key=lambda c: aciertos[c][otros].mean())
    elegidas.append(c)
    acc_cascada.append(aciertos[c][fold_fila == k].mean())
print("\nConfiguración elegida en cada rotación:", elegidas)

registro = cargar_resultados()
registro.loc["Cascada - P7c + especialista en zona de duda (config. elegida en 4 folds)"] = [
    pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), np.mean(acc_cascada), np.std(acc_cascada), np.nan, *acc_cascada, minutos_cascada]
registro.to_csv(archivo_resultados)
print(f"Cascada: accuracy {np.mean(acc_cascada):.4f} ± {np.std(acc_cascada):.4f}")
comparar_pareado("P7c (ref) - CatBoost P6b + HistGB P7a 50/50", "Cascada - P7c + especialista en zona de duda (config. elegida en 4 folds)")

Accuracy OOF total de cada configuración (solo descriptivo, no se elige con esto):
   ('0,2/0,8', 'a', 'reemplaza'): 0.9349
   ('0,2/0,8', 'a', 'promedia'): 0.9370
   ('0,2/0,8', 'b', 'reemplaza'): 0.9335
   ('0,2/0,8', 'b', 'promedia'): 0.9366
   ('0,3/0,7', 'a', 'reemplaza'): 0.9351
   ('0,3/0,7', 'a', 'promedia'): 0.9366
   ('0,3/0,7', 'b', 'reemplaza'): 0.9357
   ('0,3/0,7', 'b', 'promedia'): 0.9369

Configuración elegida en cada rotación: [('0,3/0,7', 'b', 'promedia'), ('0,3/0,7', 'b', 'promedia'), ('0,2/0,8', 'a', 'promedia'), ('0,2/0,8', 'a', 'promedia'), ('0,3/0,7', 'b', 'promedia')]
Cascada: accuracy 0.9364 ± 0.0012
Cascada - P7c + especialista en zona de duda (config. elegida en 4 folds)  vs  P7c (ref) - CatBoost P6b + HistGB P7a 50/50
  diferencia por fold: [-0.0019 -0.0009  0.0003 -0.0019 -0.0014]
  folds en que gana B: 1 de 5
  media de la diferencia: -0.0012 | desvío de la diferencia: 0.0008
  -> no pasa el criterio


**Conclusión de la cascada: no pasa.** 0,9364 ± 0,0012 contra 0,9376 de P7c: 1/5 folds, media −0,0012. Tiempo: 2,8 min.

| | zona 0,2/0,8 | zona 0,3/0,7 |
|---|---|---|
| Filas en la zona | 7.679 (11,0%) | 4.492 (6,4%) |
| % `new` en la zona | 46,6% | 48,0% |
| Accuracy de P7c en la zona | 0,6567 | 0,6013 |
| Errores de P7c que caen en la zona | 60,3% | 41,0% |
| Especialista (a) HistGB en la zona | 0,6317 | 0,5628 |
| Especialista (b) regresión logística + TF-IDF en la zona | 0,6197 | 0,5715 |

- **El diagnóstico se confirma:** fuera de la zona 0,2/0,8, P7c acierta el **97,2%**. El 60% de sus errores está en el 11% de filas donde duda.
- **Clases balanceadas en la zona** (46,6% y 48,0% `new`): no hacía falta oversampling ni SMOTE.
- **Pero los especialistas son peores que P7c dentro de su propia zona** (0,62–0,63 contra 0,66 en la zona ancha). Entrenados solo con ~6.000 casos difíciles, sin los fáciles, no encuentran un patrón nuevo: en esas filas la información disponible es genuinamente ambigua. Ahí está el ruido de etiqueta de `08_errores` (títulos "Nuevo!!!" o "Sin Uso" etiquetados `used`), mezclado con casos difíciles de etiqueta correcta (por ejemplo, "Departamento a estrenar", bien etiquetado `new`, que el modelo da como `used`).
- Promediar con P7c es la mejor configuración, pero igual queda por debajo de P7c solo.

**Cierre definitivo de experimentos: el modelo final es P7c 50/50.**